In [1]:
# ── TEMPORARY: Suppress all Plotly charts to prevent VS Code freeze ──
# Run this cell first, then run the rest of the notebook.
# Delete or skip this cell when you want charts to render again.
import plotly.graph_objects as go
import plotly.express as px
_orig_fig_show   = go.Figure.show
go.Figure.show   = lambda *a, **kw: None   # no-op
_orig_subplots = None  # subplots figures also use go.Figure.show — already covered
print("⚠️  Chart rendering SUPPRESSED — text outputs only. Delete this cell to re-enable charts.")


⚠️  Chart rendering SUPPRESSED — text outputs only. Delete this cell to re-enable charts.


# Ariel Gel Ball Pricing Strategy — Consolidated Analysis

**Objective:** Brand building via loyal user growth (Trial / Repeat / Lapse): define each size's most effective price point by understanding shopper flow.

---

## Table of Contents

| # | Section | Description |
|---|---------|-------------|
| 1 | **Analysis Brief** | Objective, definitions, parameters, analysis flow |
| 2 | **Setup & Data Pull** | Imports, parameters, SQL queries with caching |
| A | **Pane A — Total Shopper & ASP Landscape** | Market sizing, pricing trends, renewal impact |
| B | **Pane B — Trial Shopper** | Trial acquisition by size, ASP elasticity, price gap |
| C | **Pane C — Repeat Shopper** | Cohort funnel, size migration, ASP band analysis |
| D | **Pane D — Lapsed Shopper** | Lapse rate by size/ASP, post-lapse destination |
| E | **Pane E — Shopper Flow** | Sankey, ASP-annotated funnel, strategic map |
| F | **Pane F — Strategic Summary** | Synthesized findings & recommendations |

**Created:** 2026-02-19 | **Consolidated:** 2026-02-27


---
### 📏 Canonical Definitions

| Term | Definition | Window |
|------|-----------|--------|
| **Trial Shopper** | A shopper who purchases the sub-brand/category with **no purchase history of that sub-brand/category in the prior 12 months** (365 days). Rolling lookback per purchase event, NOT first-ever. | 365-day lookback |
| **Repeat Shopper** | A trial shopper who makes ≥1 subsequent purchase of the **same sub-brand** within **180 days** (6 months) after their trial event. | 180-day forward window |
| **Lapsed Shopper** | A trial shopper who makes **no subsequent purchase** of the same sub-brand within **180 days** (6 months) after their trial event. | 180-day forward window |
| **ASP** | `SUM(pos_sales_amt) / SUM(pos_unit_sales_qty)` — weighted average selling price. | Per transaction/aggregation |
| **ASP Band (50 JPY bin)** | `FLOOR(ASP / 50) * 50` — ASP floored to nearest 50 JPY. | — |

**Repeat + Lapse are mutually exclusive and exhaustive** within the trial cohort.

### Analysis Flow
```
Total Shoppers (ASP Landscape)
  └── Trial Shoppers (new-to-brand)
        ├── Repeat Shoppers (retained within 6 months)
        │     └── Size migration (entry size → repeat size)
        └── Lapsed Shoppers (no return within 6 months)
              └── Destination tracking (where did they go?)

At each stage: How does PRICE affect the shopper's behavior?
```

### Data Sources (IDPOS_REFERENCE.md)
- **Fact table:** `cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw`
- **Product dim:** `id_pos_ai_1.prod_dim_ext_vw`
- **Shopper dim:** `id_pos_ai_1.shopper_dim_generic_vw`
- **Partition keys:** `sales_period_group_end_date_part`, `data_provider_code_part`


---
# Section 2 — Setup, Parameters & Data Pull


In [2]:
# ═══════════════════════════════════════════════════════════════════════
# 2-1. Imports & Connection
# ═══════════════════════════════════════════════════════════════════════
import os, time, warnings
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np
from dotenv import load_dotenv
import databricks.sql as sql

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', 50)
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.1f}')

# ── Japanese font setup ────────────────────────────────────────────────
def _find_japanese_font():
    for name in ['MS Gothic', 'MS PGothic', 'Yu Gothic', 'Meiryo', 'IPAexGothic']:
        if name in {f.name for f in fm.fontManager.ttflist}:
            return name
    return None

_jp_font = _find_japanese_font()
if _jp_font:
    plt.rcParams['font.family'] = _jp_font
    print(f'✅ Japanese font: {_jp_font}')

# ── Databricks credentials ──────────────────────────────────────────────
load_dotenv(dotenv_path='../../.env')
DATABRICKS_HOST      = os.getenv('DATABRICKS_HOST')
DATABRICKS_TOKEN     = os.getenv('DATABRICKS_TOKEN')
DATABRICKS_HTTP_PATH = os.getenv('DATABRICKS_HTTP_PATH')
assert all([DATABRICKS_HOST, DATABRICKS_TOKEN, DATABRICKS_HTTP_PATH]), 'Missing .env credentials'
print('✅ Credentials loaded')

# ── Query helpers ────────────────────────────────────────────────────────
def execute_query(query: str) -> pd.DataFrame:
    with sql.connect(server_hostname=DATABRICKS_HOST, http_path=DATABRICKS_HTTP_PATH,
                     access_token=DATABRICKS_TOKEN) as conn:
        with conn.cursor() as cur:
            cur.execute(query)
            result = cur.fetchall()
            columns = [d[0] for d in cur.description]
            return pd.DataFrame(result, columns=columns)

def execute_query_long(query: str) -> pd.DataFrame:
    with sql.connect(server_hostname=DATABRICKS_HOST, http_path=DATABRICKS_HTTP_PATH,
                     access_token=DATABRICKS_TOKEN,
                     _retry_stop_after_attempts_duration=3600) as conn:
        with conn.cursor() as cur:
            cur.execute(query)
            result = cur.fetchall()
            columns = [d[0] for d in cur.description]
            return pd.DataFrame(result, columns=columns)

# ── Output helpers ───────────────────────────────────────────────────────
DATA_DIR   = Path('data')
OUTPUT_DIR = Path('output_ariel')
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

def strip_tz(df):
    df = df.copy()
    for col in df.select_dtypes(include=['datetimetz']).columns:
        df[col] = df[col].dt.tz_localize(None)
    for col in df.columns:
        if df[col].dtype == 'object':
            try: df[col] = df[col].astype(str)
            except: pass
    return df

def fmt_km(v):
    if v >= 1_000_000: return f'{v/1_000_000:.1f}M'
    if v >= 1_000:     return f'{v/1_000:.1f}K'
    return str(int(v))

print('✅ Setup complete')

✅ Japanese font: MS Gothic
✅ Credentials loaded
✅ Setup complete


In [3]:
# ═══════════════════════════════════════════════════════════════════════
# 2-2. ALL Parameters (single source of truth)
# ═══════════════════════════════════════════════════════════════════════

# ── Target brands (half-width katakana per IDPOS_REFERENCE.md) ────────
BOLD_GB   = 'ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ'
ARIEL_GB   = 'ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ'
SUB_CAT     = '洗濯洗剤'
CATEGORY    = 'Laundry'

# ── Time windows ──────────────────────────────────────────────────────
LOOKBACK_START = '2024-01-01'   # Start of LAG lookback window
ANALYSIS_START = '2025-01-01'   # Analysis period start
ANALYSIS_END   = '2026-01-31'   # Analysis period end
RENEWAL_MONTH  = '2025-05-01'   # Product renewal breakpoint

# ── Canonical shopper classification windows ──────────────────────────
TRIAL_LOOKBACK_DAYS       = 365  # Trial = no purchase in prior 365 days
REPEAT_LAPSE_WINDOW_DAYS  = 180  # Repeat/Lapse = 180 days post-trial
LATEST_COHORT_END         = '2025-07-31'  # Latest cohort with 6-month follow-up
LAPSE_CUTOFF_DATE         = '2025-07-31'  # Confirm lapse after this date

# ── Retailer codes (8 national retailers) ─────────────────────────────
RETAILER_CODES = [
    'cds_8005', 'cds_8006', 'cds_8007', 'cds_8008', 'cds_8009',
    'cds_8010', 'cds_8011', 'cds_8013',
]
RETAILER_IN = ', '.join(f"'{c}'" for c in RETAILER_CODES)

# ── Size hierarchy (physical size: small → large) ─────────────────────
SIZE_ORDER     = ['本体通常', '詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ', '詰替ﾒｶﾞｼﾞｬﾝﾎﾞ', '詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ', '詰替ﾃﾗｼﾞｬﾝﾎﾞ']
EXCLUDED_SIZES = ["詰替超ｼﾞｬﾝﾎﾞ","詰替超ﾃﾗｼﾞｬﾝﾎﾞ","ｿﾉﾀ","詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ","詰替超特大","詰替通常","詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ","詰替特大"]

SIZE_ROLE = {
    '本体通常':          'Trial Entry',
    '詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ': 'Intermission', 
    '詰替ﾒｶﾞｼﾞｬﾝﾎﾞ':  'Loyalty', 
    '詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ':  'Loyalty',
    '詰替ﾃﾗｼﾞｬﾝﾎﾞ':  'Loyalty'}

# Manual capacity mapping (grams) — update from Phase 0 inspection
BOLD_GB_CAPACITY_G = {
    '本体通常':          690,
    '詰替超特大':        850,
    '詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ':  1260,
    '詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ': 1520,
}




# ── Frequency thresholds ──────────────────────────────────────────────
MIN_BIN_OBS    = 2     # weekly obs per 50 JPY bin (NB02 elasticity)
MIN_OBS_FREQ   = 10    # store×week obs per heatmap cell (NB02 heatmap)
MIN_FREQ       = 30    # shoppers per ASP band for lapse rate (NB04)
MIN_WEEK_STORE = 5     # week×store coverage for ASP band (NB04)

# ── Visualization colors ──────────────────────────────────────────────
BRAND_COLOR = {BOLD_GB: '#1E90FF', ARIEL_GB: '#FF6347'}
BRAND_LABEL = {BOLD_GB: 'Bold Gel Ball', ARIEL_GB: 'Ariel Gel Ball'}

def order_and_filter_sizes(sizes_list):
    sizes_set = set(sizes_list) - set(EXCLUDED_SIZES)
    return [s for s in SIZE_ORDER if s in sizes_set]

EXCLUDED_SIZES_SQL = ', '.join(f"'{s}'" for s in EXCLUDED_SIZES)

print('📋 Parameters loaded:')
print(f'  Analysis window : {ANALYSIS_START} → {ANALYSIS_END}')
print(f'  Lookback start  : {LOOKBACK_START}')
print(f'  Renewal month   : {RENEWAL_MONTH}')
print(f'  Cohort end      : {LATEST_COHORT_END}')
print(f'  Lapse cutoff    : {LAPSE_CUTOFF_DATE}')
print(f'  Retailers       : {len(RETAILER_CODES)}')
print(f'  Size order      : {SIZE_ORDER}')


📋 Parameters loaded:
  Analysis window : 2025-01-01 → 2026-01-31
  Lookback start  : 2024-01-01
  Renewal month   : 2025-05-01
  Cohort end      : 2025-07-31
  Lapse cutoff    : 2025-07-31
  Retailers       : 8
  Size order      : ['本体通常', '詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ', '詰替ﾒｶﾞｼﾞｬﾝﾎﾞ', '詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ', '詰替ﾃﾗｼﾞｬﾝﾎﾞ']


In [4]:
# ═══════════════════════════════════════════════════════════════════════
# 2-3. Smart Cache Layer
# ═══════════════════════════════════════════════════════════════════════
# Set to True after first successful run → all queries load from disk instantly.
# Set to False to force re-query from Databricks.

RELOAD_FROM_CACHE = True   # ← Toggle this!

_query_log = []

def cached_query(name: str, sql_str: str, force: bool = False, long: bool = False) -> pd.DataFrame:
    parquet_path = DATA_DIR / f'{name}.parquet'
    csv_path     = DATA_DIR / f'{name}.csv'

    # Check cache
    if RELOAD_FROM_CACHE and not force and parquet_path.exists():
        mod_time = datetime.fromtimestamp(parquet_path.stat().st_mtime)
        age_days = (datetime.now() - mod_time).days
        df = pd.read_parquet(parquet_path)
        status = f'📂 CACHE ({age_days}d old)'
        if age_days > 7:
            status += ' ⚠️ >7 days old'
        print(f'{status} | {name}: {len(df):,} rows | {parquet_path}')
        _query_log.append({'query': name, 'rows': len(df), 'seconds': 0, 'source': 'cache'})
        return df

    # Run query
    print(f'⏳ Querying Databricks: {name}...', flush=True)
    t0 = time.time()
    df = execute_query_long(sql_str) if long else execute_query(sql_str)
    elapsed = time.time() - t0

    # Save cache
    df.to_parquet(parquet_path, index=False)
    df.to_csv(csv_path, index=False)
    print(f'✅ {name}: {len(df):,} rows | {elapsed:.1f}s | saved to {parquet_path}')
    _query_log.append({'query': name, 'rows': len(df), 'seconds': round(elapsed, 1), 'source': 'databricks'})
    return df

print(f'Cache mode: {"RELOAD FROM CACHE" if RELOAD_FROM_CACHE else "QUERY DATABRICKS"}')
print(f'Cache dir : {DATA_DIR.resolve()}')


Cache mode: RELOAD FROM CACHE
Cache dir : C:\Users\sugimoto.k.1\OneDrive - Procter and Gamble\work\01_analysis\python\IDPOS_template - Copy\workspace\SUD_pricing_strategy_20260227\data


---
## Data Pull — 8 Active Queries (reduced from 12)

| # | Query Name | Purpose | Status |
|---|-----------|---------|--------|
| Q1 | `monthly_sales` | Monthly sales/ASP/shoppers by brand × size | ✅ Active |
| ~~Q2~~ | ~~`asp_dq_check`~~ | ~~ASP data quality~~ | 💤 Commented out |
| Q3 | `product_metadata` | Product attributes for both brands | ✅ Active |
| Q4 | `trial_cohort_full` | **Unified** trial→repeat/lapse at shopper level | ✅ Active (heaviest) |
| ~~Q5~~ | ~~`category_trial`~~ | ~~Category-level trial~~ | 💤 Commented out |
| Q6 | `asp_weekly_retailer` | Weekly × retailer × store count (absorbs Q11) | ✅ Active |
| Q7 | `asp_band_universe` | All shoppers per (size, ASP band) | ✅ Active |
| Q8 | `dual_brand_all_shoppers` | **All** shoppers + `is_lapsed` flag (absorbs Q10) | ✅ Active |
| Q9 | `dual_brand_active` | Active shoppers per size, both brands | ✅ Active |
| ~~Q10~~ | ~~`at_risk_asp`~~ | ~~At-risk shoppers~~ | ♻️ Derived from Q8 |
| ~~Q11~~ | ~~`week_store_coverage`~~ | ~~Week×store count~~ | ♻️ Derived from Q6 |
| Q12 | `dual_brand_destination` | Post-lapse destination for both brands | ✅ Active |


In [5]:
# ── Q1: Monthly Sales Aggregation (replaces NB00+NB01+NB02 monthly) ──
q1_sql = f"""
SELECT
    DATE_TRUNC('month', CAST(idpos.sales_period_group_end_date_part AS DATE)) AS month,
    prod.jp_sub_brand_alter_lang_name   AS sub_brand,
    prod.jp_segment_4_name              AS size_code,
    COUNT(DISTINCT idpos.shopper_key)   AS shoppers,
    SUM(idpos.pos_unit_sales_qty)       AS total_units,
    SUM(idpos.pos_sales_amt)            AS total_sales,
    SUM(idpos.pos_sales_amt) / SUM(idpos.pos_unit_sales_qty) AS weighted_asp
FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod
       ON idpos.prod_key = prod.prod_key
LEFT JOIN id_pos_ai_1.shopper_dim_generic_vw shopper
       ON idpos.shopper_key = shopper.shopper_key
WHERE idpos.sales_period_group_end_date_part BETWEEN '{ANALYSIS_START}' AND '{ANALYSIS_END}'
  AND idpos.data_provider_code_part IN ({RETAILER_IN})
  AND prod.jp_category_name = '{CATEGORY}'
  AND prod.jp_sub_category_alter_lang_name = '{SUB_CAT}'
  AND prod.jp_sub_brand_alter_lang_name IN ('{BOLD_GB}', '{ARIEL_GB}')
  AND shopper.member_ind = 'Y'
  AND idpos.pos_unit_sales_qty > 0
GROUP BY 1, 2, 3
ORDER BY 1, 2, 3
"""
df_monthly = cached_query('monthly_sales', q1_sql)
df_monthly['month'] = pd.to_datetime(df_monthly['month'])
for col in ['shoppers', 'total_units', 'total_sales', 'weighted_asp']:
    df_monthly[col] = pd.to_numeric(df_monthly[col])


📂 CACHE (2d old) | monthly_sales: 205 rows | data\monthly_sales.parquet


In [6]:
# # ── Q2 (commented out by user. pls comment out if later cell face an error due to this cell): ASP Data Quality Check ───────────────────────────────────────
# q2_sql = f"""
# SELECT
#     prod.jp_sub_brand_alter_lang_name                   AS sub_brand,
#     prod.jp_segment_4_name                              AS size_code,
#     COUNT(*)                                            AS total_rows,
#     SUM(CASE WHEN idpos.pos_sales_amt IS NULL THEN 1 ELSE 0 END)      AS null_sales,
#     SUM(CASE WHEN idpos.pos_unit_sales_qty IS NULL THEN 1 ELSE 0 END) AS null_units,
#     SUM(CASE WHEN idpos.pos_unit_sales_qty = 0 THEN 1 ELSE 0 END)    AS zero_units,
#     AVG(idpos.pos_sales_amt / NULLIF(idpos.pos_unit_sales_qty, 0))    AS avg_asp,
#     PERCENTILE_APPROX(idpos.pos_sales_amt / NULLIF(idpos.pos_unit_sales_qty, 0), 0.5) AS median_asp,
#     MIN(idpos.pos_sales_amt / NULLIF(idpos.pos_unit_sales_qty, 0))    AS min_asp,
#     MAX(idpos.pos_sales_amt / NULLIF(idpos.pos_unit_sales_qty, 0))    AS max_asp
# FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
# LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod
#        ON idpos.prod_key = prod.prod_key
# LEFT JOIN id_pos_ai_1.shopper_dim_generic_vw shopper
#        ON idpos.shopper_key = shopper.shopper_key
# WHERE idpos.sales_period_group_end_date_part BETWEEN '{ANALYSIS_START}' AND '{ANALYSIS_END}'
#   AND idpos.data_provider_code_part IN ({RETAILER_IN})
#   AND prod.jp_category_name = '{CATEGORY}'
#   AND prod.jp_sub_category_alter_lang_name = '{SUB_CAT}'
#   AND prod.jp_sub_brand_alter_lang_name IN ('{BOLD_GB}', '{ARIEL_GB}')
#   AND shopper.member_ind = 'Y'
# GROUP BY 1, 2
# ORDER BY 1, 7 DESC
# """
# df_asp_dq = cached_query('asp_dq_check', q2_sql)
# for col in df_asp_dq.columns[2:]:
#     df_asp_dq[col] = pd.to_numeric(df_asp_dq[col])


In [7]:
# ── Q3: Product Metadata (replaces NB00 prod_family + capacity) ───────
q3_sql = f"""
SELECT DISTINCT
    prod.jp_sub_brand_alter_lang_name AS sub_brand,
    prod.jp_prod_family_1_name        AS prod_family,
    prod.jp_segment_4_name            AS size_code,
    prod.jp_prod_form_name            AS prod_form,
    prod.jp_prod_alter_lang_name      AS product_name,
    prod.jp_size_name                 AS size_name,
    prod.jp_pack_size_name            AS pack_size_name
FROM id_pos_ai_1.prod_dim_ext_vw prod
WHERE prod.jp_sub_brand_alter_lang_name IN ('{BOLD_GB}', '{ARIEL_GB}')
  AND prod.jp_sub_category_alter_lang_name = '{SUB_CAT}'
ORDER BY 1, 2, 3
"""
df_product = cached_query('product_metadata', q3_sql)


📂 CACHE (2d old) | product_metadata: 871 rows | data\product_metadata.parquet


In [8]:
# ── Q4: Unified Trial → Repeat/Lapse Cohort (THE BIG ONE) ────────────
# Replaces NB02 subbrand_trial + NB03 cohort + NB05 journey
# Returns shopper-level trial→outcome for BOTH brands
q4_sql = f"""
WITH all_purchases AS (
    SELECT
        idpos.shopper_key,
        prod.jp_sub_brand_alter_lang_name AS sub_brand,
        prod.jp_segment_4_name            AS size_code,
        CAST(idpos.sales_period_group_end_date_part AS DATE) AS purchase_date,
        SUM(idpos.pos_sales_amt) / SUM(idpos.pos_unit_sales_qty) AS asp
    FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
    LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod
           ON idpos.prod_key = prod.prod_key
    LEFT JOIN id_pos_ai_1.shopper_dim_generic_vw shopper
           ON idpos.shopper_key = shopper.shopper_key
    WHERE idpos.sales_period_group_end_date_part BETWEEN '{LOOKBACK_START}' AND '{ANALYSIS_END}'
      AND idpos.data_provider_code_part IN ({RETAILER_IN})
      AND prod.jp_category_name = '{CATEGORY}'
      AND prod.jp_sub_category_alter_lang_name = '{SUB_CAT}'
      AND prod.jp_sub_brand_alter_lang_name IN ('{BOLD_GB}', '{ARIEL_GB}')
      AND shopper.member_ind = 'Y'
      AND idpos.pos_unit_sales_qty > 0
    GROUP BY 1, 2, 3, 4
),
with_prev AS (
    SELECT *,
        LAG(purchase_date) OVER (
            PARTITION BY shopper_key, sub_brand ORDER BY purchase_date
        ) AS prev_purchase_date
    FROM all_purchases
),
trial_candidates AS (
    SELECT *
    FROM with_prev
    WHERE purchase_date BETWEEN '{ANALYSIS_START}' AND '{LATEST_COHORT_END}'
      AND (prev_purchase_date IS NULL
           OR DATEDIFF(purchase_date, prev_purchase_date) > {TRIAL_LOOKBACK_DAYS})
),
trial_events AS (
    SELECT shopper_key, sub_brand, MIN(purchase_date) AS trial_date
    FROM trial_candidates
    GROUP BY 1, 2
),
trial_detail AS (
    SELECT te.shopper_key, te.sub_brand, te.trial_date,
           tc.size_code AS trial_size, tc.asp AS trial_asp
    FROM trial_events te
    INNER JOIN trial_candidates tc
           ON te.shopper_key = tc.shopper_key
          AND te.sub_brand   = tc.sub_brand
          AND te.trial_date  = tc.purchase_date
),
repeat_events AS (
    SELECT
        td.shopper_key, td.sub_brand, td.trial_date, td.trial_size, td.trial_asp,
        MIN(ap.purchase_date) AS repeat_date,
        MIN(ap.size_code)     AS repeat_size,
        MIN(ap.asp)           AS repeat_asp
    FROM trial_detail td
    LEFT JOIN all_purchases ap
           ON td.shopper_key = ap.shopper_key
          AND td.sub_brand   = ap.sub_brand
          AND ap.purchase_date > td.trial_date
          AND ap.purchase_date <= DATE_ADD(td.trial_date, {REPEAT_LAPSE_WINDOW_DAYS})
    GROUP BY 1, 2, 3, 4, 5
)
SELECT
    re.shopper_key, re.sub_brand, re.trial_date, re.trial_size, re.trial_asp,
    re.repeat_date, re.repeat_size, re.repeat_asp,
    CASE WHEN re.repeat_date IS NOT NULL THEN 'Repeat' ELSE 'Lapse' END AS outcome,
    DATEDIFF(re.repeat_date, re.trial_date) AS days_to_repeat
FROM repeat_events re
ORDER BY re.sub_brand, re.trial_date
"""
df_cohort = cached_query('trial_cohort_full', q4_sql, long=True)
df_cohort['trial_date']  = pd.to_datetime(df_cohort['trial_date'])
df_cohort['repeat_date'] = pd.to_datetime(df_cohort['repeat_date'])
for col in ['trial_asp', 'repeat_asp', 'days_to_repeat']:
    df_cohort[col] = pd.to_numeric(df_cohort[col])
print(f'  Bold: {len(df_cohort[df_cohort["sub_brand"]==BOLD_GB]):,} | '
      f'Ariel: {len(df_cohort[df_cohort["sub_brand"]==ARIEL_GB]):,}')


📂 CACHE (2d old) | trial_cohort_full: 1,135,036 rows | data\trial_cohort_full.parquet
  Bold: 606,038 | Ariel: 528,998


In [9]:
# # ── Q5 (commented out by user. pls revise the query if later cell face an error due to this cell): Category Trial (category-level LAG) ──────────────────────────
# q5_sql = f"""
# WITH all_category_purchases AS (
#     SELECT
#         idpos.shopper_key,
#         prod.jp_sub_brand_alter_lang_name AS sub_brand,
#         prod.jp_segment_4_name            AS size_code,
#         CAST(idpos.sales_period_group_end_date_part AS DATE) AS purchase_date
#     FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
#     LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod ON idpos.prod_key = prod.prod_key
#     LEFT JOIN id_pos_ai_1.shopper_dim_generic_vw shopper ON idpos.shopper_key = shopper.shopper_key
#     WHERE idpos.sales_period_group_end_date_part BETWEEN '{LOOKBACK_START}' AND '{ANALYSIS_END}'
#       AND idpos.data_provider_code_part IN ({RETAILER_IN})
#       AND prod.jp_category_name = '{CATEGORY}'
#       AND prod.jp_sub_category_alter_lang_name = '{SUB_CAT}'
#       AND shopper.member_ind = 'Y'
#       AND idpos.pos_unit_sales_qty > 0
#     GROUP BY 1, 2, 3, 4
# ),
# with_prev_cat AS (
#     SELECT *, LAG(purchase_date) OVER (PARTITION BY shopper_key ORDER BY purchase_date) AS prev_cat_date
#     FROM all_category_purchases
# ),
# cat_trial_candidates AS (
#     SELECT * FROM with_prev_cat
#     WHERE purchase_date BETWEEN '{ANALYSIS_START}' AND '{ANALYSIS_END}'
#       AND (prev_cat_date IS NULL OR DATEDIFF(purchase_date, prev_cat_date) > 365)
#       AND sub_brand IN ('{BOLD_GB}', '{ARIEL_GB}')
# ),
# cat_trial_events AS (
#     SELECT shopper_key, MIN(purchase_date) AS cat_trial_date FROM cat_trial_candidates GROUP BY 1
# ),
# cat_trial_detail AS (
#     SELECT cte.shopper_key, ctc.sub_brand, ctc.size_code, cte.cat_trial_date AS purchase_date
#     FROM cat_trial_events cte
#     INNER JOIN cat_trial_candidates ctc
#            ON cte.shopper_key = ctc.shopper_key AND cte.cat_trial_date = ctc.purchase_date
# )
# SELECT DATE_TRUNC('month', ctd.purchase_date) AS trial_month, ctd.sub_brand, ctd.size_code,
#        COUNT(DISTINCT ctd.shopper_key) AS category_trial_shoppers
# FROM cat_trial_detail ctd
# GROUP BY 1, 2, 3
# ORDER BY 1, 2, 3
# """
# df_cat_trial = cached_query('category_trial', q5_sql)
# df_cat_trial['trial_month'] = pd.to_datetime(df_cat_trial['trial_month'])
# df_cat_trial['category_trial_shoppers'] = pd.to_numeric(df_cat_trial['category_trial_shoppers'])


In [10]:
# ── Q6: Weekly Retailer-Level ASP + Units + Store Count (absorbs Q11) ─
# Added n_stores = COUNT(DISTINCT site_key) to derive week×store coverage
# in Python, eliminating the separate Q11 query.
q6_sql = f"""
SELECT
    CAST(idpos.sales_period_group_end_date_part AS DATE) AS week_end,
    idpos.data_provider_code_part AS retailer,
    prod.jp_sub_brand_alter_lang_name   AS sub_brand,
    prod.jp_segment_4_name              AS size_code,
    SUM(idpos.pos_sales_amt) / SUM(idpos.pos_unit_sales_qty) AS weighted_asp,
    SUM(idpos.pos_unit_sales_qty) AS total_units,
    COUNT(DISTINCT idpos.shopper_key) AS buyer_count,
    COUNT(DISTINCT idpos.site_key) AS n_stores
FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod ON idpos.prod_key = prod.prod_key
LEFT JOIN id_pos_ai_1.shopper_dim_generic_vw shopper ON idpos.shopper_key = shopper.shopper_key
WHERE idpos.sales_period_group_end_date_part BETWEEN '{ANALYSIS_START}' AND '{ANALYSIS_END}'
  AND idpos.data_provider_code_part IN ({RETAILER_IN})
  AND prod.jp_category_name = '{CATEGORY}'
  AND prod.jp_sub_category_alter_lang_name = '{SUB_CAT}'
  AND prod.jp_sub_brand_alter_lang_name IN ('{BOLD_GB}', '{ARIEL_GB}')
  AND shopper.member_ind = 'Y'
  AND idpos.pos_unit_sales_qty > 0
GROUP BY 1, 2, 3, 4
ORDER BY 1, 2, 3, 4
"""
df_asp_weekly = cached_query('asp_weekly_retailer', q6_sql)
df_asp_weekly['week_end'] = pd.to_datetime(df_asp_weekly['week_end'])

for c in ['weighted_asp', 'total_units', 'buyer_count', 'n_stores']:
    df_asp_weekly[c] = pd.to_numeric(df_asp_weekly[c])

📂 CACHE (2d old) | asp_weekly_retailer: 28,040 rows | data\asp_weekly_retailer.parquet


In [11]:
# ── Q7: ASP Band Universe (all Ariel shoppers per size × ASP band) ───
q7_sql = f"""
WITH txn_asp AS (
    SELECT idpos.shopper_key, prod.jp_segment_4_name AS trial_size,
        CAST(FLOOR((SUM(idpos.pos_sales_amt)/SUM(idpos.pos_unit_sales_qty))/50)*50 AS BIGINT) AS asp_band
    FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
    LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod ON idpos.prod_key = prod.prod_key
    LEFT JOIN id_pos_ai_1.shopper_dim_generic_vw shopper ON idpos.shopper_key = shopper.shopper_key
    WHERE idpos.sales_period_group_end_date_part BETWEEN '{ANALYSIS_START}' AND '{ANALYSIS_END}'
      AND idpos.data_provider_code_part IN ({RETAILER_IN})
      AND prod.jp_category_name = '{CATEGORY}' AND prod.jp_sub_category_alter_lang_name = '{SUB_CAT}'
      AND prod.jp_sub_brand_alter_lang_name = '{ARIEL_GB}'
      AND prod.jp_segment_4_name NOT IN ({EXCLUDED_SIZES_SQL})
      AND shopper.member_ind = 'Y' AND idpos.pos_unit_sales_qty > 0
    GROUP BY idpos.shopper_key, prod.jp_segment_4_name, idpos.sales_period_group_end_date_part
)
SELECT trial_size, asp_band, COUNT(DISTINCT shopper_key) AS all_shoppers
FROM txn_asp GROUP BY trial_size, asp_band
"""
df_universe = cached_query('ariel_asp_band_universe', q7_sql)
df_universe['asp_band']     = pd.to_numeric(df_universe['asp_band'])
df_universe['all_shoppers'] = pd.to_numeric(df_universe['all_shoppers'])

📂 CACHE (1d old) | ariel_asp_band_universe: 231 rows | data\ariel_asp_band_universe.parquet


In [12]:
# ── Q8: Dual-Brand ALL Shoppers + is_lapsed flag (absorbs Q10) ────────
# Returns ALL shoppers (active + lapsed) with their last purchase details.
# df_lapsed is derived in Python as a filter. Eliminates separate Q10 query.
q8_sql = f"""
WITH brand_purchases AS (
    SELECT idpos.shopper_key, prod.jp_sub_brand_alter_lang_name AS sub_brand,
        prod.jp_segment_4_name AS size_code,
        CAST(idpos.sales_period_group_end_date_part AS DATE) AS purchase_date,
        SUM(idpos.pos_sales_amt)/SUM(idpos.pos_unit_sales_qty) AS asp
    FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
    LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod ON idpos.prod_key = prod.prod_key
    LEFT JOIN id_pos_ai_1.shopper_dim_generic_vw shopper ON idpos.shopper_key = shopper.shopper_key
    WHERE idpos.sales_period_group_end_date_part BETWEEN '{ANALYSIS_START}' AND '{ANALYSIS_END}'
      AND idpos.data_provider_code_part IN ({RETAILER_IN})
      AND prod.jp_category_name = '{CATEGORY}' AND prod.jp_sub_category_alter_lang_name = '{SUB_CAT}'
      AND prod.jp_sub_brand_alter_lang_name IN ('{BOLD_GB}', '{ARIEL_GB}')
      AND shopper.member_ind = 'Y' AND idpos.pos_unit_sales_qty > 0
    GROUP BY 1, 2, 3, 4
),
last_purchase AS (
    SELECT shopper_key, sub_brand, MAX(purchase_date) AS last_date
    FROM brand_purchases GROUP BY 1, 2
    HAVING MAX(purchase_date) <= '{LAPSE_CUTOFF_DATE}'
),
lapse_check AS (
    SELECT lp.shopper_key, lp.sub_brand, lp.last_date,
        MAX(CASE WHEN bp.purchase_date > lp.last_date
                  AND bp.purchase_date <= DATE_ADD(lp.last_date, {REPEAT_LAPSE_WINDOW_DAYS})
                 THEN 1 ELSE 0 END) AS returned
    FROM last_purchase lp
    LEFT JOIN brand_purchases bp ON lp.shopper_key = bp.shopper_key AND lp.sub_brand = bp.sub_brand
        AND bp.purchase_date > lp.last_date
    GROUP BY 1, 2, 3
)
SELECT lc.shopper_key, lc.sub_brand, lc.last_date AS last_purchase_date,
       bp.size_code AS last_size, bp.asp AS last_asp,
       CASE WHEN lc.returned = 0 THEN 1 ELSE 0 END AS is_lapsed
FROM lapse_check lc
INNER JOIN brand_purchases bp ON lc.shopper_key = bp.shopper_key
    AND lc.sub_brand = bp.sub_brand AND lc.last_date = bp.purchase_date
"""
df_all_shoppers = cached_query('dual_brand_all_shoppers', q8_sql)
df_all_shoppers['last_purchase_date'] = pd.to_datetime(df_all_shoppers['last_purchase_date'])
df_all_shoppers['last_asp'] = pd.to_numeric(df_all_shoppers['last_asp'])
df_all_shoppers['is_lapsed'] = pd.to_numeric(df_all_shoppers['is_lapsed']).astype(int)

# Backward-compatible df_lapsed = only lapsed shoppers
df_lapsed = df_all_shoppers[df_all_shoppers['is_lapsed'] == 1].copy()
print(f'  All shoppers: {len(df_all_shoppers):,}')
print(f'  Bold lapsed: {len(df_lapsed[df_lapsed["sub_brand"]==BOLD_GB]):,}')
print(f'  Ariel lapsed: {len(df_lapsed[df_lapsed["sub_brand"]==ARIEL_GB]):,}')

📂 CACHE (2d old) | dual_brand_all_shoppers: 1,229,149 rows | data\dual_brand_all_shoppers.parquet
  All shoppers: 1,229,149
  Bold lapsed: 635,923
  Ariel lapsed: 593,226


In [13]:
# ── Q9: Dual-Brand Active Shoppers per Size ───────────────────────────
q9_sql = f"""
SELECT prod.jp_sub_brand_alter_lang_name AS sub_brand, prod.jp_segment_4_name AS size_code,
       COUNT(DISTINCT idpos.shopper_key) AS active_shoppers
FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod ON idpos.prod_key = prod.prod_key
LEFT JOIN id_pos_ai_1.shopper_dim_generic_vw shopper ON idpos.shopper_key = shopper.shopper_key
WHERE idpos.sales_period_group_end_date_part BETWEEN '{ANALYSIS_START}' AND '{LAPSE_CUTOFF_DATE}'
  AND idpos.data_provider_code_part IN ({RETAILER_IN})
  AND prod.jp_category_name = '{CATEGORY}' AND prod.jp_sub_category_alter_lang_name = '{SUB_CAT}'
  AND prod.jp_sub_brand_alter_lang_name IN ('{BOLD_GB}', '{ARIEL_GB}')
  AND shopper.member_ind = 'Y' AND idpos.pos_unit_sales_qty > 0
GROUP BY 1, 2
"""
df_active = cached_query('dual_brand_active', q9_sql)
df_active['active_shoppers'] = pd.to_numeric(df_active['active_shoppers'])


📂 CACHE (2d old) | dual_brand_active: 18 rows | data\dual_brand_active.parquet


In [14]:
# ── Q10: REMOVED — derived from Q8 (df_all_shoppers) ─────────────────
# Q10 was: all Ariel shoppers with last ASP + is_lapsed flag.
# Now derived from expanded Q8 which returns ALL shoppers + is_lapsed.
df_at_risk = df_all_shoppers[df_all_shoppers['sub_brand'] == ARIEL_GB][[
    'shopper_key', 'last_purchase_date', 'last_size', 'last_asp', 'is_lapsed'
]].rename(columns={'last_size': 'size_code', 'last_asp': 'asp'}).copy()
print(f'♻️ Q10 derived from Q8: {len(df_at_risk):,} Ariel shoppers | {df_at_risk["is_lapsed"].sum():,} lapsed')


♻️ Q10 derived from Q8: 593,226 Ariel shoppers | 593,226 lapsed


In [15]:
# ── Q11: REMOVED — derived from Q6 (df_asp_weekly + n_stores) ─────────
# Q11 was: (size, asp_band) → week_store_count.
# Now derived from Q6 which includes n_stores per (week, retailer, brand, size).
# We compute: for each (size, asp_band), sum n_stores across all weeks × retailers.
_q6_ariel = df_asp_weekly[
    (df_asp_weekly['sub_brand'] == ARIEL_GB) &
    (df_asp_weekly['week_end'] <= pd.Timestamp(LAPSE_CUTOFF_DATE))
].copy()
_q6_ariel['asp_band'] = (_q6_ariel['weighted_asp'] // 50 * 50).astype(int)
df_week_store = _q6_ariel.groupby(['size_code', 'asp_band']).agg(
    week_store_count=('n_stores', 'sum')
).reset_index()
df_week_store['asp_band'] = df_week_store['asp_band'].astype(int)
df_week_store['week_store_count'] = df_week_store['week_store_count'].astype(int)
print(f'♻️ Q11 derived from Q6: {len(df_week_store):,} (size × asp_band) combinations')


♻️ Q11 derived from Q6: 142 (size × asp_band) combinations


In [16]:
# ── Q12: Dual-Brand Post-Lapse Destination ────────────────────────────
q12_sql = f"""
WITH brand_last AS (
    SELECT idpos.shopper_key, prod.jp_sub_brand_alter_lang_name AS source_brand,
        MAX(CAST(idpos.sales_period_group_end_date_part AS DATE)) AS last_date
    FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
    INNER JOIN id_pos_ai_1.prod_dim_ext_vw prod ON idpos.prod_key = prod.prod_key
    INNER JOIN id_pos_ai_1.shopper_dim_generic_vw shopper ON idpos.shopper_key = shopper.shopper_key
    WHERE idpos.sales_period_group_end_date_part BETWEEN '{ANALYSIS_START}' AND '{ANALYSIS_END}'
      AND idpos.data_provider_code_part IN ({RETAILER_IN})
      AND prod.jp_category_name = '{CATEGORY}' AND prod.jp_sub_category_alter_lang_name = '{SUB_CAT}'
      AND prod.jp_sub_brand_alter_lang_name IN ('{BOLD_GB}', '{ARIEL_GB}')
      AND shopper.member_ind = 'Y' AND idpos.pos_unit_sales_qty > 0
    GROUP BY 1, 2
    HAVING MAX(CAST(idpos.sales_period_group_end_date_part AS DATE)) <= DATE('{LAPSE_CUTOFF_DATE}')
),
next_laundry AS (
    SELECT bl.shopper_key, bl.source_brand,
        prod.jp_sub_brand_alter_lang_name AS next_sub_brand,
        prod.jp_segment_4_name            AS next_size,
        ROW_NUMBER() OVER (PARTITION BY bl.shopper_key, bl.source_brand
                           ORDER BY CAST(idpos.sales_period_group_end_date_part AS DATE)) AS rn
    FROM brand_last bl
    INNER JOIN cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
           ON bl.shopper_key = idpos.shopper_key
    INNER JOIN id_pos_ai_1.prod_dim_ext_vw prod ON idpos.prod_key = prod.prod_key
    WHERE CAST(idpos.sales_period_group_end_date_part AS DATE) > bl.last_date
      AND idpos.sales_period_group_end_date_part <= '{ANALYSIS_END}'
      AND idpos.data_provider_code_part IN ({RETAILER_IN})
      AND prod.jp_category_name = '{CATEGORY}' AND prod.jp_sub_category_alter_lang_name = '{SUB_CAT}'
      AND idpos.pos_unit_sales_qty > 0
)
SELECT shopper_key, source_brand, next_sub_brand, next_size
FROM next_laundry WHERE rn = 1
"""
df_destination = cached_query('dual_brand_destination', q12_sql, long=True)
print(f'  Bold→dest: {len(df_destination[df_destination["source_brand"]==BOLD_GB]):,}')
print(f'  Ariel→dest: {len(df_destination[df_destination["source_brand"]==ARIEL_GB]):,}')


📂 CACHE (2d old) | dual_brand_destination: 640,033 rows | data\dual_brand_destination.parquet
  Bold→dest: 325,821
  Ariel→dest: 314,212


In [17]:
# ── Data Pull Summary ─────────────────────────────────────────────────
print('\n' + '=' * 70)
print('DATA PULL SUMMARY')
print('=' * 70)
summary_df = pd.DataFrame(_query_log)
print(summary_df.to_string(index=False))
total_rows = summary_df['rows'].sum()
total_secs = summary_df['seconds'].sum()
cache_pct  = (summary_df['source'] == 'cache').sum() / len(summary_df) * 100
print(f'\nTotal: {total_rows:,} rows | {total_secs:.0f}s elapsed | {cache_pct:.0f}% from cache')



DATA PULL SUMMARY
                  query    rows  seconds source
          monthly_sales     205        0  cache
       product_metadata     871        0  cache
      trial_cohort_full 1135036        0  cache
    asp_weekly_retailer   28040        0  cache
ariel_asp_band_universe     231        0  cache
dual_brand_all_shoppers 1229149        0  cache
      dual_brand_active      18        0  cache
 dual_brand_destination  640033        0  cache

Total: 3,033,583 rows | 0s elapsed | 100% from cache


---
# Pane A — Total Shopper & ASP Landscape
*Market sizing, pricing trends, and renewal impact across all sizes.*


In [18]:
# ── A-1: Monthly Sales Summary Table ──────────────────────────────────
df_asp = df_monthly[~df_monthly['size_code'].isin(EXCLUDED_SIZES)].copy()
renewal_date = pd.Timestamp(RENEWAL_MONTH)

# Summary table per brand × size
size_summary = df_asp.groupby(['sub_brand', 'size_code']).agg(
    total_shoppers=('shoppers', 'sum'),
    total_units=('total_units', 'sum'),
    total_sales=('total_sales', 'sum'),
).reset_index()
size_summary['weighted_asp'] = (size_summary['total_sales'] / size_summary['total_units']).round(1)
size_summary['sales_share_%'] = size_summary.groupby('sub_brand')['total_sales'].transform(
    lambda x: (x / x.sum() * 100).round(1)
)

print('📊 DATA TABLE: Size Summary by Brand')
print('=' * 90)
for brand in [BOLD_GB, ARIEL_GB]:
    print(f'\n▶ {BRAND_LABEL.get(brand, brand)}')
    b = size_summary[size_summary['sub_brand'] == brand].copy()
    b['size_code'] = pd.Categorical(b['size_code'],
        categories=order_and_filter_sizes(b['size_code']), ordered=True)
    b = b.sort_values('size_code')
    print(b[['size_code', 'total_shoppers', 'total_units', 'total_sales', 'weighted_asp', 'sales_share_%']].to_string(index=False))

# Save table
size_summary.to_csv(OUTPUT_DIR / 'pane_a_size_summary.csv', index=False)


📊 DATA TABLE: Size Summary by Brand

▶ Bold Gel Ball
    size_code  total_shoppers  total_units     total_sales  weighted_asp  sales_share_%
         本体通常          864883  1,549,928.0   391,028,566.0         252.3            8.1
詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ         1362482  1,819,825.0 1,656,843,474.0         910.4           34.1
  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ          413674    489,851.0   891,948,231.0       1,820.9           18.4
 詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ          397210    519,975.0 1,240,702,805.0       2,386.1           25.6
   詰替ﾃﾗｼﾞｬﾝﾎﾞ          195984    230,150.0   674,463,474.0       2,930.5           13.9

▶ Ariel Gel Ball
    size_code  total_shoppers  total_units     total_sales  weighted_asp  sales_share_%
         本体通常          606503  1,127,080.0   275,965,584.0         244.9            5.5
詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ         1275593  1,692,880.0 1,558,195,176.0         920.4           31.0
  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ          486042    575,691.0 1,061,432,648.0       1,843.8           21.1
 詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ          432524    558,659.0 1,313,

In [19]:
# ── A-2: Combined Bold vs Ariel ASP Trend per Size ──────────────────
combined_data = df_asp.copy()
combined_sizes = order_and_filter_sizes(combined_data['size_code'].unique())

if combined_sizes:
    n_s = len(combined_sizes)
    n_c = 2
    n_r = (n_s + n_c - 1) // n_c

    fig_comb = make_subplots(rows=n_r, cols=n_c, subplot_titles=combined_sizes,
                             vertical_spacing=0.1, horizontal_spacing=0.1)

    for idx, size in enumerate(combined_sizes):
        r, c = idx // n_c + 1, idx % n_c + 1
        for brand, cfg in {BOLD_GB: ('#1E90FF', 'solid', 'Bold Gel Ball'),
                           ARIEL_GB: ('#FF6347', 'dot', 'Ariel Gel Ball')}.items():
            sub = combined_data[(combined_data['sub_brand']==brand) & (combined_data['size_code']==size)].sort_values('month')
            if len(sub) == 0: continue
            fig_comb.add_trace(go.Scatter(
                x=sub['month'], y=sub['weighted_asp'], mode='lines+markers',
                name=cfg[2], line=dict(color=cfg[0], dash=cfg[1]), marker=dict(size=5),
                showlegend=(idx==0)), row=r, col=c)
        fig_comb.add_vline(x=RENEWAL_MONTH, line_dash='dash', line_color='gray', opacity=0.4, row=r, col=c)

    fig_comb.update_layout(height=320*n_r, title_text='Bold vs Ariel — Monthly ASP per Size',
                           template='plotly_white', legend=dict(orientation='h', yanchor='bottom', y=-0.08))
    fig_comb.update_yaxes(title_text='ASP (JPY)')
    fig_comb.show()

# ── DATA TABLE: Monthly ASP per Brand × Size ─────────────────────────
asp_table = combined_data[combined_data['size_code'].isin(combined_sizes)].pivot_table(
    index=['sub_brand', 'size_code'], columns='month', values='weighted_asp', aggfunc='mean'
).round(0)
print('\n📊 DATA TABLE: Monthly ASP — Bold vs Ariel per Size (JPY)')
print('=' * 120)
print(asp_table.to_string())
asp_table.reset_index().to_csv(OUTPUT_DIR / 'pane_a_asp_trend.csv', index=False)


📊 DATA TABLE: Monthly ASP — Bold vs Ariel per Size (JPY)
month                         2025-01-01 00:00:00+00:00  2025-02-01 00:00:00+00:00  2025-03-01 00:00:00+00:00  2025-04-01 00:00:00+00:00  2025-05-01 00:00:00+00:00  2025-06-01 00:00:00+00:00  2025-07-01 00:00:00+00:00  2025-08-01 00:00:00+00:00  2025-09-01 00:00:00+00:00  2025-10-01 00:00:00+00:00  2025-11-01 00:00:00+00:00  2025-12-01 00:00:00+00:00  2026-01-01 00:00:00+00:00
sub_brand      size_code                                                                                                                                                                                                                                                                                                                                                                   
ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ  本体通常                               321.0                      223.0                      191.0                      241.0                      302.0                   

In [20]:
# # ── A-3 (commented out by user): Pre vs Post Renewal Comparison ───────────────────────────────
# df_asp['period'] = df_asp['month'].apply(lambda x: 'Pre-Renewal' if x < renewal_date else 'Post-Renewal')

# comparison = df_asp.groupby(['sub_brand', 'size_code', 'period']).agg(
#     total_sales=('total_sales', 'sum'), total_units=('total_units', 'sum'),
#     avg_shoppers_per_month=('shoppers', 'mean'),
# ).reset_index()
# comparison['weighted_asp'] = comparison['total_sales'] / comparison['total_units']

# flat = comparison.pivot_table(index=['sub_brand', 'size_code'], columns='period',
#                               values='weighted_asp', aggfunc='first').reset_index()
# if 'Pre-Renewal' in flat.columns and 'Post-Renewal' in flat.columns:
#     flat['asp_change_%'] = ((flat['Post-Renewal'] - flat['Pre-Renewal']) / flat['Pre-Renewal'] * 100).round(1)
#     flat['asp_change_jpy'] = (flat['Post-Renewal'] - flat['Pre-Renewal']).round(0)

# print('📊 DATA TABLE: Pre vs Post Renewal ASP')
# print('=' * 80)
# for brand in [BOLD_GB, ARIEL_GB]:
#     print(f'\n▶ {BRAND_LABEL.get(brand, brand)}')
#     print(flat[flat['sub_brand'] == brand].to_string(index=False))

# # Grouped bar chart
# if 'Pre-Renewal' in flat.columns and 'Post-Renewal' in flat.columns:
#     plot_data = flat.copy()
#     plot_data['label'] = plot_data.apply(
#         lambda r: BRAND_LABEL.get(r['sub_brand'], r['sub_brand']) + ' | ' + r['size_code'], axis=1)
#     fig3 = go.Figure()
#     fig3.add_trace(go.Bar(name='Pre-Renewal', x=plot_data['label'], y=plot_data['Pre-Renewal'], marker_color='#6495ED'))
#     fig3.add_trace(go.Bar(name='Post-Renewal', x=plot_data['label'], y=plot_data['Post-Renewal'], marker_color='#FF6347'))
#     fig3.update_layout(barmode='group', title='ASP Before vs After Renewal', yaxis_title='Weighted ASP (JPY)',
#                        template='plotly_white', height=500)
#     fig3.show()

# flat.to_csv(OUTPUT_DIR / 'pane_a_pre_post_renewal.csv', index=False)


In [21]:
# ── A-4: Unit Ratio vs ASP Gap per Size ───────────────────────────────
_ug = df_asp.copy()
_plot_sizes = order_and_filter_sizes(_ug['size_code'].unique())
_units = _ug.groupby(['month', 'sub_brand', 'size_code'])['total_units'].sum().reset_index()
_asp_m = _ug.groupby(['month', 'sub_brand', 'size_code']).apply(
    lambda d: d['total_sales'].sum() / d['total_units'].sum()).reset_index(name='weighted_asp')

n_s = len(_plot_sizes); n_c = 2; n_r = (n_s + n_c - 1) // n_c
_specs = [[{"secondary_y": True}, {"secondary_y": True}] for _ in range(n_r)]
fig_ug = make_subplots(rows=n_r, cols=n_c, subplot_titles=_plot_sizes, specs=_specs,
                       vertical_spacing=0.14, horizontal_spacing=0.12)

ratio_gap_rows = []
for idx, size in enumerate(_plot_sizes):
    r, c = idx // n_c + 1, idx % n_c + 1
    _ariel_u = _units[(_units['sub_brand']==ARIEL_GB) & (_units['size_code']==size)][['month','total_units']].rename(columns={'total_units':'ariel_units'})
    _bold_u = _units[(_units['sub_brand']==BOLD_GB) & (_units['size_code']==size)][['month','total_units']].rename(columns={'total_units':'bold_units'})
    _ratio = pd.merge(_ariel_u, _bold_u, on='month', how='inner').sort_values('month')
    _ratio['unit_ratio'] = (_ratio['ariel_units'] / _ratio['bold_units'] * 100).round(1)

    if len(_ratio) > 0:
        fig_ug.add_trace(go.Scatter(x=_ratio['month'], y=_ratio['unit_ratio'], mode='lines+markers',
            name='Unit Ratio (%)', line=dict(color='#1E90FF', width=2.5),
            marker=dict(size=7, color=['#1E90FF' if v>=100 else '#FF6347' for v in _ratio['unit_ratio']]),
            showlegend=(idx==0), legendgroup='ratio'), row=r, col=c, secondary_y=False)
        fig_ug.add_hline(y=100, line_dash='dot', line_color='#888', opacity=0.7, row=r, col=c)

    _ariel_a = _asp_m[(_asp_m['sub_brand']==ARIEL_GB) & (_asp_m['size_code']==size)][['month','weighted_asp']].rename(columns={'weighted_asp':'ariel_asp'})
    _bold_a = _asp_m[(_asp_m['sub_brand']==BOLD_GB) & (_asp_m['size_code']==size)][['month','weighted_asp']].rename(columns={'weighted_asp':'bold_asp'})
    _gap = pd.merge(_ariel_a, _bold_a, on='month', how='inner').sort_values('month')
    _gap['asp_gap'] = (_gap['ariel_asp'] - _gap['bold_asp']).round(0)

    if len(_gap) > 0:
        fig_ug.add_trace(go.Bar(x=_gap['month'], y=_gap['asp_gap'], name='ASP Gap (¥)',
            marker_color=['#FFB347' if v>=0 else '#66CDAA' for v in _gap['asp_gap']], opacity=0.4,
            showlegend=(idx==0), legendgroup='gap'), row=r, col=c, secondary_y=True)

    fig_ug.add_vline(x=RENEWAL_MONTH, line_dash='dash', line_color='gray', opacity=0.4, row=r, col=c)
    fig_ug.update_yaxes(title_text='Unit Ratio (%)', secondary_y=False, row=r, col=c)
    fig_ug.update_yaxes(title_text='ASP Gap (¥)', secondary_y=True, row=r, col=c)

    # Collect table data
    _merged = _ratio.merge(_gap[['month','ariel_asp','bold_asp','asp_gap']], on='month', how='outer')
    _merged['size_code'] = size
    ratio_gap_rows.append(_merged)

fig_ug.update_layout(height=380*n_r, title_text='Ariel / Bold Unit Ratio vs ASP Gap — by Size',
                     template='plotly_white', barmode='overlay',
                     legend=dict(orientation='h', yanchor='bottom', y=-0.06, x=0))
fig_ug.show()

# ── DATA TABLE: Unit Ratio & ASP Gap ─────────────────────────────────
if ratio_gap_rows:
    ratio_gap_df = pd.concat(ratio_gap_rows, ignore_index=True)
    ratio_gap_df = ratio_gap_df[['size_code','month','ariel_units','bold_units','unit_ratio','ariel_asp','bold_asp','asp_gap']]
    ratio_gap_df = ratio_gap_df.sort_values(['size_code','month'])
    print('\n📊 DATA TABLE: Unit Ratio & ASP Gap — by Size × Month')
    print('=' * 120)
    print(ratio_gap_df.to_string(index=False))
    ratio_gap_df.to_csv(OUTPUT_DIR / 'pane_a_ratio_gap.csv', index=False)


📊 DATA TABLE: Unit Ratio & ASP Gap — by Size × Month
    size_code                     month  ariel_units  bold_units  unit_ratio  ariel_asp  bold_asp  asp_gap
         本体通常 2025-01-01 00:00:00+00:00     35,488.0    62,280.0        57.0      320.7     309.7     11.0
         本体通常 2025-02-01 00:00:00+00:00     84,778.0    63,717.0       133.1      223.0     290.3    -67.0
         本体通常 2025-03-01 00:00:00+00:00    273,898.0    59,431.0       460.9      191.2     290.5    -99.0
         本体通常 2025-04-01 00:00:00+00:00     90,287.0   168,165.0        53.7      241.5     215.7     26.0
         本体通常 2025-05-01 00:00:00+00:00     44,372.0   275,563.0        16.1      302.0     202.4    100.0
         本体通常 2025-06-01 00:00:00+00:00     48,629.0   138,581.0        35.1      323.1     243.8     79.0
         本体通常 2025-07-01 00:00:00+00:00     43,135.0    83,125.0        51.9      353.4     301.1     52.0
         本体通常 2025-08-01 00:00:00+00:00     48,254.0    66,272.0        72.8      335.4   

In [22]:
# # ── A-5(commented out by user): Per-Dose ASP Comparison ──────────────────────────────────────
# bold_post = df_asp[(df_asp['sub_brand']==BOLD_GB) & (df_asp['month']>=renewal_date)]
# bold_asp_by_size = bold_post.groupby('size_code').agg(total_sales=('total_sales','sum'), total_units=('total_units','sum')).reset_index()
# bold_asp_by_size['weighted_asp'] = bold_asp_by_size['total_sales'] / bold_asp_by_size['total_units']
# bold_asp_by_size['capacity_g'] = bold_asp_by_size['size_code'].map(BOLD_GB_CAPACITY_G)
# bold_asp_by_size['asp_per_gram'] = (bold_asp_by_size['weighted_asp'] / bold_asp_by_size['capacity_g']).round(2)
# bold_asp_by_size['sub_brand'] = BOLD_GB

# ariel_post = df_asp[(df_asp['sub_brand']==ARIEL_GB) & (df_asp['month']>=renewal_date)]
# ariel_asp_by_size = ariel_post.groupby('size_code').agg(total_sales=('total_sales','sum'), total_units=('total_units','sum')).reset_index()
# ariel_asp_by_size['weighted_asp'] = ariel_asp_by_size['total_sales'] / ariel_asp_by_size['total_units']
# ariel_asp_by_size['capacity_g'] = np.nan
# ariel_asp_by_size['asp_per_gram'] = np.nan
# ariel_asp_by_size['sub_brand'] = ARIEL_GB

# dose_table = pd.concat([bold_asp_by_size, ariel_asp_by_size], ignore_index=True)
# print('📊 DATA TABLE: Per-Dose ASP (Post-Renewal)')
# print('=' * 80)
# print(dose_table[['sub_brand','size_code','weighted_asp','capacity_g','asp_per_gram']].to_string(index=False))
# dose_table.to_csv(OUTPUT_DIR / 'pane_a_dose_asp.csv', index=False)


---
# Pane B — Trial Shopper
*Trial acquisition by size, head-to-head comparison, ASP elasticity, and price gap analysis.*


In [23]:
# ── B-1: Derive monthly trial from unified cohort ─────────────────────
# Sub-brand trial per month per size (from Q4 cohort)
# Revert rename if it was already applied (idempotent)
if 'size_code' in df_cohort.columns and 'trial_size' not in df_cohort.columns:
    df_cohort.rename(columns={'size_code': 'trial_size'}, inplace=True)
df_cohort['trial_month'] = df_cohort['trial_date'].dt.to_period('M').dt.to_timestamp()
df_subbrand_trial = df_cohort.groupby(['trial_month', 'sub_brand', 'trial_size']).agg(
    subbrand_trial_shoppers=('shopper_key', 'nunique')
).reset_index().rename(columns={'trial_size': 'size_code'})

# Q5 (category trial) is commented out — use sub-brand trial only
df_trial = df_subbrand_trial.copy()

# Merge total monthly buyers (from Q1 monthly_sales — rename month)
_dm = df_monthly.copy()
_dm['month'] = pd.to_datetime(_dm['month']).dt.tz_localize(None)  # strip tz
df_monthly_buyers = _dm.rename(columns={'month': 'trial_month'}).groupby(
    ['trial_month', 'sub_brand', 'size_code'])['shoppers'].first().reset_index().rename(
    columns={'shoppers': 'total_shoppers'})
df_trial = df_trial.merge(df_monthly_buyers, on=['trial_month', 'sub_brand', 'size_code'], how='left')
df_trial['trial_rate'] = (df_trial['subbrand_trial_shoppers'] / df_trial['total_shoppers'].replace(0, np.nan) * 100).round(2)

# Data table
print('📊 DATA TABLE: Trial Summary by Brand × Size')
print('=' * 80)
summary = df_trial.groupby(['sub_brand', 'size_code']).agg(
    total_subbrand_trial=('subbrand_trial_shoppers', 'sum'),
).reset_index()
for brand in [BOLD_GB, ARIEL_GB]:
    print(f'\n▶ {BRAND_LABEL.get(brand, brand)}')
    print(summary[summary['sub_brand']==brand].to_string(index=False))

summary.to_csv(OUTPUT_DIR / 'pane_b_trial_summary.csv', index=False)

📊 DATA TABLE: Trial Summary by Brand × Size

▶ Bold Gel Ball
     sub_brand     size_code  total_subbrand_trial
ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ          本体通常                251152
ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ         詰替超特大                    11
ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ                     2
ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ     詰替超ｼﾞｬﾝﾎﾞ                   115
ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ  詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ                 65377
ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ          詰替通常                     2
ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ    詰替ﾃﾗｼﾞｬﾝﾎﾞ                 17432
ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ 詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ                210104
ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ   詰替ﾒｶﾞｼﾞｬﾝﾎﾞ                 61642
ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ           ｿﾉﾀ                   201

▶ Ariel Gel Ball
    sub_brand     size_code  total_subbrand_trial
ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ          本体通常                184693
ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ         詰替超特大                    12
ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ     詰替超ｼﾞｬﾝﾎﾞ                   230
ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ  詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ                 63122
ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ    詰替ﾃﾗｼﾞｬﾝﾎﾞ                 26329
ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ 詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ                184783
ｱﾘｴｰﾙｼﾞｪﾙﾎ

In [24]:
# ── B-2: Combined Trial Trend — Bars + Trial Rate + Index ─────────────
combined_trial = df_trial[~df_trial['size_code'].isin(EXCLUDED_SIZES)].copy()
comb_sizes = order_and_filter_sizes(combined_trial['size_code'].unique())

if comb_sizes:
    _piv = combined_trial.groupby(['trial_month', 'sub_brand', 'size_code']).agg(
        subbrand_trial=('subbrand_trial_shoppers', 'sum'), trial_rate=('trial_rate', 'mean')).reset_index()
    _a = _piv[_piv['sub_brand']==ARIEL_GB][['trial_month','size_code','subbrand_trial','trial_rate']].rename(
        columns={'subbrand_trial':'bold_trial','trial_rate':'bold_rate'})
    _b = _piv[_piv['sub_brand']==BOLD_GB][['trial_month','size_code','subbrand_trial']].rename(
        columns={'subbrand_trial':'attack_trial'})
    _idx_df = _a.merge(_b, on=['trial_month','size_code'], how='outer').sort_values('trial_month')
    _idx_df['trial_index'] = (_idx_df['bold_trial'] / _idx_df['attack_trial'].replace(0, np.nan) * 100).round(1)

    n_s=len(comb_sizes); n_c=min(2,n_s); n_r=(n_s+n_c-1)//n_c
    specs = [[{'secondary_y':True}]*n_c for _ in range(n_r)]
    fig_comb = make_subplots(rows=n_r, cols=n_c, specs=specs, subplot_titles=comb_sizes,
                             vertical_spacing=0.15, horizontal_spacing=0.14)

    for idx, size in enumerate(comb_sizes):
        r, c = idx//n_c+1, idx%n_c+1
        for brand in [ARIEL_GB, BOLD_GB]:
            sub = combined_trial[(combined_trial['sub_brand']==brand) & (combined_trial['size_code']==size)].sort_values('trial_month')
            if len(sub)==0: continue
            fig_comb.add_trace(go.Bar(x=sub['trial_month'], y=sub['subbrand_trial_shoppers'],
                name=BRAND_LABEL[brand], marker_color=BRAND_COLOR[brand], opacity=0.85,
                showlegend=(idx==0), legendgroup=BRAND_LABEL[brand]), row=r, col=c, secondary_y=False)

        _s = _idx_df[_idx_df['size_code']==size]
        if len(_s)>0 and _s['trial_index'].notna().any():
            fig_comb.add_trace(go.Scatter(x=_s['trial_month'], y=_s['trial_index'], name='Index Ariel/Attack×100',
                mode='lines+markers', line=dict(color='#2CA02C', dash='dash', width=2),
                marker=dict(size=6, symbol='diamond'), showlegend=(idx==0), legendgroup='Index'),
                row=r, col=c, secondary_y=True)
            fig_comb.add_hline(y=100, line_dash='dot', line_color='gray', line_width=1, row=r, col=c, secondary_y=True)

        fig_comb.update_yaxes(title_text='Trial Shoppers', row=r, col=c, secondary_y=False)
        fig_comb.update_yaxes(title_text='Index / Trial Rate%', row=r, col=c, secondary_y=True, showgrid=False)

    fig_comb.update_layout(height=420*n_r, barmode='group',
        title_text='Monthly Trial: Ariel vs Bold per Size', template='plotly_white',
        legend=dict(orientation='h', yanchor='bottom', y=-0.15, x=0))
    fig_comb.show()

# ── DATA TABLE: Monthly Trial Shoppers — Ariel vs Bold per Size ────
trial_table = _idx_df[['trial_month','size_code','bold_trial','attack_trial','bold_rate','trial_index']].copy()
trial_table.columns = ['Month','Size','Ariel Trial','Bold Trial','Ariel Trial Rate%','Trial Index']
trial_table = trial_table.sort_values(['Size','Month'])
print('\n📊 DATA TABLE: Monthly Trial — Ariel vs Bold per Size')
print('=' * 100)
print(trial_table.to_string(index=False))
trial_table.to_csv(OUTPUT_DIR / 'pane_b_trial_trend.csv', index=False)


📊 DATA TABLE: Monthly Trial — Ariel vs Bold per Size
     Month          Size  Ariel Trial  Bold Trial  Ariel Trial Rate%  Trial Index
2025-01-01          本体通常        11010    19,624.0               45.6         56.1
2025-02-01          本体通常        25874    18,586.0               55.2        139.2
2025-03-01          本体通常        81317    19,203.0               65.0        423.5
2025-04-01          本体通常        24936    50,541.0               49.7         49.3
2025-05-01          本体通常        13202    79,921.0               46.6         16.5
2025-06-01          本体通常        14406    39,115.0               46.4         36.8
2025-07-01          本体通常        13948    24,162.0               47.5         57.7
2025-01-01  詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ        10666     9,734.0               22.9        109.6
2025-02-01  詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ        11732     8,629.0               27.0        136.0
2025-03-01  詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ        11755    12,397.0               28.4         94.8
2025-04-01  詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ         8110    11,

In [25]:
# ── B-3: Head-to-Head Sub-brand Trial by Size ─────────────────────────
h2h = df_trial[~df_trial['size_code'].isin(EXCLUDED_SIZES)].groupby(['sub_brand','size_code']).agg(
    subbrand_trial=('subbrand_trial_shoppers','sum'), avg_trial_rate=('trial_rate','mean')).reset_index()
h2h_sizes = order_and_filter_sizes(h2h['size_code'].unique())
h2h = h2h[h2h['size_code'].isin(h2h_sizes)].copy()

_ha = h2h[h2h['sub_brand']==ARIEL_GB][['size_code','subbrand_trial','avg_trial_rate']].rename(
    columns={'subbrand_trial':'bold_trial','avg_trial_rate':'bold_rate'})
_hb = h2h[h2h['sub_brand']==BOLD_GB][['size_code','subbrand_trial','avg_trial_rate']].rename(
    columns={'subbrand_trial':'attack_trial','avg_trial_rate':'attack_rate'})
h2h_idx = _ha.merge(_hb, on='size_code', how='outer')
h2h_idx['index'] = (h2h_idx['bold_trial'] / h2h_idx['attack_trial'].replace(0,np.nan) * 100).round(1)

print('📊 DATA TABLE: Head-to-Head Trial by Size')
print('=' * 80)
print(h2h_idx.to_string(index=False))

fig_h2h = make_subplots(specs=[[{'secondary_y':True}]])
for brand_code, brand_label, color in [(ARIEL_GB, 'Ariel Gel Ball', '#2980B9'), (BOLD_GB, 'Bold Gel Ball', '#E67E22')]:
    d = h2h[h2h['sub_brand']==brand_code]
    fig_h2h.add_trace(go.Bar(x=d['size_code'], y=d['subbrand_trial'], name=brand_label,
        marker_color=color, opacity=0.8, text=[fmt_km(v) for v in d['subbrand_trial']], textposition='outside'),
        secondary_y=False)
fig_h2h.add_trace(go.Scatter(x=h2h_idx['size_code'], y=h2h_idx['index'], name='Index (Ariel/Attack×100)',
    mode='lines+markers+text', line=dict(color='#27AE60', dash='dash', width=2.5),
    marker=dict(size=10, symbol='diamond', color='#27AE60'),
    text=[f'{v:.0f}' for v in h2h_idx['index']], textposition='bottom center'), secondary_y=True)
fig_h2h.add_hline(y=100, line_dash='dot', line_color='#BDC3C7', line_width=1.5, secondary_y=True)
fig_h2h.update_layout(title='Head-to-Head: Sub-brand Trial by Size', barmode='group',
    template='plotly_white', height=550, xaxis=dict(categoryorder='array', categoryarray=h2h_sizes))
fig_h2h.update_yaxes(title_text='Trial Shoppers', secondary_y=False)
fig_h2h.update_yaxes(title_text='Index', secondary_y=True, showgrid=False)
fig_h2h.show()

h2h_idx.to_csv(OUTPUT_DIR / 'pane_b_h2h_trial.csv', index=False)


📊 DATA TABLE: Head-to-Head Trial by Size
    size_code  bold_trial  bold_rate  attack_trial  attack_rate  index
         本体通常      184693       50.8        251152         50.6   73.5
 詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ       63122       23.9         65377         27.2   96.6
   詰替ﾃﾗｼﾞｬﾝﾎﾞ       26329       26.9         17432         28.9  151.0
詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ      184783       25.9        210104         28.7   87.9
  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ       69350       25.2         61642         27.0  112.5


In [26]:
# ── B-4: Price Gap Heatmaps ───────────────────────────────────────────
ariel_w = df_asp_weekly[(df_asp_weekly['sub_brand']==ARIEL_GB) & (~df_asp_weekly['size_code'].isin(EXCLUDED_SIZES))]\
    [['week_end','retailer','size_code','weighted_asp','total_units','buyer_count']].copy()
bold_w = df_asp_weekly[(df_asp_weekly['sub_brand']==BOLD_GB) & (~df_asp_weekly['size_code'].isin(EXCLUDED_SIZES))]\
    [['week_end','retailer','size_code','weighted_asp','total_units']].copy()
paired = ariel_w.merge(bold_w, on=['week_end','retailer','size_code'], suffixes=('_bold','_attack'), how='inner')
paired['ariel_asp_bin'] = (paired['weighted_asp_bold']//50*50).astype(int)
paired['bold_asp_bin'] = (paired['weighted_asp_attack']//50*50).astype(int)

sizes_for_hm = order_and_filter_sizes(paired['size_code'].unique())
heatmap_tables = []

for size in sizes_for_hm:
    s = paired[paired['size_code']==size].copy()
    if len(s) < MIN_OBS_FREQ: continue

    # Count observations per bin combination
    obs = s.groupby(['bold_asp_bin','ariel_asp_bin']).size().reset_index(name='obs')

    # Compute unit index based on average units sold per store (retailer)
    # Step 1: Average units per retailer within each bin combination
    store_avg = s.groupby(['bold_asp_bin','ariel_asp_bin','retailer']).agg(
        avg_units_bold=('total_units_bold','mean'),
        avg_units_attack=('total_units_attack','mean')
    ).reset_index()
    # Step 2: Average across retailers for the bin combination
    bin_agg = store_avg.groupby(['bold_asp_bin','ariel_asp_bin']).agg(
        avg_units_bold=('avg_units_bold','mean'),
        avg_units_attack=('avg_units_attack','mean')
    ).reset_index()
    bin_agg = bin_agg.merge(obs, on=['bold_asp_bin','ariel_asp_bin'], how='left')
    # Suppress low-frequency bins
    bin_agg.loc[bin_agg['obs'] < MIN_OBS_FREQ, ['avg_units_bold','avg_units_attack']] = np.nan
    bin_agg['unit_index'] = bin_agg['avg_units_bold'] / bin_agg['avg_units_attack'] * 100

    # Pivot for heatmap — Y axis ascending for intuitive reading
    hm = bin_agg.pivot_table(index='bold_asp_bin', columns='ariel_asp_bin', values='unit_index')
    hm = hm.sort_index(ascending=True)
    hm_obs = bin_agg.pivot_table(index='bold_asp_bin', columns='ariel_asp_bin', values='obs')
    hm_obs = hm_obs.reindex(index=hm.index, columns=hm.columns)

    if hm.isnull().all().all(): continue

    # Build custom text: index value + obs count
    custom_text = []
    for i in range(hm.shape[0]):
        row = []
        for j in range(hm.shape[1]):
            val = hm.iloc[i, j]
            n = hm_obs.iloc[i, j]
            if pd.isna(val) or pd.isna(n):
                row.append('')
            else:
                row.append(f'{val:.0f}<br><sub>n={int(n)}</sub>')
        custom_text.append(row)

    fig = go.Figure(data=go.Heatmap(
        z=hm.values, x=[str(c) for c in hm.columns], y=[str(r) for r in hm.index],
        text=custom_text, texttemplate='%{text}', textfont=dict(size=11),
        colorscale='RdYlGn', zmin=50, zmax=150,
        colorbar=dict(title='Unit Index'),
        hovertemplate='Ariel ASP: %{x}<br>Bold ASP: %{y}<br>Index: %{z:.0f}<extra></extra>'
    ))
    fig.update_layout(
        title=f'【{size}】Price Gap × Ariel Unit Index vs Bold (avg units/store, ≥{MIN_OBS_FREQ} obs)',
        xaxis_title='Ariel ASP (50JPY bin)', yaxis_title='Bold ASP (50JPY bin)',
        width=800, height=580, template='plotly_white')
    fig.show()

    # Collect table data
    bin_agg['size_code'] = size
    heatmap_tables.append(bin_agg[['bold_asp_bin','ariel_asp_bin','unit_index','obs','size_code']])

# ── DATA TABLE: Price Gap Heatmap Values ─────────────────────────────
if heatmap_tables:
    hm_all = pd.concat(heatmap_tables, ignore_index=True).dropna(subset=['unit_index'])
    hm_all['unit_index'] = hm_all['unit_index'].round(1)
    print('\n📊 DATA TABLE: Price Gap × Unit Index — per Size (avg units/store)')
    print('=' * 100)
    for size in sizes_for_hm:
        sz_data = hm_all[hm_all['size_code']==size]
        if len(sz_data) == 0: continue
        print(f'\n▶ {size}')
        print(sz_data[['bold_asp_bin','ariel_asp_bin','unit_index','obs']].to_string(index=False))
    hm_all.to_csv(OUTPUT_DIR / 'pane_b_heatmap_values.csv', index=False)


📊 DATA TABLE: Price Gap × Unit Index — per Size (avg units/store)

▶ 本体通常
 bold_asp_bin  ariel_asp_bin  unit_index  obs
          150            150        46.6  137
          150            200        31.4   27
          150            250        14.4  143
          150            300        14.1   56
          150            350         9.8  109
          200            150       101.3   27
          200            200        52.2   80
          200            250        36.1   73
          200            300        17.7   94
          200            350        20.4   64
          200            400        15.3   30
          250            150       360.7  116
          250            200       123.0   46
          250            250        62.6  346
          250            300        46.1  102
          250            350        40.6  145
          250            400        26.3   12
          300            150       569.2   82
          300            200       258.4   64
     

---
# Pane C — Repeat Shopper
*Cohort funnel, size migration, pre/post renewal, ASP band analysis.*


In [27]:
# ── C-1: Cohort Funnel by Entry Size ──────────────────────────────────
funnel = df_cohort.groupby(['sub_brand', 'trial_size', 'outcome']).agg(
    shoppers=('shopper_key', 'nunique')).reset_index()
funnel_pivot = funnel.pivot_table(index=['sub_brand','trial_size'], columns='outcome',
    values='shoppers', fill_value=0).reset_index()
if 'Repeat' in funnel_pivot.columns and 'Lapse' in funnel_pivot.columns:
    funnel_pivot['total'] = funnel_pivot['Repeat'] + funnel_pivot['Lapse']
    funnel_pivot['repeat_rate_%'] = (funnel_pivot['Repeat'] / funnel_pivot['total'] * 100).round(1)
    funnel_pivot['lapse_rate_%'] = (funnel_pivot['Lapse'] / funnel_pivot['total'] * 100).round(1)

print('📊 DATA TABLE: 6-Month Cohort Funnel')
print('=' * 80)
for brand in [BOLD_GB, ARIEL_GB]:
    print(f'\n▶ {BRAND_LABEL.get(brand, brand)}')
    print(funnel_pivot[funnel_pivot['sub_brand']==brand].to_string(index=False))

# Bar chart
if 'repeat_rate_%' in funnel_pivot.columns:
    valid_sizes = order_and_filter_sizes(funnel_pivot['trial_size'].unique())
    plot_df = funnel_pivot[funnel_pivot['trial_size'].isin(valid_sizes)].copy()
    plot_df['trial_size'] = pd.Categorical(plot_df['trial_size'], categories=valid_sizes, ordered=True)
    plot_df = plot_df.sort_values('trial_size')
    fig = px.bar(plot_df, x='trial_size', y='repeat_rate_%', color='sub_brand', barmode='group',
        color_discrete_map={BOLD_GB:'#1E90FF', ARIEL_GB:'#FF6347'}, text='repeat_rate_%',
        title='6-Month Repeat Rate by Trial Entry Size — Higher = Better Retention')
    fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
    fig.update_layout(template='plotly_white', height=500)
    fig.show()

funnel_pivot.to_csv(OUTPUT_DIR / 'pane_c_cohort_funnel.csv', index=False)


📊 DATA TABLE: 6-Month Cohort Funnel

▶ Bold Gel Ball
     sub_brand    trial_size     Lapse   Repeat     total  repeat_rate_%  lapse_rate_%
ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ          本体通常 165,842.0 85,310.0 251,152.0           34.0          66.0
ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ         詰替超特大       9.0      2.0      11.0           18.2          81.8
ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ       1.0      1.0       2.0           50.0          50.0
ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ     詰替超ｼﾞｬﾝﾎﾞ      91.0     24.0     115.0           20.9          79.1
ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ  詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ  45,209.0 20,168.0  65,377.0           30.8          69.2
ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ          詰替通常       2.0      0.0       2.0            0.0         100.0
ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ    詰替ﾃﾗｼﾞｬﾝﾎﾞ  12,276.0  5,156.0  17,432.0           29.6          70.4
ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ 詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ 134,002.0 76,102.0 210,104.0           36.2          63.8
ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ   詰替ﾒｶﾞｼﾞｬﾝﾎﾞ  41,051.0 20,591.0  61,642.0           33.4          66.6
ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ           ｿﾉﾀ     172.0     29.0     201.0           14.4     

In [28]:
# ── C-2: Size Migration Matrix ────────────────────────────────────────
for brand, brand_label, cscale in [(BOLD_GB, 'ボールドジェルボール', 'Blues'), (ARIEL_GB, 'アリエールジェルボール', 'Oranges')]:
    repeat_shoppers = df_cohort[
        (df_cohort['outcome']=='Repeat') & (df_cohort['sub_brand']==brand) &
        (~df_cohort['trial_size'].isin(EXCLUDED_SIZES)) & (~df_cohort['repeat_size'].isin(EXCLUDED_SIZES))
    ]
    if len(repeat_shoppers) == 0: continue
    migration = repeat_shoppers.groupby(['trial_size','repeat_size']).agg(shoppers=('shopper_key','nunique')).reset_index()
    migration_matrix = migration.pivot_table(index='trial_size', columns='repeat_size', values='shoppers', fill_value=0)
    ordered = order_and_filter_sizes(list(migration_matrix.index) + list(migration_matrix.columns))
    migration_matrix = migration_matrix.reindex(
        index=[s for s in ordered if s in migration_matrix.index],
        columns=[s for s in ordered if s in migration_matrix.columns], fill_value=0)
    migration_pct = migration_matrix.div(migration_matrix.sum(axis=1), axis=0) * 100

    print(f'\n📊 DATA TABLE: Size Migration — {brand_label} (Trial → Repeat %)')
    print(migration_pct.round(1).to_string())

    fig = px.imshow(migration_pct.values, x=migration_pct.columns.tolist(), y=migration_pct.index.tolist(),
        text_auto='.1f', color_continuous_scale=cscale,
        labels=dict(x='Repeat Size', y='Trial Size', color='%'),
        title=f'[{brand_label}] Size Migration: Where Do Trial Shoppers Repeat? (%)')
    fig.update_layout(height=500, template='plotly_white')
    fig.show()



📊 DATA TABLE: Size Migration — ボールドジェルボール (Trial → Repeat %)
repeat_size    本体通常  詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ  詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ  詰替ﾃﾗｼﾞｬﾝﾎﾞ
trial_size                                                               
本体通常           75.1           18.6          2.0           3.2         1.1
詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ  17.3           62.4          7.9           9.2         3.2
詰替ﾒｶﾞｼﾞｬﾝﾎﾞ     9.7           31.5         30.3          15.1        13.4
詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ    8.0           12.2          8.4          53.8        17.6
詰替ﾃﾗｼﾞｬﾝﾎﾞ      6.5            8.2         10.2           4.2        71.0

📊 DATA TABLE: Size Migration — アリエールジェルボール (Trial → Repeat %)
repeat_size    本体通常  詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ  詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ  詰替ﾃﾗｼﾞｬﾝﾎﾞ
trial_size                                                               
本体通常           68.3           22.9          3.1           4.0         1.7
詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ  10.5           67.6          9.8           8.4         3.8
詰替ﾒｶﾞｼﾞｬﾝﾎﾞ     5.4           26.8         36.7          15.6

In [29]:
# # ── C-3: Pre vs Post Renewal Repeat Rate ──────────────────────────────
# df_cohort['renewal_period'] = df_cohort['trial_date'].apply(lambda x: 'Pre-Renewal' if x < renewal_date else 'Post-Renewal')
# bold_cohort = df_cohort[df_cohort['sub_brand']==BOLD_GB].copy()

# period_funnel = bold_cohort.groupby(['renewal_period','trial_size','outcome']).agg(
#     shoppers=('shopper_key','nunique')).reset_index()
# period_pivot = period_funnel.pivot_table(index=['renewal_period','trial_size'], columns='outcome',
#     values='shoppers', fill_value=0).reset_index()
# if 'Repeat' in period_pivot.columns and 'Lapse' in period_pivot.columns:
#     period_pivot['total'] = period_pivot['Repeat'] + period_pivot['Lapse']
#     period_pivot['repeat_rate_%'] = (period_pivot['Repeat'] / period_pivot['total'] * 100).round(1)

# print('📊 DATA TABLE: Pre vs Post Renewal Repeat Rate — Bold Gel Ball')
# print('=' * 80)
# print(period_pivot.to_string(index=False))

# if 'repeat_rate_%' in period_pivot.columns:
#     fig = px.bar(period_pivot, x='trial_size', y='repeat_rate_%', color='renewal_period', barmode='group',
#         text='repeat_rate_%', color_discrete_map={'Pre-Renewal':'#6495ED', 'Post-Renewal':'#FF6347'},
#         title='Did the Renewal Improve Repeat Rates? (Bold Gel Ball)')
#     fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
#     fig.update_layout(template='plotly_white', height=500, yaxis_title='Repeat Rate (%)')
#     fig.show()

# period_pivot.to_csv(OUTPUT_DIR / 'pane_c_pre_post_renewal.csv', index=False)


In [30]:
# ── C-3b: Size Migration Direction — Same / Size Down / Size Up ──────
# Stacked bar comparing Bold Gel Ball vs Ariel Gel Ball
# X = trial entry size, Y = % of repeat shoppers by migration direction

def classify_migration(trial_size, repeat_size, size_order):
    """Return migration direction based on SIZE_ORDER index."""
    if trial_size not in size_order or repeat_size not in size_order:
        return 'Other'
    ti, ri = size_order.index(trial_size), size_order.index(repeat_size)
    if ti == ri:
        return '① Same Size'
    elif ri > ti:
        return '③ Size Up (Larger)'
    else:
        return '② Size Down (Smaller)'

MIGRATE_ORDER  = ['① Same Size', '② Size Down (Smaller)', '③ Size Up (Larger)']
MIGRATE_COLORS = {
    '① Same Size':           '#4ECDC4',
    '② Size Down (Smaller)': '#E74C3C',
    '③ Size Up (Larger)':    '#2ECC71',
}

dfs_mig = []
for brand in [BOLD_GB, ARIEL_GB]:
    repeat_shoppers_mig = df_cohort[
        (df_cohort['outcome']=='Repeat') & (df_cohort['sub_brand']==brand) &
        (~df_cohort['trial_size'].isin(EXCLUDED_SIZES)) &
        (~df_cohort['repeat_size'].isin(EXCLUDED_SIZES))
    ].copy()
    if len(repeat_shoppers_mig) == 0:
        continue
    repeat_shoppers_mig['migration_type'] = repeat_shoppers_mig.apply(
        lambda row: classify_migration(row['trial_size'], row['repeat_size'], SIZE_ORDER), axis=1
    )
    tmp_agg = repeat_shoppers_mig.groupby(['trial_size', 'migration_type']).agg(
        shoppers=('shopper_key', 'nunique')
    ).reset_index()
    tmp_agg['brand'] = brand
    dfs_mig.append(tmp_agg)

if dfs_mig:
    mig_compare = pd.concat(dfs_mig, ignore_index=True)
    mig_compare = mig_compare[mig_compare['migration_type'] != 'Other']

    # % within each brand × trial_size bucket
    totals = mig_compare.groupby(['brand', 'trial_size'])['shoppers'].transform('sum')
    mig_compare['pct'] = (mig_compare['shoppers'] / totals * 100).round(1)

    # Apply SIZE_ORDER ordering
    valid_sizes = order_and_filter_sizes(mig_compare['trial_size'].unique())
    mig_compare['trial_size'] = pd.Categorical(mig_compare['trial_size'], categories=valid_sizes, ordered=True)
    mig_compare = mig_compare[mig_compare['trial_size'].notna()].sort_values('trial_size')

    brands_list = [BOLD_GB, ARIEL_GB]
    brands_available = [b for b in brands_list if b in mig_compare['brand'].values]

    fig_mig = make_subplots(
        rows=len(brands_available), cols=1,
        subplot_titles=[BRAND_LABEL.get(b, b) for b in brands_available],
        vertical_spacing=0.18
    )

    for row_idx, brand in enumerate(brands_available, 1):
        bd = mig_compare[mig_compare['brand'] == brand]
        for mtype in MIGRATE_ORDER:
            sub = bd[bd['migration_type'] == mtype]
            if len(sub) == 0:
                continue
            fig_mig.add_trace(go.Bar(
                x=sub['trial_size'].astype(str),
                y=sub['pct'],
                name=mtype,
                marker_color=MIGRATE_COLORS[mtype],
                text=sub['pct'].apply(lambda v: f'{v:.1f}%'),
                textposition='inside',
                insidetextanchor='middle',
                legendgroup=mtype,
                showlegend=(row_idx == 1)
            ), row=row_idx, col=1)

    fig_mig.update_layout(
        height=420 * len(brands_available),
        title_text=(
            'Size Migration Direction: Bold Gel Ball vs Ariel Gel Ball<br>'
            '<sup>% of repeat shoppers — Same size vs Size Down (smaller) vs Size Up (larger refill)</sup>'
        ),
        barmode='stack',
        template='plotly_white',
        legend=dict(orientation='h', y=-0.05)
    )
    fig_mig.update_yaxes(title_text='% of Repeat Shoppers', range=[0, 105])
    fig_mig.update_xaxes(title_text='Trial Entry Size')
    fig_mig.show()

    # ── DATA TABLE ────────────────────────────────────────────────────
    print('\n📊 DATA TABLE: Size Migration Direction Summary')
    print('=' * 100)
    summary_tbl = mig_compare.pivot_table(
        index=['brand', 'trial_size'],
        columns='migration_type',
        values=['pct', 'shoppers'],
        fill_value=0
    ).round(1)
    print(summary_tbl.to_string())
    mig_compare.to_csv(OUTPUT_DIR / 'pane_c_migration_direction.csv', index=False)
else:
    print('⚠️ No migration data available')


📊 DATA TABLE: Size Migration Direction Summary
                                     pct                                             shoppers                                         
migration_type               ① Same Size ② Size Down (Smaller) ③ Size Up (Larger) ① Same Size ② Size Down (Smaller) ③ Size Up (Larger)
brand          trial_size                                                                                                             
ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ  本体通常                 68.3                   0.0               31.7    34,690.0                   0.0           16,125.0
               詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ        67.6                  10.5               21.9    44,006.0               6,848.0           14,272.0
               詰替ﾒｶﾞｼﾞｬﾝﾎﾞ          36.7                  32.2               31.1     8,298.0               7,283.0            7,028.0
               詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ         61.2                  25.0               13.8    11,510.0               4,705.0            2,606.0
       

In [31]:
# ── C-3c: Trial ASP Distribution — Repeat vs Lapse (Box Plots) ───────
# Shows whether repeaters and lapsers entered at different price points

ariel_boxplot_data = df_cohort[
    (df_cohort['sub_brand']==ARIEL_GB) &
    (~df_cohort['trial_size'].isin(EXCLUDED_SIZES)) &
    (df_cohort['trial_asp'].notna())
].copy()

box_sizes = order_and_filter_sizes(ariel_boxplot_data['trial_size'].unique())
if box_sizes:
    n_s = len(box_sizes); n_c = min(2, n_s); n_r = (n_s + n_c - 1) // n_c
    fig_box = make_subplots(rows=n_r, cols=n_c, subplot_titles=[f'Ariel {s}' for s in box_sizes],
                            vertical_spacing=0.16, horizontal_spacing=0.14)

    BOX_COLORS = {'Repeat': '#2ECC71', 'Lapse': '#E74C3C'}
    for idx, size in enumerate(box_sizes):
        r, c = idx // n_c + 1, idx % n_c + 1
        for outcome in ['Repeat', 'Lapse']:
            subset = ariel_boxplot_data[
                (ariel_boxplot_data['trial_size']==size) &
                (ariel_boxplot_data['outcome']==outcome)
            ]
            if len(subset) == 0: continue
            fig_box.add_trace(go.Box(
                y=subset['trial_asp'], name=outcome,
                marker_color=BOX_COLORS[outcome],
                boxmean='sd',
                showlegend=(idx == 0),
                legendgroup=outcome,
            ), row=r, col=c)
        fig_box.update_yaxes(title_text='Trial ASP (JPY)', row=r, col=c)

    fig_box.update_layout(
        height=420 * n_r,
        title_text=(
            'Trial ASP Distribution: Repeat vs Lapse Shoppers (Ariel Gel Ball)<br>'
            '<sup>Did repeat shoppers enter at different prices than lapsers?</sup>'
        ),
        template='plotly_white',
        legend=dict(orientation='h', y=-0.04)
    )
    fig_box.show()

    # ── DATA TABLE ────────────────────────────────────────────────────
    box_stats = ariel_boxplot_data.groupby(['trial_size', 'outcome'])['trial_asp'].agg(
        ['count', 'mean', 'median', 'std', 'min', 'max']
    ).round(0).reset_index()
    box_stats.columns = ['Size', 'Outcome', 'Shoppers', 'Mean ASP', 'Median ASP', 'Std', 'Min', 'Max']
    print('\n📊 DATA TABLE: Trial ASP Statistics — Repeat vs Lapse')
    print('=' * 100)
    print(box_stats.to_string(index=False))
    box_stats.to_csv(OUTPUT_DIR / 'pane_c_asp_boxplot_stats.csv', index=False)
else:
    print('⚠️ No data for box plots')


📊 DATA TABLE: Trial ASP Statistics — Repeat vs Lapse
         Size Outcome  Shoppers  Mean ASP  Median ASP   Std  Min     Max
         本体通常   Lapse    133860     238.0       199.0  87.0  0.0   699.0
         本体通常  Repeat     50833     246.0       199.0  87.0  0.0   519.0
 詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ   Lapse     44293   2,340.0     2,278.0 475.0  0.0 3,278.0
 詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ  Repeat     18829   2,371.0     2,278.0 386.0  0.0 3,278.0
   詰替ﾃﾗｼﾞｬﾝﾎﾞ   Lapse     18323   3,067.0     3,076.0 415.0  0.0 3,608.0
   詰替ﾃﾗｼﾞｬﾝﾎﾞ  Repeat      8006   2,957.0     2,980.0 442.0  0.0 3,608.0
詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ   Lapse    119638     896.0       931.0 168.0  0.0 1,960.0
詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ  Repeat     65145     909.0       930.0 143.0  0.0 1,324.0
  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ   Lapse     46740   1,778.0     1,880.0 380.0  0.0 3,549.0
  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ  Repeat     22610   1,814.0     1,880.0 320.0  0.0 2,398.0


In [32]:
# ── C-4: ASP Band — Trial/Repeat/Lapse Rates ─────────────────────────
ariel_trials = df_cohort[(df_cohort['sub_brand']==ARIEL_GB) & (~df_cohort['trial_size'].isin(EXCLUDED_SIZES))].copy()
ariel_trials['asp_band'] = (ariel_trials['trial_asp'] // 50 * 50).astype('Int64')

asp_bin_agg = ariel_trials.groupby(['trial_size','asp_band']).agg(
    trial_shoppers=('shopper_key','nunique'), obs_count=('shopper_key','count')).reset_index()
asp_bin_split = ariel_trials.groupby(['trial_size','asp_band','outcome']).agg(
    shoppers=('shopper_key','nunique')).reset_index()

repeat_counts = asp_bin_split[asp_bin_split['outcome']=='Repeat'][['trial_size','asp_band','shoppers']].rename(columns={'shoppers':'repeat_shoppers'})
lapse_counts = asp_bin_split[asp_bin_split['outcome']=='Lapse'][['trial_size','asp_band','shoppers']].rename(columns={'shoppers':'lapse_shoppers'})

rate_df = asp_bin_agg.merge(df_universe, on=['trial_size','asp_band'], how='left')\
    .merge(repeat_counts, on=['trial_size','asp_band'], how='left')\
    .merge(lapse_counts, on=['trial_size','asp_band'], how='left')
rate_df[['repeat_shoppers','lapse_shoppers']] = rate_df[['repeat_shoppers','lapse_shoppers']].fillna(0)
rate_df['trial_rate_%'] = (rate_df['trial_shoppers'] / rate_df['all_shoppers'] * 100).round(1)
rate_df['repeat_rate_%'] = (rate_df['repeat_shoppers'] / rate_df['trial_shoppers'] * 100).round(1)
rate_df['lapse_rate_%'] = (rate_df['lapse_shoppers'] / rate_df['trial_shoppers'] * 100).round(1)
for col in ['trial_rate_%','repeat_rate_%','lapse_rate_%']:
    rate_df[col] = rate_df[col].clip(upper=100)

print('📊 DATA TABLE: Trial/Repeat/Lapse Rate by ASP Band (sample)')
print(rate_df[['trial_size','asp_band','all_shoppers','trial_shoppers','trial_rate_%','repeat_rate_%','lapse_rate_%']].head(15).to_string(index=False))

sizes_rate = order_and_filter_sizes(rate_df['trial_size'].unique())
if sizes_rate:
    n_s=len(sizes_rate); n_c=min(2,n_s); n_r=(n_s+n_c-1)//n_c
    specs = [[{"secondary_y":True}]*n_c for _ in range(n_r)]
    fig_rate = make_subplots(rows=n_r, cols=n_c, specs=specs,
        subplot_titles=[f'Ariel {s}' for s in sizes_rate], vertical_spacing=0.16, horizontal_spacing=0.14)
    RATE_COLORS = {'trial_rate_%':'#1E90FF', 'repeat_rate_%':'#2ECC71', 'lapse_rate_%':'#E74C3C'}
    for idx, size in enumerate(sizes_rate):
        r, c = idx//n_c+1, idx%n_c+1
        subset = rate_df[rate_df['trial_size']==size].sort_values('asp_band')
        if len(subset)==0: continue
        x_labels = subset['asp_band'].astype(str) + '~'
        fig_rate.add_trace(go.Bar(x=x_labels, y=subset['all_shoppers'], name='All Shoppers',
            marker_color='#D0D0D0', opacity=0.6, showlegend=(idx==0), legendgroup='all_vol'), row=r, col=c, secondary_y=False)
        for rate_col, color in RATE_COLORS.items():
            fig_rate.add_trace(go.Scatter(x=x_labels, y=subset[rate_col], name=rate_col.replace('_',' '),
                mode='lines+markers', line=dict(color=color, width=2), marker=dict(size=6),
                showlegend=(idx==0), legendgroup=rate_col), row=r, col=c, secondary_y=True)
    fig_rate.update_layout(height=440*n_r, title_text='ASP Band — Trial/Repeat/Lapse Rates (Ariel by Size)',
        template='plotly_white', legend=dict(orientation='h', y=-0.04, x=0.5, xanchor='center'))
    fig_rate.update_xaxes(title_text='ASP Band (JPY)')
    for _r in range(1, n_r+1):
        for _c in range(1, n_c+1):
            fig_rate.update_yaxes(title_text='All Shoppers', secondary_y=False, row=_r, col=_c)
            fig_rate.update_yaxes(title_text='Rate (%)', secondary_y=True, row=_r, col=_c, range=[0,105], showgrid=False)
    fig_rate.show()

rate_df.to_csv(OUTPUT_DIR / 'pane_c_asp_band_rates.csv', index=False)


📊 DATA TABLE: Trial/Repeat/Lapse Rate by ASP Band (sample)
  trial_size  asp_band  all_shoppers  trial_shoppers  trial_rate_%  repeat_rate_%  lapse_rate_%
        本体通常         0          7680            2037          26.5           26.5          73.5
        本体通常        50          7907            4644          58.7           21.4          78.6
        本体通常       100          7408            2675          36.1           19.0          81.0
        本体通常       150        229126          104100          45.4           27.0          73.0
        本体通常       200         55004            6919          12.6           17.6          82.4
        本体通常       250         69446           28142          40.5           30.4          69.6
        本体通常       300         36216            7788          21.5           29.6          70.4
        本体通常       350         78140           22193          28.4           32.0          68.0
        本体通常       400         19338            6190          32.0           

In [33]:
# ── C-5: Trial/Repeat/Lapse Price-Point Productivity (per store) + Frequency ──
# 3 charts: ALL trial, REPEAT only, LAPSE only — each faceted by size
# Blue bars = avg shoppers per store (left Y), Orange line = store execution freq (right Y)
# Per-store normalisation: shoppers / total_stores at that ASP band
# store_exec_freq = SUM(n_stores) across weeks = week × #stores executing that price point

ariel_pp = df_cohort[
    (df_cohort['sub_brand']==ARIEL_GB) &
    (~df_cohort['trial_size'].isin(EXCLUDED_SIZES))
].copy()
ariel_pp['asp_band'] = (ariel_pp['trial_asp'] // 50 * 50).astype('Int64')

# ── Build store execution frequency from df_asp_weekly (week × # stores per ASP band × size) ──
ariel_weekly = df_asp_weekly[
    (df_asp_weekly['sub_brand']==ARIEL_GB) &
    (~df_asp_weekly['size_code'].isin(EXCLUDED_SIZES))
].copy()
ariel_weekly['asp_band'] = (ariel_weekly['weighted_asp'] // 50 * 50).astype('Int64')
market_freq = ariel_weekly.groupby(['size_code', 'asp_band']).agg(
    store_exec_freq=('n_stores', 'sum')   # SUM of stores executing that price point across weeks
).reset_index().rename(columns={'size_code': 'trial_size'})

# Aggregate: all trial + split by outcome
pp_all = ariel_pp.groupby(['trial_size', 'asp_band']).agg(
    trial_shoppers=('shopper_key', 'nunique')
).reset_index()

pp_split = ariel_pp.groupby(['trial_size', 'asp_band', 'outcome']).agg(
    shoppers=('shopper_key', 'nunique')
).reset_index()

# Merge store execution frequency onto shopper tables
pp_all = pp_all.merge(market_freq, on=['trial_size', 'asp_band'], how='left')
pp_split = pp_split.merge(market_freq, on=['trial_size', 'asp_band'], how='left')

# Compute avg shoppers per store — 2 significant figures (有効数字第二位)
def _to_2sf(x):
    if pd.isna(x) or x == 0:
        return x
    return float(f'{x:.2g}')

pp_all['shoppers_per_store'] = (pp_all['trial_shoppers'] / pp_all['store_exec_freq']).apply(_to_2sf)
pp_split['shoppers_per_store'] = (pp_split['shoppers'] / pp_split['store_exec_freq']).apply(_to_2sf)

# Filter out ASP bands with store execution freq < 100
MIN_STORE_EXEC = 10000
pp_all = pp_all[pp_all['store_exec_freq'] >= MIN_STORE_EXEC].reset_index(drop=True)
pp_split = pp_split[pp_split['store_exec_freq'] >= MIN_STORE_EXEC].reset_index(drop=True)

# Additional filter: 25th-pct threshold (min=5 shoppers) to skip low-data bands
pp_all_filt = (
    pp_all.groupby('trial_size', group_keys=False)
    .apply(lambda g: g[g['trial_shoppers'] >= max(g['trial_shoppers'].quantile(0.25), 5)])
    .reset_index(drop=True)
)
pp_split_filt = (
    pp_split.groupby(['trial_size', 'outcome'], group_keys=False)
    .apply(lambda g: g[g['shoppers'] >= max(g['shoppers'].quantile(0.25), 5)])
    .reset_index(drop=True)
)

sizes_pp = order_and_filter_sizes(pp_all_filt['trial_size'].unique())

def make_asp_bin_chart(data_df, y_col, freq_col, sizes, title, y_label):
    """Bar chart of avg shoppers/store by ASP band; store execution freq on secondary Y."""
    if len(sizes) == 0:
        print(f'⚠️ No data for: {title}')
        return
    n_s = len(sizes)
    n_c = min(2, n_s)
    n_r = (n_s + n_c - 1) // n_c
    specs = [[{"secondary_y": True} for _ in range(n_c)] for _ in range(n_r)]
    fig = make_subplots(
        rows=n_r, cols=n_c, specs=specs,
        subplot_titles=[f'Ariel {s}' for s in sizes],
        vertical_spacing=0.16, horizontal_spacing=0.14
    )
    for idx, size in enumerate(sizes):
        r = idx // n_c + 1
        c = idx % n_c + 1
        subset = data_df[data_df['trial_size'] == size].sort_values('asp_band')
        if len(subset) == 0: continue
        x_labels = subset['asp_band'].astype(str) + '~'
        fig.add_trace(go.Bar(
            x=x_labels, y=subset[y_col], name=y_label,
            marker_color='#4C9BE8', opacity=0.8,
            text=[f'{v:.2g}' for v in subset[y_col]],
            textposition='outside', textfont=dict(size=10),
            showlegend=(idx == 0), legendgroup='shoppers'
        ), row=r, col=c, secondary_y=False)
        fig.add_trace(go.Scatter(
            x=x_labels, y=subset[freq_col], name='Store Execution Freq (week × #stores)',
            mode='lines+markers', line=dict(color='#FF6347', width=2),
            marker=dict(size=6, symbol='circle'),
            showlegend=(idx == 0), legendgroup='frequency'
        ), row=r, col=c, secondary_y=True)
    fig.update_layout(
        height=420 * n_r, title_text=title, template='plotly_white',
        legend=dict(orientation='h', y=-0.04, x=0.5, xanchor='center')
    )
    fig.update_xaxes(title_text='ASP (50 JPY bin)')
    for row in range(1, n_r + 1):
        for col in range(1, n_c + 1):
            fig.update_yaxes(title_text=y_label, secondary_y=False, row=row, col=col)
            fig.update_yaxes(title_text='Store Exec Freq', secondary_y=True, row=row, col=col, showgrid=False)
    fig.show()

# ── Chart 1: All trial shoppers (per store) ──────────────────────────
make_asp_bin_chart(
    data_df=pp_all_filt, y_col='shoppers_per_store', freq_col='store_exec_freq', sizes=sizes_pp,
    title=('アリエールジェルボール: Trial Entry ASP Band (50JPY) — Avg Trial Shoppers per Store<br>'
           '<sup>Blue bars = avg trial shoppers / store | Orange line = store execution freq (week × #stores)</sup>'),
    y_label='Avg Trial Shoppers / Store'
)

# ── Chart 2: Repeat shoppers only (per store) ────────────────────────
repeat_pp = pp_split_filt[pp_split_filt['outcome'] == 'Repeat']
sizes_repeat_pp = order_and_filter_sizes(repeat_pp['trial_size'].unique())
make_asp_bin_chart(
    data_df=repeat_pp, y_col='shoppers_per_store', freq_col='store_exec_freq', sizes=sizes_repeat_pp,
    title=('アリエールジェルボール: Trial ASP Band — Avg Repeat Shoppers per Store (6-month repeat)<br>'
           '<sup>Blue bars = avg repeat shoppers / store | Orange line = store execution freq (week × #stores)</sup>'),
    y_label='Avg Repeat Shoppers / Store'
)

# ── Chart 3: Lapse shoppers only (per store) ─────────────────────────
lapse_pp = pp_split_filt[pp_split_filt['outcome'] == 'Lapse']
sizes_lapse_pp = order_and_filter_sizes(lapse_pp['trial_size'].unique())
make_asp_bin_chart(
    data_df=lapse_pp, y_col='shoppers_per_store', freq_col='store_exec_freq', sizes=sizes_lapse_pp,
    title=('アリエールジェルボール: Trial ASP Band — Avg Lapse Shoppers per Store (no repeat within 6 months)<br>'
           '<sup>Blue bars = avg lapse shoppers / store | Orange line = store execution freq (week × #stores)</sup>'),
    y_label='Avg Lapse Shoppers / Store'
)

# ── DATA TABLES ──────────────────────────────────────────────────────
print('\n📊 DATA TABLE: ASP Band × Avg Shoppers/Store + Store Exec Freq — ALL Trial')
print('=' * 110)
print(pp_all_filt[['trial_size','asp_band','trial_shoppers','store_exec_freq','shoppers_per_store']].to_string(index=False))

print('\n📊 DATA TABLE: ASP Band × Avg Shoppers/Store + Store Exec Freq — REPEAT only')
print('=' * 100)
print(repeat_pp[['trial_size','asp_band','shoppers','store_exec_freq','shoppers_per_store']].to_string(index=False))

print('\n📊 DATA TABLE: ASP Band × Avg Shoppers/Store + Store Exec Freq — LAPSE only')
print('=' * 100)
print(lapse_pp[['trial_size','asp_band','shoppers','store_exec_freq','shoppers_per_store']].to_string(index=False))

pp_all_filt.to_csv(OUTPUT_DIR / 'pane_c_pp_all.csv', index=False)
pp_split_filt.to_csv(OUTPUT_DIR / 'pane_c_pp_split.csv', index=False)


📊 DATA TABLE: ASP Band × Avg Shoppers/Store + Store Exec Freq — ALL Trial
   trial_size  asp_band  trial_shoppers  store_exec_freq  shoppers_per_store
         本体通常       150          104100        207,010.0                 0.5
         本体通常       250           28142         80,212.0                 0.3
         本体通常       300            7788         70,102.0                 0.1
         本体通常       350           22193         85,973.0                 0.3
 詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ      2150            1275         42,828.0                 0.0
 詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ      2200            1091         45,433.0                 0.0
 詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ      2250           14987         35,131.0                 0.4
 詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ      2350            1672        103,706.0                 0.0
 詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ      2450            1831         12,253.0                 0.1
 詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ      2500            6523         12,324.0                 0.5
 詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ      2550            1519         29,177.0                 0.1
 

In [34]:
# ── C-5b: Total Shoppers (Repeat+Lapse) per Store + Repeat Rate by ASP Band ──
# Blue bars = total shoppers per store (left Y), Orange line = repeat rate % (right Y)

# Build total (repeat + lapse) per ASP band × size
pp_total = pp_split_filt.groupby(['trial_size', 'asp_band']).agg(
    total_shoppers=('shoppers', 'sum'),
    store_exec_freq=('store_exec_freq', 'first')
).reset_index()

# Get repeat count per ASP band × size
pp_repeat_only = pp_split_filt[pp_split_filt['outcome'] == 'Repeat'][['trial_size', 'asp_band', 'shoppers']].rename(
    columns={'shoppers': 'repeat_shoppers'}
)

pp_total = pp_total.merge(pp_repeat_only, on=['trial_size', 'asp_band'], how='left')
pp_total['repeat_shoppers'] = pp_total['repeat_shoppers'].fillna(0)

# Compute metrics
pp_total['total_per_store'] = (pp_total['total_shoppers'] / pp_total['store_exec_freq']).apply(_to_2sf)
pp_total['repeat_rate'] = (pp_total['repeat_shoppers'] / pp_total['total_shoppers'] * 100).round(1)

sizes_total = order_and_filter_sizes(pp_total['trial_size'].unique())

def make_total_repeat_rate_chart(data_df, sizes, title):
    """Bar chart of total shoppers/store by ASP band; repeat rate % on secondary Y."""
    if len(sizes) == 0:
        print(f'⚠️ No data for: {title}')
        return
    n_s = len(sizes)
    n_c = min(2, n_s)
    n_r = (n_s + n_c - 1) // n_c
    specs = [[{"secondary_y": True} for _ in range(n_c)] for _ in range(n_r)]
    fig = make_subplots(
        rows=n_r, cols=n_c, specs=specs,
        subplot_titles=[f'Ariel {s}' for s in sizes],
        vertical_spacing=0.16, horizontal_spacing=0.14
    )
    for idx, size in enumerate(sizes):
        r = idx // n_c + 1
        c = idx % n_c + 1
        subset = data_df[data_df['trial_size'] == size].sort_values('asp_band')
        if len(subset) == 0: continue
        x_labels = subset['asp_band'].astype(str) + '~'
        fig.add_trace(go.Bar(
            x=x_labels, y=subset['total_per_store'], name='Total Shoppers / Store',
            marker_color='#4C9BE8', opacity=0.8,
            text=[f'{v:.2g}' for v in subset['total_per_store']],
            textposition='outside', textfont=dict(size=10),
            showlegend=(idx == 0), legendgroup='total'
        ), row=r, col=c, secondary_y=False)
        fig.add_trace(go.Scatter(
            x=x_labels, y=subset['repeat_rate'], name='Repeat Rate (%)',
            mode='lines+markers', line=dict(color='#FF6347', width=2),
            marker=dict(size=6, symbol='circle'),
            showlegend=(idx == 0), legendgroup='rate'
        ), row=r, col=c, secondary_y=True)
    fig.update_layout(
        height=420 * n_r, title_text=title, template='plotly_white',
        legend=dict(orientation='h', y=-0.04, x=0.5, xanchor='center')
    )
    fig.update_xaxes(title_text='ASP (50 JPY bin)')
    for row in range(1, n_r + 1):
        for col in range(1, n_c + 1):
            fig.update_yaxes(title_text='Total Shoppers / Store', secondary_y=False, row=row, col=col)
            fig.update_yaxes(title_text='Repeat Rate (%)', secondary_y=True, row=row, col=col, showgrid=False)
    fig.show()

make_total_repeat_rate_chart(
    data_df=pp_total, sizes=sizes_total,
    title=('アリエールジェルボール: Trial ASP Band — Total Shoppers (Repeat+Lapse) per Store + Repeat Rate<br>'
           '<sup>Blue bars = avg total shoppers / store | Orange line = repeat rate (repeat / total × 100)</sup>')
)

# ── DATA TABLE ──────────────────────────────────────────────────────
print('\n📊 DATA TABLE: ASP Band × Total Shoppers/Store + Repeat Rate')
print('=' * 110)
print(pp_total[['trial_size','asp_band','total_shoppers','repeat_shoppers','store_exec_freq','total_per_store','repeat_rate']].to_string(index=False))


📊 DATA TABLE: ASP Band × Total Shoppers/Store + Repeat Rate
   trial_size  asp_band  total_shoppers  repeat_shoppers  store_exec_freq  total_per_store  repeat_rate
         本体通常       150          104100         28,128.0        207,010.0              0.5         27.0
         本体通常       200            5703              0.0         61,212.0              0.1          0.0
         本体通常       250           28142          8,552.0         80,212.0              0.3         30.4
         本体通常       300            2309          2,309.0         70,102.0              0.0        100.0
         本体通常       350           22193          7,091.0         85,973.0              0.3         32.0
 詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ      2150            1275            405.0         42,828.0              0.0         31.8
 詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ      2200            1091            280.0         45,433.0              0.0         25.7
 詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ      2250           14987          5,512.0         35,131.0              0.4         36.8
 詰替

---
# Pane D — Lapsed Shopper
*Lapse rate by size and ASP band, post-lapse destination tracking.*


In [35]:
# ── D-1: Lapse Rate by Size ──────────────────────────────────────────
df_active_filt = df_active[~df_active['size_code'].isin(EXCLUDED_SIZES)]

for brand, brand_label in [(BOLD_GB, 'Bold Gel Ball'), (ARIEL_GB, 'Ariel Gel Ball')]:
    lapse_by_size = df_lapsed[(df_lapsed['sub_brand']==brand) & (~df_lapsed['last_size'].isin(EXCLUDED_SIZES))].groupby('last_size').agg(
        lapsed_shoppers=('shopper_key','nunique')).reset_index().rename(columns={'last_size':'size_code'})
    active_brand = df_active_filt[df_active_filt['sub_brand']==brand]
    lapse_rate = lapse_by_size.merge(active_brand[['size_code','active_shoppers']], on='size_code', how='left')
    lapse_rate['lapse_rate_%'] = (lapse_rate['lapsed_shoppers'] / lapse_rate['active_shoppers'] * 100).round(1)
    lapse_rate['_sort'] = lapse_rate['size_code'].map({s:i for i,s in enumerate(SIZE_ORDER)}).fillna(99)
    lapse_rate = lapse_rate.sort_values('_sort').drop(columns='_sort')

    print(f'\n📊 DATA TABLE: Lapse Rate by Size — {brand_label}')
    print('=' * 60)
    print(lapse_rate.to_string(index=False))



📊 DATA TABLE: Lapse Rate by Size — Bold Gel Ball
    size_code  lapsed_shoppers  active_shoppers  lapse_rate_%
         本体通常           226751           364160          62.3
詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ           231456           465754          49.7
  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ            74170           173043          42.9
 詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ            80552           176958          45.5
   詰替ﾃﾗｼﾞｬﾝﾎﾞ            22643            50348          45.0

📊 DATA TABLE: Lapse Rate by Size — Ariel Gel Ball
    size_code  lapsed_shoppers  active_shoppers  lapse_rate_%
         本体通常           177822           268713          66.2
詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ           213632           433670          49.3
  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ            86811           201133          43.2
 詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ            81217           182328          44.5
   詰替ﾃﾗｼﾞｬﾝﾎﾞ            32908            70448          46.7


In [36]:
# # ── D-2: Lapse Rate by ASP Band (Ariel, dual-filtered) ───────────────
# df_at_risk_filt = df_at_risk[~df_at_risk['size_code'].isin(EXCLUDED_SIZES)].copy()
# df_at_risk_filt['asp_band'] = (df_at_risk_filt['asp'] // 50 * 50).astype(int)

# asp_lapse = df_at_risk_filt.groupby(['size_code','asp_band']).agg(
#     total_shoppers=('shopper_key','nunique'), lapsed_shoppers=('is_lapsed','sum')).reset_index()
# asp_lapse['lapse_rate_%'] = (asp_lapse['lapsed_shoppers'] / asp_lapse['total_shoppers'] * 100).round(1)
# asp_lapse = asp_lapse.merge(df_week_store[['size_code','asp_band','week_store_count']], on=['size_code','asp_band'], how='left')
# asp_lapse['week_store_count'] = asp_lapse['week_store_count'].fillna(0).astype(int)
# asp_lapse_plot = asp_lapse[(asp_lapse['total_shoppers']>=MIN_FREQ) & (asp_lapse['week_store_count']>=MIN_WEEK_STORE)]

# # All sizes combined
# asp_lapse_all = df_at_risk_filt.groupby('asp_band').agg(
#     total_shoppers=('shopper_key','nunique'), lapsed_shoppers=('is_lapsed','sum')).reset_index()
# asp_lapse_all['lapse_rate_%'] = (asp_lapse_all['lapsed_shoppers'] / asp_lapse_all['total_shoppers'] * 100).round(1)
# asp_lapse_all['size_code'] = '(All Sizes)'
# df_ws_all = df_week_store.groupby('asp_band')['week_store_count'].sum().reset_index()
# asp_lapse_all = asp_lapse_all.merge(df_ws_all, on='asp_band', how='left')
# asp_lapse_all['week_store_count'] = asp_lapse_all['week_store_count'].fillna(0).astype(int)
# asp_lapse_all_plot = asp_lapse_all[(asp_lapse_all['total_shoppers']>=MIN_FREQ) & (asp_lapse_all['week_store_count']>=MIN_WEEK_STORE)]

# print('📊 DATA TABLE: Lapse Rate by ASP Band — All Sizes')
# print(asp_lapse_all_plot.to_string(index=False))

# # Chart
# combined_plot = pd.concat([asp_lapse_all_plot, asp_lapse_plot], ignore_index=True)
# plot_sizes = ['(All Sizes)'] + order_and_filter_sizes(asp_lapse_plot['size_code'].unique())
# n_s=len(plot_sizes); n_c=min(2,n_s); n_r=(n_s+n_c-1)//n_c
# fig = make_subplots(rows=n_r, cols=n_c, subplot_titles=[f'{s}' for s in plot_sizes], vertical_spacing=0.10)
# for idx, size in enumerate(plot_sizes):
#     r, c = idx//n_c+1, idx%n_c+1
#     subset = combined_plot[combined_plot['size_code']==size].sort_values('asp_band')
#     if len(subset)==0: continue
#     fig.add_trace(go.Bar(x=subset['asp_band'].astype(str)+'~', y=subset['lapse_rate_%'],
#         marker_color='#FF6B6B',
#         text=[f'{r:.0f}%\n({int(l)}/{int(t)})' for r,l,t in zip(subset['lapse_rate_%'],subset['lapsed_shoppers'],subset['total_shoppers'])],
#         textposition='outside', showlegend=False), row=r, col=c)
#     fig.update_xaxes(title_text='ASP (50 JPY bin)', row=r, col=c)
#     fig.update_yaxes(title_text='Lapse Rate (%)', row=r, col=c)
# fig.update_layout(height=350*n_r, title_text=f'Lapse Rate by ASP Band (shoppers≥{MIN_FREQ}, wk×store≥{MIN_WEEK_STORE})',
#     template='plotly_white')
# fig.show()

# asp_lapse.to_csv(OUTPUT_DIR / 'pane_d_lapse_by_asp.csv', index=False)


In [37]:
# ── D-3: Post-Lapse Destination (Bold) ───────────────────────────────
ariel_dest = df_destination[df_destination['source_brand']==ARIEL_GB]
ariel_lapsed_full = df_lapsed[df_lapsed['sub_brand']==ARIEL_GB]

dest_summary = ariel_dest.groupby(['next_sub_brand','next_size']).agg(shoppers=('shopper_key','nunique')).reset_index().sort_values('shoppers', ascending=False)
total_lapsed = ariel_lapsed_full['shopper_key'].nunique()
total_tracked = dest_summary['shoppers'].sum()
category_exit = total_lapsed - total_tracked

print('📊 DATA TABLE: Post-Lapse Destination (Bold)')
print(f'Total lapsed: {total_lapsed:,} | Tracked: {total_tracked:,} | Category exit: {category_exit:,}')
print('=' * 80)
dest_summary['share_%'] = (dest_summary['shoppers'] / total_lapsed * 100).round(1)
print(dest_summary.head(15).to_string(index=False))

# Treemap
tree_data = pd.concat([dest_summary, pd.DataFrame([{
    'next_sub_brand':'(Category Exit)', 'next_size':'No Purchase',
    'shoppers':category_exit, 'share_%':round(category_exit/total_lapsed*100,1)}])], ignore_index=True)
fig = px.treemap(tree_data, path=['next_sub_brand','next_size'], values='shoppers',
    title='Post-Lapse Destination: Where Do Lost Ariel Shoppers Go?',
    color='shoppers', color_continuous_scale='Reds')
fig.update_layout(height=600)
fig.show()

dest_summary.to_csv(OUTPUT_DIR / 'pane_d_destination.csv', index=False)


📊 DATA TABLE: Post-Lapse Destination (Bold)
Total lapsed: 585,076 | Tracked: 314,212 | Category exit: 270,864
       next_sub_brand     next_size  shoppers  share_%
       ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ 詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ     25895      4.4
       ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ          本体通常     23225      4.0
             ｱﾀｯｸ抗菌EX 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ     17358      3.0
            ｱﾘｴｰﾙｼﾞｪﾙ          本体通常     16846      2.9
            ｱﾘｴｰﾙｼﾞｪﾙ         詰替超特大     16108      2.8
            ｱﾘｴｰﾙｼﾞｪﾙ 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ     12621      2.2
             ｱﾀｯｸ抗菌EX         詰替超特大     11278      1.9
ｱﾀｯｸZERO ﾊﾟｰﾌｪｸﾄｽﾃｨｯｸ     ﾒｶﾞｼﾞｬﾝﾎﾞ     10908      1.9
             ｱﾀｯｸ抗菌EX  詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ      9727      1.7
                 ｴﾏｰﾙ         詰替超特大      9490      1.6
       ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ  詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ      9017      1.5
       ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ   詰替ﾒｶﾞｼﾞｬﾝﾎﾞ      8462      1.4
            ｱﾘｴｰﾙｼﾞｪﾙ  詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ      6685      1.1
ｱﾀｯｸZERO ﾊﾟｰﾌｪｸﾄｽﾃｨｯｸ    ｳﾙﾄﾗｼﾞｬﾝﾎﾞ      6566      1.1
             ｱﾀｯｸ抗菌EX          本体通常      6101      1.0


In [38]:
# ── D-3b: Per-Size Destination Treemaps (Bold) ──────────────────────
# Individual treemap for each Ariel exit size showing where lapsed shoppers go

bold_lapsed_with_size = df_lapsed[
    (df_lapsed['sub_brand']==ARIEL_GB) & (~df_lapsed['last_size'].isin(EXCLUDED_SIZES))
]
ariel_dest_with_size = df_destination[df_destination['source_brand']==ARIEL_GB]

lapse_sizes = order_and_filter_sizes(bold_lapsed_with_size['last_size'].unique())
per_size_dest_rows = []

for lapse_size in lapse_sizes:
    # Get lapsed shoppers from this size
    lapsed_keys = bold_lapsed_with_size[bold_lapsed_with_size['last_size']==lapse_size]['shopper_key']
    size_dest = ariel_dest_with_size[ariel_dest_with_size['shopper_key'].isin(lapsed_keys)]

    size_dest_agg = size_dest.groupby(['next_sub_brand','next_size']).agg(
        shoppers=('shopper_key','nunique')).reset_index().sort_values('shoppers', ascending=False)

    total_from_size = lapsed_keys.nunique()
    tracked_from_size = size_dest_agg['shoppers'].sum()
    exit_from_size = total_from_size - tracked_from_size

    size_dest_agg['share_%'] = (size_dest_agg['shoppers'] / max(total_from_size,1) * 100).round(1)
    size_dest_agg['source_size'] = lapse_size

    # Add category exit
    tree_size = pd.concat([size_dest_agg, pd.DataFrame([{
        'next_sub_brand':'(Category Exit)', 'next_size':'No Purchase',
        'shoppers': max(exit_from_size, 0),
        'share_%': round(max(exit_from_size, 0)/max(total_from_size,1)*100,1),
        'source_size': lapse_size
    }])], ignore_index=True)

    if len(tree_size[tree_size['shoppers']>0]) > 0:
        fig_ts = px.treemap(tree_size[tree_size['shoppers']>0],
            path=['next_sub_brand','next_size'], values='shoppers',
            title=f'Post-Lapse Destination: 【{lapse_size}】 (Total lapsed: {total_from_size:,})',
            color='shoppers', color_continuous_scale='Reds')
        fig_ts.update_layout(height=500)
        fig_ts.show()

    per_size_dest_rows.append(size_dest_agg)

# ── DATA TABLE ────────────────────────────────────────────────────────
if per_size_dest_rows:
    per_size_dest_df = pd.concat(per_size_dest_rows, ignore_index=True)
    print('\n📊 DATA TABLE: Per-Size Post-Lapse Destination')
    print('=' * 100)
    for lapse_size in lapse_sizes:
        sd = per_size_dest_df[per_size_dest_df['source_size']==lapse_size]
        print(f'\n▶ From {lapse_size}:')
        print(sd[['next_sub_brand','next_size','shoppers','share_%']].head(10).to_string(index=False))
    per_size_dest_df.to_csv(OUTPUT_DIR / 'pane_d_per_size_destination.csv', index=False)


📊 DATA TABLE: Per-Size Post-Lapse Destination

▶ From 本体通常:
next_sub_brand     next_size  shoppers  share_%
ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ          本体通常     18059     10.2
     ｱﾘｴｰﾙｼﾞｪﾙ          本体通常     10714      6.0
     ｱﾘｴｰﾙｼﾞｪﾙ         詰替超特大      7472      4.2
      ｱﾀｯｸ抗菌EX 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ      5865      3.3
      ｱﾀｯｸ抗菌EX         詰替超特大      5602      3.2
ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ 詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ      4816      2.7
      ｱﾀｯｸ抗菌EX  詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ      3843      2.2
     ｱﾘｴｰﾙｼﾞｪﾙ 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ      3189      1.8
      ｱﾀｯｸ抗菌EX          本体通常      2834      1.6
       ﾅﾉｯｸｽﾜﾝ           本体大      2358      1.3

▶ From 詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ:
       next_sub_brand     next_size  shoppers  share_%
       ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ 詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ     16477      7.7
            ｱﾘｴｰﾙｼﾞｪﾙ         詰替超特大      6358      3.0
             ｱﾀｯｸ抗菌EX 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ      5460      2.6
            ｱﾘｴｰﾙｼﾞｪﾙ 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ      4371      2.0
ｱﾀｯｸZERO ﾊﾟｰﾌｪｸﾄｽﾃｨｯｸ    ｳﾙﾄﾗｼﾞｬﾝﾎﾞ      4130      1.9
             ｱﾀｯｸ抗菌EX         詰替超特大      4080      1.9
   

In [39]:
# ── D-4: Bold Lapse & Reverse Flow ──────────────────────────────────
bold_dest = df_destination[df_destination['source_brand']==BOLD_GB]
bold_lapsed_full = df_lapsed[df_lapsed['sub_brand']==BOLD_GB]

bold_dest_summary = bold_dest.groupby(['next_sub_brand','next_size']).agg(
    shoppers=('shopper_key','nunique')).reset_index().sort_values('shoppers', ascending=False)
total_atk_lapsed = bold_lapsed_full['shopper_key'].nunique()
total_atk_tracked = bold_dest_summary['shoppers'].sum()

print(f'\n📊 DATA TABLE: Bold Post-Lapse Destination')
print(f'Total lapsed: {total_atk_lapsed:,} | Tracked: {total_atk_tracked:,}')
bold_dest_summary['share_%'] = (bold_dest_summary['shoppers'] / total_atk_lapsed * 100).round(1)
print(bold_dest_summary.head(15).to_string(index=False))

# Ariel ↔ Bold flow summary
ariel_to_bold = dest_summary[dest_summary['next_sub_brand']==BOLD_GB]['shoppers'].sum()
bold_to_ariel = bold_dest_summary[bold_dest_summary['next_sub_brand']==ARIEL_GB]['shoppers'].sum() if len(bold_dest_summary) > 0 else 0
print(f'\n🔄 Cross-Flow: Ariel→Bold: {ariel_to_bold:,} | Bold→Ariel: {bold_to_ariel:,} | Net: {bold_to_ariel - ariel_to_bold:+,}')



📊 DATA TABLE: Bold Post-Lapse Destination
Total lapsed: 626,767 | Tracked: 325,821
next_sub_brand     next_size  shoppers  share_%
 ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ 詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ     21781      3.5
      ｱﾀｯｸ抗菌EX 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ     21227      3.4
      ｱﾀｯｸ抗菌EX         詰替超特大     15578      2.5
     ｱﾘｴｰﾙｼﾞｪﾙ         詰替超特大     14000      2.2
 ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ          本体通常     13709      2.2
      ｱﾀｯｸ抗菌EX  詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ     13080      2.1
    ﾎﾞｰﾙﾄﾞｼﾞｪﾙ         詰替超特大     12910      2.1
 ﾆｭｰﾋﾞｰｽﾞ ｼﾞｪﾙ         詰替超特大     11453      1.8
     ｱﾘｴｰﾙｼﾞｪﾙ          本体通常     10366      1.7
     ｱﾘｴｰﾙｼﾞｪﾙ 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ      8870      1.4
 ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ   詰替ﾒｶﾞｼﾞｬﾝﾎﾞ      8467      1.4
      ｱﾀｯｸ抗菌EX          本体通常      8464      1.4
    ﾎﾞｰﾙﾄﾞｼﾞｪﾙ          本体通常      8219      1.3
          ｴﾏｰﾙ         詰替超特大      8040      1.3
 ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ  詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ      6876      1.1

🔄 Cross-Flow: Ariel→Bold: 71,145 | Bold→Ariel: 55,438 | Net: -15,707


In [40]:
# ── E-3b: Bold Gel Ball Sankey — Trial Size → Repeat/Lapse ───────────
# Competitor shopper flow for comparison

atk_journey = df_cohort[
    (df_cohort['sub_brand']==BOLD_GB) &
    (~df_cohort['trial_size'].isin(EXCLUDED_SIZES))
].copy()

if len(atk_journey) > 0:
    atk_stage1 = atk_journey.groupby(['trial_size','outcome']).agg(
        count=('shopper_key','nunique'), avg_asp=('trial_asp','mean')).reset_index()

    atk_repeat_flow = atk_journey[atk_journey['outcome']=='Repeat'].groupby('repeat_size').agg(
        count=('shopper_key','nunique')).reset_index()
    atk_repeat_flow.columns = ['dest_label','count']
    atk_repeat_flow['dest_label'] = 'Bold ' + atk_repeat_flow['dest_label']
    atk_repeat_flow['outcome'] = 'Repeat'

    # Lapse destinations for Bold
    atk_lapsed_keys = atk_journey[atk_journey['outcome']=='Lapse']['shopper_key']
    atk_lapse_dest_raw = df_destination[
        (df_destination['source_brand']==BOLD_GB) &
        (df_destination['shopper_key'].isin(atk_lapsed_keys))
    ]
    if len(atk_lapse_dest_raw) > 0:
        atk_lapse_dest = atk_lapse_dest_raw.groupby(['next_sub_brand','next_size']).agg(
            count=('shopper_key','nunique')).reset_index()
        atk_lapse_dest['dest_label'] = atk_lapse_dest['next_sub_brand'] + ' ' + atk_lapse_dest['next_size'].fillna('')
    else:
        atk_lapse_dest = pd.DataFrame(columns=['dest_label','count'])

    total_atk_lapsed_s = atk_lapsed_keys.nunique()
    tracked_atk_s = atk_lapse_dest['count'].sum() if len(atk_lapse_dest) > 0 else 0
    cat_exit_atk = max(0, total_atk_lapsed_s - tracked_atk_s)

    atk_lapse_final = pd.concat([
        atk_lapse_dest[['dest_label','count']],
        pd.DataFrame([{'dest_label':'Category Exit', 'count': cat_exit_atk}])
    ], ignore_index=True)
    atk_lapse_final['outcome'] = 'Lapse'

    atk_stage2 = pd.concat([atk_repeat_flow, atk_lapse_final], ignore_index=True)
    atk_top = atk_stage2.nlargest(15, 'count')['dest_label'].tolist()
    atk_stage2['dest_simplified'] = atk_stage2['dest_label'].apply(lambda x: x if x in atk_top else 'Other Brands')
    atk_stage2 = atk_stage2.groupby(['outcome','dest_simplified']).agg(count=('count','sum')).reset_index()

    atk_trial_sizes = sorted(atk_journey['trial_size'].unique())
    atk_dests = sorted(atk_stage2['dest_simplified'].unique())
    atk_nodes = [f'Trial: {s}' for s in atk_trial_sizes] + ['Repeat','Lapse'] + [f'→ {d}' for d in atk_dests]
    atk_nidx = {n: i for i, n in enumerate(atk_nodes)}

    atk_src, atk_tgt, atk_val, bold_col = [], [], [], []
    for _, row in atk_stage1.iterrows():
        atk_src.append(atk_nidx[f'Trial: {row["trial_size"]}'])
        atk_tgt.append(atk_nidx[row['outcome']])
        atk_val.append(row['count'])
        bold_col.append('rgba(46,139,87,0.4)' if row['outcome']=='Repeat' else 'rgba(204,51,51,0.4)')
    for _, row in atk_stage2.iterrows():
        atk_src.append(atk_nidx[row['outcome']])
        atk_tgt.append(atk_nidx[f'→ {row["dest_simplified"]}'])
        atk_val.append(row['count'])
        if ARIEL_GB in row['dest_simplified']: bold_col.append('rgba(30,144,255,0.4)')
        elif 'Bold Gel Ball' in row['dest_simplified']: bold_col.append('rgba(255,99,71,0.5)')
        elif 'Exit' in row['dest_simplified']: bold_col.append('rgba(128,128,128,0.3)')
        else: bold_col.append('rgba(255,165,0,0.3)')

    atk_ncol = ['#FF6347']*len(atk_trial_sizes) + ['#2E8B57','#CC3333'] + ['#6495ED']*len(atk_dests)
    fig_atk_s = go.Figure(go.Sankey(
        node=dict(pad=15, thickness=20, label=atk_nodes, color=atk_ncol),
        link=dict(source=atk_src, target=atk_tgt, value=atk_val, color=bold_col)))
    fig_atk_s.update_layout(
        title_text='Bold Gel Ball: Shopper Flow — Trial Size → Repeat/Lapse → Destination',
        font_size=11, height=700, template='plotly_white')
    fig_atk_s.show()

    # ── DATA TABLE ────────────────────────────────────────────────────
    print('\n📊 DATA TABLE: Bold Gel Ball Shopper Flow — Stage 1')
    print('=' * 80)
    atk_s1_display = atk_stage1[['trial_size','outcome','count','avg_asp']].copy()
    atk_s1_display['avg_asp'] = atk_s1_display['avg_asp'].round(0)
    print(atk_s1_display.to_string(index=False))

    print('\n📊 DATA TABLE: Bold Gel Ball Shopper Flow — Stage 2 (Destinations)')
    print('=' * 80)
    print(atk_stage2.sort_values(['outcome','count'], ascending=[True,False]).to_string(index=False))

    atk_stage1.to_csv(OUTPUT_DIR / 'pane_e_attack_sankey_stage1.csv', index=False)
    atk_stage2.to_csv(OUTPUT_DIR / 'pane_e_attack_sankey_stage2.csv', index=False)
else:
    print('⚠️ No Bold cohort data available for Sankey')


📊 DATA TABLE: Bold Gel Ball Shopper Flow — Stage 1
   trial_size outcome  count  avg_asp
         本体通常   Lapse 165842    245.0
         本体通常  Repeat  85310    240.0
 詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ   Lapse  45209  2,444.0
 詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ  Repeat  20168  2,432.0
   詰替ﾃﾗｼﾞｬﾝﾎﾞ   Lapse  12276  3,089.0
   詰替ﾃﾗｼﾞｬﾝﾎﾞ  Repeat   5156  2,997.0
詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ   Lapse 134002    891.0
詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ  Repeat  76102    901.0
  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ   Lapse  41051  1,778.0
  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ  Repeat  20591  1,797.0

📊 DATA TABLE: Bold Gel Ball Shopper Flow — Stage 2 (Destinations)
outcome             dest_simplified  count
  Lapse               Category Exit 217460
  Lapse                Other Brands 105134
  Lapse      ｱﾀｯｸ抗菌EX 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ  13647
  Lapse ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ 詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ  10178
  Lapse              ｱﾀｯｸ抗菌EX 詰替超特大   9541
  Lapse             ｱﾘｴｰﾙｼﾞｪﾙ 詰替超特大   8295
  Lapse       ｱﾀｯｸ抗菌EX 詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ   8224
  Lapse            ﾎﾞｰﾙﾄﾞｼﾞｪﾙ 詰替超特大   6834
  Lapse          ｱﾘｴｰﾙｼﾞｪﾙﾎﾞｰﾙ 本体通常   6790
  Lapse         ﾆｭｰﾋﾞｰｽﾞ ｼﾞｪﾙ 詰替超

---
# Pane E — Shopper Flow & Price Effectiveness
*Sankey flow visualization, ASP-annotated funnel, and strategic bubble map.*


In [41]:
# ── E-1: Funnel by Entry Size ─────────────────────────────────────────
df_journey = df_cohort[df_cohort['sub_brand']==ARIEL_GB].copy()
df_journey_filt = df_journey[~df_journey['trial_size'].isin(EXCLUDED_SIZES)]

funnel_data = df_journey_filt.groupby('trial_size').agg(
    total_trial=('shopper_key','nunique'),
    repeat_shoppers=('outcome', lambda x: (x=='Repeat').sum()),
    lapse_shoppers=('outcome', lambda x: (x=='Lapse').sum()),
    avg_trial_asp=('trial_asp', 'mean'),
).reset_index()
funnel_data['repeat_rate_%'] = (funnel_data['repeat_shoppers'] / funnel_data['total_trial'] * 100).round(1)
funnel_data['lapse_rate_%'] = (funnel_data['lapse_shoppers'] / funnel_data['total_trial'] * 100).round(1)
funnel_data['_sort'] = funnel_data['trial_size'].map({s:i for i,s in enumerate(SIZE_ORDER)}).fillna(99)
funnel_data = funnel_data.sort_values('_sort').drop(columns='_sort')

print('📊 DATA TABLE: Shopper Funnel by Entry Size')
print('=' * 90)
print(funnel_data.to_string(index=False))

# Stacked bar
sizes = order_and_filter_sizes(funnel_data['trial_size'].unique())
sizes_display = list(reversed(sizes))
fd = funnel_data.set_index('trial_size')

fig = go.Figure()
fig.add_trace(go.Bar(y=sizes_display, x=fd.loc[sizes_display,'repeat_shoppers'], name='Repeat',
    orientation='h', marker_color='#2E8B57',
    text=[f"{v:,} ({r:.1f}%)" for v,r in zip(fd.loc[sizes_display,'repeat_shoppers'], fd.loc[sizes_display,'repeat_rate_%'])],
    textposition='inside'))
fig.add_trace(go.Bar(y=sizes_display, x=fd.loc[sizes_display,'lapse_shoppers'], name='Lapse',
    orientation='h', marker_color='#CC3333',
    text=[f"{v:,} ({r:.1f}%)" for v,r in zip(fd.loc[sizes_display,'lapse_shoppers'], fd.loc[sizes_display,'lapse_rate_%'])],
    textposition='inside'))
fig.update_layout(barmode='stack', title='Trial Outcome by Entry Size — Where Do We Lose Shoppers?',
    xaxis_title='Shoppers', template='plotly_white', height=400)
fig.show()

funnel_data.to_csv(OUTPUT_DIR / 'pane_e_funnel.csv', index=False)


📊 DATA TABLE: Shopper Funnel by Entry Size
   trial_size  total_trial  repeat_shoppers  lapse_shoppers  avg_trial_asp  repeat_rate_%  lapse_rate_%
         本体通常       184693            50833          133860          240.3           27.5          72.5
詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ       184783            65145          119638          900.5           35.3          64.7
  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ        69350            22610           46740        1,790.1           32.6          67.4
 詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ        63122            18829           44293        2,349.0           29.8          70.2
   詰替ﾃﾗｼﾞｬﾝﾎﾞ        26329             8006           18323        3,033.3           30.4          69.6


In [42]:
# ── E-2: Sankey — Trial Size → Repeat/Lapse → Destination ────────────
flow_stage1 = df_journey.groupby(['trial_size','outcome']).agg(
    count=('shopper_key','nunique'), avg_asp=('trial_asp','mean')).reset_index()

repeat_flow = df_journey[df_journey['outcome']=='Repeat'].groupby('repeat_size').agg(
    count=('shopper_key','nunique')).reset_index()
repeat_flow.columns = ['dest_label', 'count']
repeat_flow['dest_label'] = 'Ariel ' + repeat_flow['dest_label']
repeat_flow['outcome'] = 'Repeat'

lapse_dest = dest_summary.copy()
lapse_dest['dest_label'] = lapse_dest['next_sub_brand'] + ' ' + lapse_dest['next_size'].fillna('')
lapse_dest = lapse_dest.rename(columns={'shoppers':'count'})
total_lapsed_j = (df_journey['outcome']=='Lapse').sum()
cat_exit = max(0, total_lapsed_j - lapse_dest['count'].sum())
lapse_dest = pd.concat([lapse_dest[['dest_label','count']],
    pd.DataFrame([{'dest_label':'Category Exit', 'count':cat_exit}])], ignore_index=True)
lapse_dest['outcome'] = 'Lapse'

flow_stage2 = pd.concat([repeat_flow, lapse_dest], ignore_index=True)
top_dests = flow_stage2.nlargest(15, 'count')['dest_label'].tolist()
flow_stage2['dest_simplified'] = flow_stage2['dest_label'].apply(lambda x: x if x in top_dests else 'Other Brands')
flow_stage2 = flow_stage2.groupby(['outcome','dest_simplified']).agg(count=('count','sum')).reset_index()

trial_sizes = sorted(df_journey['trial_size'].unique())
destinations = sorted(flow_stage2['dest_simplified'].unique())
all_nodes = [f'Trial: {s}' for s in trial_sizes] + ['Repeat','Lapse'] + [f'→ {d}' for d in destinations]
node_idx = {n:i for i,n in enumerate(all_nodes)}

sources, targets, values, colors = [], [], [], []
for _, row in flow_stage1.iterrows():
    sources.append(node_idx[f'Trial: {row["trial_size"]}'])
    targets.append(node_idx[row['outcome']])
    values.append(row['count'])
    colors.append('rgba(46,139,87,0.4)' if row['outcome']=='Repeat' else 'rgba(204,51,51,0.4)')
for _, row in flow_stage2.iterrows():
    sources.append(node_idx[row['outcome']])
    targets.append(node_idx[f'→ {row["dest_simplified"]}'])
    values.append(row['count'])
    if 'Ariel' in row['dest_simplified']: colors.append('rgba(30,144,255,0.4)')
    elif BOLD_GB in row['dest_simplified']: colors.append('rgba(255,99,71,0.5)')
    elif 'Exit' in row['dest_simplified']: colors.append('rgba(128,128,128,0.3)')
    else: colors.append('rgba(255,165,0,0.3)')

node_colors = ['#1E90FF']*len(trial_sizes) + ['#2E8B57','#CC3333'] + ['#6495ED']*len(destinations)
fig = go.Figure(go.Sankey(
    node=dict(pad=15, thickness=20, label=all_nodes, color=node_colors),
    link=dict(source=sources, target=targets, value=values, color=colors)))
fig.update_layout(title_text='Shopper Flow: Trial Size → Repeat/Lapse → Destination', font_size=11, height=700, template='plotly_white')
fig.show()

# ── DATA TABLE: Sankey Flow Summary ──────────────────────────────────
print('\n📊 DATA TABLE: Shopper Flow — Stage 1 (Trial Size → Outcome)')
print('=' * 80)
fs1_display = flow_stage1[['trial_size','outcome','count','avg_asp']].copy()
fs1_display['avg_asp'] = fs1_display['avg_asp'].round(0)
print(fs1_display.to_string(index=False))

print('\n📊 DATA TABLE: Shopper Flow — Stage 2 (Outcome → Destination)')
print('=' * 80)
fs2_display = flow_stage2[['outcome','dest_simplified','count']].sort_values(['outcome','count'], ascending=[True,False])
print(fs2_display.to_string(index=False))

flow_stage1.to_csv(OUTPUT_DIR / 'pane_e_sankey_stage1.csv', index=False)
flow_stage2.to_csv(OUTPUT_DIR / 'pane_e_sankey_stage2.csv', index=False)


📊 DATA TABLE: Shopper Flow — Stage 1 (Trial Size → Outcome)
   trial_size outcome  count  avg_asp
         本体通常   Lapse 133860    238.0
         本体通常  Repeat  50833    246.0
        詰替超特大   Lapse     11    497.0
        詰替超特大  Repeat      1    718.0
    詰替超ｼﾞｬﾝﾎﾞ   Lapse    182    447.0
    詰替超ｼﾞｬﾝﾎﾞ  Repeat     48    460.0
 詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ   Lapse  44293  2,340.0
 詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ  Repeat  18829  2,371.0
   詰替ﾃﾗｼﾞｬﾝﾎﾞ   Lapse  18323  3,067.0
   詰替ﾃﾗｼﾞｬﾝﾎﾞ  Repeat   8006  2,957.0
詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ   Lapse 119638    896.0
詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ  Repeat  65145    909.0
  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ   Lapse  46740  1,778.0
  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ  Repeat  22610  1,814.0
          ｿﾉﾀ   Lapse    396    243.0
          ｿﾉﾀ  Repeat     83    263.0

📊 DATA TABLE: Shopper Flow — Stage 2 (Outcome → Destination)
outcome                 dest_simplified  count
  Lapse                    Other Brands 170246
  Lapse                   Category Exit  49231
  Lapse    ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ 詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ  25895
  Lapse             ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ 本体通常  23

In [43]:
# ── E-2b: Per-Size Sankey — Trial Size → Repeat/Lapse → Destination ───
# One standalone Sankey per size for detailed view

ariel_dest_all = df_destination[df_destination['source_brand'] == ARIEL_GB]
per_size_sizes = order_and_filter_sizes(df_journey['trial_size'].unique())

for size in per_size_sizes:
    # --- Stage 1: Trial → Repeat / Lapse ---
    size_journey = df_journey[df_journey['trial_size'] == size]
    s1 = size_journey.groupby('outcome').agg(
        count=('shopper_key', 'nunique'),
        avg_asp=('trial_asp', 'mean')
    ).reset_index()

    # --- Stage 2a: Repeat → destination (repeat size within Bold) ---
    rep_flow = size_journey[size_journey['outcome'] == 'Repeat'].groupby('repeat_size').agg(
        count=('shopper_key', 'nunique')
    ).reset_index()
    rep_flow.columns = ['dest_label', 'count']
    rep_flow['dest_label'] = 'Ariel ' + rep_flow['dest_label']
    rep_flow['outcome'] = 'Repeat'

    # --- Stage 2b: Lapse → destination (next sub-brand / size from df_destination) ---
    lapsed_keys = size_journey[size_journey['outcome'] == 'Lapse']['shopper_key']
    lapse_raw = ariel_dest_all[ariel_dest_all['shopper_key'].isin(lapsed_keys)]
    if len(lapse_raw) > 0:
        lapse_dest = lapse_raw.groupby(['next_sub_brand', 'next_size']).agg(
            count=('shopper_key', 'nunique')
        ).reset_index()
        lapse_dest['dest_label'] = lapse_dest['next_sub_brand'] + ' ' + lapse_dest['next_size'].fillna('')
    else:
        lapse_dest = pd.DataFrame(columns=['dest_label', 'count'])

    total_lapsed_s = lapsed_keys.nunique()
    tracked_lapsed = lapse_dest['count'].sum() if len(lapse_dest) > 0 else 0
    cat_exit_s = max(0, total_lapsed_s - tracked_lapsed)
    lapse_final = pd.concat([
        lapse_dest[['dest_label', 'count']],
        pd.DataFrame([{'dest_label': 'Category Exit', 'count': cat_exit_s}])
    ], ignore_index=True)
    lapse_final['outcome'] = 'Lapse'

    # --- Combine stage 2 and simplify ---
    s2 = pd.concat([rep_flow, lapse_final], ignore_index=True)
    s2['count'] = pd.to_numeric(s2['count'], errors='coerce').fillna(0).astype(int)
    top_d = s2.nlargest(12, 'count')['dest_label'].tolist()
    s2['dest_simplified'] = s2['dest_label'].apply(lambda x: x if x in top_d else 'Other Brands')
    s2 = s2.groupby(['outcome', 'dest_simplified']).agg(count=('count', 'sum')).reset_index()

    # --- Build Sankey nodes & links ---
    nodes_list = [f'Trial: {size}'] + ['Repeat', 'Lapse'] + [f'→ {d}' for d in sorted(s2['dest_simplified'].unique())]
    nidx = {n: i for i, n in enumerate(nodes_list)}

    sources, targets, values, colors = [], [], [], []
    for _, row in s1.iterrows():
        sources.append(nidx[f'Trial: {size}'])
        targets.append(nidx[row['outcome']])
        values.append(row['count'])
        colors.append('rgba(46,139,87,0.4)' if row['outcome'] == 'Repeat' else 'rgba(204,51,51,0.4)')
    for _, row in s2.iterrows():
        sources.append(nidx[row['outcome']])
        targets.append(nidx[f'→ {row["dest_simplified"]}'])
        values.append(row['count'])
        if 'Ariel' in row['dest_simplified']:
            colors.append('rgba(30,144,255,0.4)')
        elif BOLD_GB in row['dest_simplified']:
            colors.append('rgba(255,99,71,0.5)')
        elif 'Exit' in row['dest_simplified']:
            colors.append('rgba(128,128,128,0.3)')
        else:
            colors.append('rgba(255,165,0,0.3)')

    n_dests = len(sorted(s2['dest_simplified'].unique()))
    node_colors = ['#1E90FF'] + ['#2E8B57', '#CC3333'] + ['#6495ED'] * n_dests

    fig_s = go.Figure(go.Sankey(
        node=dict(pad=15, thickness=20, label=nodes_list, color=node_colors),
        link=dict(source=sources, target=targets, value=values, color=colors)
    ))
    # Compute repeat rate for subtitle
    total_s = s1['count'].sum()
    repeat_s = s1.loc[s1['outcome'] == 'Repeat', 'count'].sum() if 'Repeat' in s1['outcome'].values else 0
    rr = round(repeat_s / max(total_s, 1) * 100, 1)
    fig_s.update_layout(
        title_text=(f'【{size}】Shopper Flow: Trial → Repeat/Lapse → Destination<br>'
                    f'<sup>Total trial: {total_s:,} | Repeat: {repeat_s:,} ({rr}%) | Lapse: {total_s - repeat_s:,} ({round(100-rr,1)}%)</sup>'),
        font_size=11, height=600, template='plotly_white'
    )
    fig_s.show()

    # Data table
    print(f'\n📊 {size} — Stage 1')
    print(s1.to_string(index=False))
    print(f'\n📊 {size} — Stage 2 (Top destinations)')
    print(s2.sort_values(['outcome', 'count'], ascending=[True, False]).to_string(index=False))


📊 本体通常 — Stage 1
outcome  count  avg_asp
  Lapse 133860    238.3
 Repeat  50833    245.7

📊 本体通常 — Stage 2 (Top destinations)
outcome              dest_simplified  count
  Lapse                Category Exit  55951
  Lapse                 Other Brands  38982
  Lapse          ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ 本体通常  10350
  Lapse               ｱﾘｴｰﾙｼﾞｪﾙ 本体通常   6522
  Lapse              ｱﾘｴｰﾙｼﾞｪﾙ 詰替超特大   4981
  Lapse       ｱﾀｯｸ抗菌EX 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ   4601
  Lapse               ｱﾀｯｸ抗菌EX 詰替超特大   4053
  Lapse ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ 詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ   3160
  Lapse        ｱﾀｯｸ抗菌EX 詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ   2932
  Lapse      ｱﾘｴｰﾙｼﾞｪﾙ 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ   2328
 Repeat                   Ariel 本体通常  34690
 Repeat          Ariel 詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ  11655
 Repeat                 Other Brands   2473
 Repeat           Ariel 詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ   2015

📊 詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ — Stage 1
outcome  count  avg_asp
  Lapse 119638    895.9
 Repeat  65145    909.0

📊 詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ — Stage 2 (Top destinations)
outcome              dest_simplified  count
  Lapse                C

In [44]:
# ── E-3: Strategic Bubble Map ─────────────────────────────────────────
asp_flow = []
for size in order_and_filter_sizes(df_journey['trial_size'].unique()):
    sd = df_journey[df_journey['trial_size']==size]
    total = len(sd)
    rep_d = sd[sd['outcome']=='Repeat']
    lap_d = sd[sd['outcome']=='Lapse']
    asp_flow.append({
        'size': size, 'trial_shoppers': total,
        'avg_trial_asp': sd['trial_asp'].mean(),
        'repeat_shoppers': len(rep_d),
        'repeat_rate_%': round(len(rep_d)/max(total,1)*100, 1),
        'avg_repeat_asp': rep_d['repeat_asp'].mean() if len(rep_d)>0 else None,
        'lapse_shoppers': len(lap_d),
        'lapse_rate_%': round(len(lap_d)/max(total,1)*100, 1),
    })
df_asp_flow = pd.DataFrame(asp_flow)

print('📊 DATA TABLE: ASP-Annotated Funnel')
print('=' * 100)
print(df_asp_flow.to_string(index=False))

fig = px.scatter(df_asp_flow, x='avg_trial_asp', y='repeat_rate_%', size='trial_shoppers',
    text='size', color='lapse_rate_%', color_continuous_scale='RdYlGn_r',
    labels={'avg_trial_asp':'Avg Trial ASP (JPY)', 'repeat_rate_%':'Repeat Rate (%)',
            'trial_shoppers':'Trial Shoppers', 'lapse_rate_%':'Lapse Rate (%)'},
    title='Strategic Map: Which Size + Price Needs Fixing? (Big bubble = big opportunity, Red = high lapse)')
fig.update_traces(textposition='top center', marker=dict(sizemin=10))
fig.update_layout(template='plotly_white', height=600)
fig.show()

df_asp_flow.to_csv(OUTPUT_DIR / 'pane_e_strategic_map.csv', index=False)


📊 DATA TABLE: ASP-Annotated Funnel
         size  trial_shoppers  avg_trial_asp  repeat_shoppers  repeat_rate_%  avg_repeat_asp  lapse_shoppers  lapse_rate_%
         本体通常          184693          240.3            50833           27.5           530.0          133860          72.5
詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ          184783          900.5            65145           35.3         1,042.6          119638          64.7
  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ           69350        1,790.1            22610           32.6         1,593.2           46740          67.4
 詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ           63122        2,349.0            18829           29.8         1,983.9           44293          70.2
   詰替ﾃﾗｼﾞｬﾝﾎﾞ           26329        3,033.3             8006           30.4         2,406.5           18323          69.6


---
# Pane F — Price Point Impact Analysis: 詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ (Hyper Jumbo)
*How does the ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ selling price affect shopper behavior? At ¥798 / ¥848 / ¥898 / ¥980, how do shoppers shift sizes, lapse, or switch brands?*

### Key Questions
1. **Repeat vs Lapse** — At each price point, what % of shoppers repeat vs lapse?
2. **Size Migration** — Among repeaters, do they stay on ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ or shift to larger/smaller sizes?
3. **Brand Switching** — Among lapsers, do they switch to Ariel or exit the category?
4. **Price Gap Effect** — As ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ price rises, does the gap to ﾒｶﾞｼﾞｬﾝﾎﾞ/超ﾒｶﾞｼﾞｬﾝﾎﾞ drive size-up?

### Price Bands
| Target Price | Band Range | Label |
|:---:|:---:|:---|
| ≤¥748 | ≤ 773 | ≤¥748 |
| ¥798 | 774–824 | ~¥798 |
| ¥848 | 825–874 | ~¥848 |
| ¥898 | 875–939 | ~¥898 |
| ¥980 | ≥ 940 | ~¥980 |

In [45]:
# ═══════════════════════════════════════════════════════════════════════
# F-1: ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ ASP Band → Shopper Outcome (Repeat Rate & Lapse Rate)
# ═══════════════════════════════════════════════════════════════════════
# Segment Ariel ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ trial shoppers by 50-JPY ASP bins
# (uniform bin width = consistent methodology with G-1)

HJ_SIZE = '詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ'

# Filter: Ariel ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ trial shoppers
hj_cohort = df_cohort[
    (df_cohort['sub_brand'] == ARIEL_GB) &
    (df_cohort['trial_size'] == HJ_SIZE) &
    (df_cohort['trial_asp'].notna())
].copy()
hj_cohort['asp_band'] = (hj_cohort['trial_asp'] // 50 * 50).astype('Int64')
hj_cohort['price_band'] = hj_cohort['asp_band'].apply(lambda b: f'~¥{b + 49:,}')

# ── Outcome summary per asp_band ─────────────────────────────────────
hj_outcome = hj_cohort.groupby(['asp_band', 'price_band', 'outcome']).agg(
    shoppers=('shopper_key', 'nunique')
).reset_index()

hj_pivot = hj_outcome.pivot_table(index=['asp_band', 'price_band'], columns='outcome',
    values='shoppers', fill_value=0).reset_index()
if 'Repeat' in hj_pivot.columns and 'Lapse' in hj_pivot.columns:
    hj_pivot['total'] = hj_pivot['Repeat'] + hj_pivot['Lapse']
    hj_pivot['repeat_rate_%'] = (hj_pivot['Repeat'] / hj_pivot['total'] * 100).round(1)
    hj_pivot['lapse_rate_%'] = (hj_pivot['Lapse'] / hj_pivot['total'] * 100).round(1)
    hj_pivot['avg_asp'] = hj_cohort.groupby('asp_band')['trial_asp'].mean().reindex(
        hj_pivot['asp_band']).values

# Order by asp_band (numeric sort)
hj_pivot = hj_pivot.sort_values('asp_band').reset_index(drop=True)
band_order = hj_pivot['asp_band'].tolist()          # list of int asp_band values
band_label = {b: f'~¥{b + 49:,}' for b in band_order}  # display labels

print(f'📊 F-1: ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ Repeat/Lapse Rate by ASP Band (50-JPY bins, {len(band_order)} bands)')
print('=' * 100)
print(hj_pivot.to_string(index=False))

# ── Bar chart: Repeat Rate & Lapse Rate by asp_band ──────────────────
fig_f1 = go.Figure()
fig_f1.add_trace(go.Bar(
    x=hj_pivot['price_band'], y=hj_pivot['repeat_rate_%'],
    name='Repeat Rate (%)', marker_color='#2E8B57',
    text=[f"{v:.1f}%<br>({n:,})" for v, n in zip(hj_pivot['repeat_rate_%'], hj_pivot['Repeat'])],
    textposition='outside'
))
fig_f1.add_trace(go.Bar(
    x=hj_pivot['price_band'], y=hj_pivot['lapse_rate_%'],
    name='Lapse Rate (%)', marker_color='#CC3333',
    text=[f"{v:.1f}%<br>({n:,})" for v, n in zip(hj_pivot['lapse_rate_%'], hj_pivot['Lapse'])],
    textposition='outside'
))

# Add total shoppers annotation
for i, row in hj_pivot.iterrows():
    fig_f1.add_annotation(x=row['price_band'], y=105, text=f"N={int(row['total']):,}",
        showarrow=False, font=dict(size=11, color='#555'))

fig_f1.update_layout(
    barmode='group',
    title=('詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ — Repeat Rate vs Lapse Rate by ASP Band (50-JPY bins)<br>'
           f'<sup>{len(band_order)} bins from ~¥{band_order[0]+49:,} to ~¥{band_order[-1]+49:,}</sup>'),
    yaxis_title='Rate (%)', xaxis_title='ASP Band',
    template='plotly_white', height=550,
    yaxis=dict(range=[0, 110])
)
fig_f1.show()

hj_pivot.to_csv(OUTPUT_DIR / 'pane_f_hj_outcome_by_price.csv', index=False)

📊 F-1: ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ Repeat/Lapse Rate by ASP Band (50-JPY bins, 28 bands)
 asp_band price_band    Lapse   Repeat    total  repeat_rate_%  lapse_rate_%  avg_asp
        0       ~¥49    693.0    468.0  1,161.0           40.3          59.7      1.1
       50       ~¥99     39.0     21.0     60.0           35.0          65.0     77.2
      100      ~¥149     61.0     18.0     79.0           22.8          77.2    124.9
      150      ~¥199     72.0     36.0    108.0           33.3          66.7    176.6
      200      ~¥249     78.0     30.0    108.0           27.8          72.2    228.5
      250      ~¥299    282.0     58.0    340.0           17.1          82.9    272.5
      300      ~¥349    207.0     66.0    273.0           24.2          75.8    325.9
      350      ~¥399    404.0     98.0    502.0           19.5          80.5    379.9
      400      ~¥449  1,233.0    292.0  1,525.0           19.1          80.9    430.1
      450      ~¥499  3,785.0    910.0  4,695.0           19.4     

In [46]:
# ═══════════════════════════════════════════════════════════════════════
# F-2: Size Migration by ASP Band — Where Do Repeaters Go?
# ═══════════════════════════════════════════════════════════════════════
# Among repeaters at each asp_band: same size / size up / size down?

hj_repeaters = hj_cohort[
    (hj_cohort['outcome'] == 'Repeat') &
    (hj_cohort['repeat_size'].notna()) &
    (~hj_cohort['repeat_size'].isin(EXCLUDED_SIZES))
].copy()

hj_repeaters['migration'] = hj_repeaters.apply(
    lambda row: classify_migration(row['trial_size'], row['repeat_size'], SIZE_ORDER), axis=1
)

# More granular: map to actual repeat size
hj_repeaters['repeat_size_label'] = hj_repeaters['repeat_size'].apply(
    lambda s: s if s in SIZE_ORDER else 'Other'
)

# ── Migration direction summary ──────────────────────────────────────
mig_by_band = hj_repeaters.groupby(['asp_band', 'price_band', 'migration']).agg(
    shoppers=('shopper_key', 'nunique')
).reset_index()

mig_totals = mig_by_band.groupby('asp_band')['shoppers'].transform('sum')
mig_by_band['pct'] = (mig_by_band['shoppers'] / mig_totals * 100).round(1)
mig_by_band = mig_by_band.sort_values(['asp_band', 'migration'])

print('📊 F-2a: Size Migration Direction (Same / Size Down / Size Up) by ASP Band')
print('=' * 80)
print(mig_by_band.to_string(index=False))

# ── Specific repeat size breakdown ───────────────────────────────────
size_by_band = hj_repeaters.groupby(['asp_band', 'price_band', 'repeat_size_label']).agg(
    shoppers=('shopper_key', 'nunique')
).reset_index()
size_totals = size_by_band.groupby('asp_band')['shoppers'].transform('sum')
size_by_band['pct'] = (size_by_band['shoppers'] / size_totals * 100).round(1)
size_by_band = size_by_band.sort_values(['asp_band', 'repeat_size_label'])

print('\n📊 F-2b: Specific Repeat Size Breakdown by ASP Band')
print('=' * 80)
for bv in band_order:
    sub = size_by_band[size_by_band['asp_band'] == bv]
    if len(sub) == 0: continue
    print(f'\n▶ ASP Band: {band_label[bv]}')
    print(sub[['repeat_size_label', 'shoppers', 'pct']].sort_values('pct', ascending=False).to_string(index=False))

# ── Stacked bar: Migration direction per asp_band ────────────────────
_x_labels = [band_label[b] for b in band_order]
fig_mig = go.Figure()
for mtype in MIGRATE_ORDER:
    sub = mig_by_band[mig_by_band['migration'] == mtype].copy()
    sub = sub.set_index('asp_band').reindex(band_order).reset_index()
    sub['_label'] = sub['asp_band'].map(band_label)
    fig_mig.add_trace(go.Bar(
        x=sub['_label'], y=sub['pct'],
        name=mtype, marker_color=MIGRATE_COLORS.get(mtype, '#888'),
        text=sub['pct'].apply(lambda v: f'{v:.1f}%' if pd.notna(v) else ''),
        textposition='inside', insidetextanchor='middle'
    ))

fig_mig.update_layout(
    barmode='stack',
    title=('詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ Repeaters — Size Migration by ASP Band (50-JPY bins)<br>'
           '<sup>As price rises, do repeaters stay on same size or shift to larger/smaller?</sup>'),
    yaxis_title='% of Repeaters', xaxis_title='ASP Band',
    template='plotly_white', height=550,
    yaxis=dict(range=[0, 105]),
    legend=dict(orientation='h', y=-0.12)
)
fig_mig.show()

# ── Grouped bar: Specific repeat size per asp_band ───────────────────
repeat_sizes_ordered = [s for s in SIZE_ORDER if s in size_by_band['repeat_size_label'].values]
SIZE_COLORS = {
    '本体通常': '#FF9800', '詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ': '#4ECDC4',
    '詰替ﾒｶﾞｼﾞｬﾝﾎﾞ': '#2196F3', '詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ': '#9C27B0',
    '詰替ﾃﾗｼﾞｬﾝﾎﾞ': '#E91E63', 'Other': '#999'
}

fig_size = go.Figure()
for size_label in repeat_sizes_ordered:
    sub = size_by_band[size_by_band['repeat_size_label'] == size_label].copy()
    sub = sub.set_index('asp_band').reindex(band_order).reset_index()
    sub['_label'] = sub['asp_band'].map(band_label)
    fig_size.add_trace(go.Bar(
        x=sub['_label'], y=sub['pct'],
        name=size_label, marker_color=SIZE_COLORS.get(size_label, '#888'),
        text=sub['pct'].apply(lambda v: f'{v:.1f}%' if pd.notna(v) else ''),
        textposition='inside', insidetextanchor='middle'
    ))

fig_size.update_layout(
    barmode='stack',
    title=('詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ Repeaters — Repeat Size Distribution by ASP Band (50-JPY bins)<br>'
           '<sup>Which specific size do repeaters choose at each price level?</sup>'),
    yaxis_title='% of Repeaters', xaxis_title='ASP Band',
    template='plotly_white', height=550,
    yaxis=dict(range=[0, 105]),
    legend=dict(orientation='h', y=-0.12)
)
fig_size.show()

mig_by_band.to_csv(OUTPUT_DIR / 'pane_f_hj_migration_direction.csv', index=False)
size_by_band.to_csv(OUTPUT_DIR / 'pane_f_hj_repeat_size_detail.csv', index=False)

📊 F-2a: Size Migration Direction (Same / Size Down / Size Up) by ASP Band
 asp_band price_band             migration  shoppers   pct
        0       ~¥49           ① Same Size       329  70.3
        0       ~¥49 ② Size Down (Smaller)        41   8.8
        0       ~¥49    ③ Size Up (Larger)        98  20.9
       50       ~¥99           ① Same Size        14  66.7
       50       ~¥99 ② Size Down (Smaller)         2   9.5
       50       ~¥99    ③ Size Up (Larger)         5  23.8
      100      ~¥149           ① Same Size        11  61.1
      100      ~¥149 ② Size Down (Smaller)         3  16.7
      100      ~¥149    ③ Size Up (Larger)         4  22.2
      150      ~¥199           ① Same Size        28  77.8
      150      ~¥199 ② Size Down (Smaller)         1   2.8
      150      ~¥199    ③ Size Up (Larger)         7  19.4
      200      ~¥249           ① Same Size        14  46.7
      200      ~¥249 ② Size Down (Smaller)         4  13.3
      200      ~¥249    ③ Size Up (Larger

In [47]:

# ═══════════════════════════════════════════════════════════════════════
# F-3: Lapse Destination by ASP Band — Where Do Lapsers Go?
# ═══════════════════════════════════════════════════════════════════════
# Among lapsed ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ shoppers at each asp_band:
# → which brand/size do they buy next? Or exit the category?

hj_lapsers = hj_cohort[hj_cohort['outcome'] == 'Lapse'].copy()

# Join with destination data
ariel_dest_all = df_destination[df_destination['source_brand'] == ARIEL_GB]

dest_rows = []
for bv in band_order:
    lapsed_keys = hj_lapsers[hj_lapsers['asp_band'] == bv]['shopper_key']
    total_lapsed = lapsed_keys.nunique()
    if total_lapsed == 0:
        continue

    # Find destination
    band_dest = ariel_dest_all[ariel_dest_all['shopper_key'].isin(lapsed_keys)]
    if len(band_dest) > 0:
        dest_agg = band_dest.groupby(['next_sub_brand', 'next_size']).agg(
            shoppers=('shopper_key', 'nunique')
        ).reset_index()
    else:
        dest_agg = pd.DataFrame(columns=['next_sub_brand', 'next_size', 'shoppers'])

    tracked = pd.to_numeric(dest_agg['shoppers'], errors='coerce').sum() if len(dest_agg) > 0 else 0
    cat_exit = max(0, total_lapsed - int(tracked))

    # Simplify destinations into categories
    def classify_dest(row_sb, row_sz):
        if row_sb == ARIEL_GB:
            return f'Ariel {row_sz}' if row_sz else 'Ariel (other)'
        elif row_sb == BOLD_GB:
            return f'Bold {row_sz}' if row_sz else 'Bold (other)'
        else:
            return f'{row_sb}' if row_sb else 'Other'

    if len(dest_agg) > 0:
        dest_agg['dest_category'] = dest_agg.apply(
            lambda r: classify_dest(r['next_sub_brand'], r['next_size']), axis=1
        )
        dest_summary_band = dest_agg.groupby('dest_category')['shoppers'].sum().reset_index()
    else:
        dest_summary_band = pd.DataFrame(columns=['dest_category', 'shoppers'])

    # Add category exit
    dest_summary_band = pd.concat([
        dest_summary_band,
        pd.DataFrame([{'dest_category': 'Category Exit', 'shoppers': cat_exit}])
    ], ignore_index=True)

    # Ensure shoppers is numeric
    dest_summary_band['shoppers'] = pd.to_numeric(dest_summary_band['shoppers'], errors='coerce').fillna(0)

    dest_summary_band['asp_band'] = bv
    dest_summary_band['price_band'] = band_label[bv]
    dest_summary_band['total_lapsed'] = total_lapsed
    dest_summary_band['pct'] = (dest_summary_band['shoppers'] / total_lapsed * 100).round(1)
    dest_rows.append(dest_summary_band)

if dest_rows:
    df_hj_dest = pd.concat(dest_rows, ignore_index=True)

    print('📊 F-3: Post-Lapse Destination by ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ ASP Band')
    print('=' * 100)
    for bv in band_order:
        sub = df_hj_dest[df_hj_dest['asp_band'] == bv].sort_values('shoppers', ascending=False)
        if len(sub) == 0: continue
        total = sub['total_lapsed'].iloc[0]
        print(f'\n▶ {band_label[bv]} (Total lapsed: {total:,})')
        print(sub[['dest_category', 'shoppers', 'pct']].head(10).to_string(index=False))

    # ── Simplified destination: Ariel stay / Bold switch / Other brand / Category Exit ──
    def simplify_dest(cat):
        if cat.startswith('Ariel'):
            return '① Stay Ariel (other size)'
        elif cat.startswith('Bold') or BOLD_GB in cat:
            return '② Switch to Bold'
        elif cat == 'Category Exit':
            return '④ Category Exit'
        else:
            return '③ Other Brand'

    df_hj_dest['dest_simplified'] = df_hj_dest['dest_category'].apply(simplify_dest)
    dest_simple = df_hj_dest.groupby(['asp_band', 'price_band', 'dest_simplified']).agg(
        shoppers=('shoppers', 'sum'),
        total_lapsed=('total_lapsed', 'first')
    ).reset_index()
    dest_simple['pct'] = (dest_simple['shoppers'] / dest_simple['total_lapsed'] * 100).round(1)

    print('\n📊 F-3b: Simplified Lapse Destination by ASP Band')
    print('=' * 80)
    dest_simple_pivot = dest_simple.pivot_table(
        index='asp_band', columns='dest_simplified', values='pct', fill_value=0
    )
    dest_simple_pivot = dest_simple_pivot.reindex(band_order)
    print(dest_simple_pivot.round(1).to_string())

    # ── Stacked bar chart ─────────────────────────────────────────────
    DEST_COLORS = {
        '① Stay Ariel (other size)': '#1E90FF',
        '② Switch to Bold':       '#FF6347',
        '③ Other Brand':           '#FFB347',
        '④ Category Exit':         '#999999',
    }
    DEST_ORDER = ['① Stay Ariel (other size)', '② Switch to Bold', '③ Other Brand', '④ Category Exit']

    fig_dest = go.Figure()
    for dest_type in DEST_ORDER:
        sub = dest_simple[dest_simple['dest_simplified'] == dest_type].copy()
        sub = sub.set_index('asp_band').reindex(band_order).reset_index()
        sub['_label'] = sub['asp_band'].map(band_label)
        fig_dest.add_trace(go.Bar(
            x=sub['_label'], y=sub['pct'],
            name=dest_type, marker_color=DEST_COLORS.get(dest_type, '#888'),
            text=sub['pct'].apply(lambda v: f'{v:.1f}%' if pd.notna(v) else ''),
            textposition='inside', insidetextanchor='middle'
        ))

    fig_dest.update_layout(
        barmode='stack',
        title=('詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ Lapsers — Post-Lapse Destination by ASP Band (50-JPY bins)<br>'
               '<sup>As price rises, do lapsers switch to Bold, stay Ariel, or exit the category?</sup>'),
        yaxis_title='% of Lapsed Shoppers', xaxis_title='ASP Band',
        template='plotly_white', height=550,
        yaxis=dict(range=[0, 105]),
        legend=dict(orientation='h', y=-0.12)
    )
    fig_dest.show()

    df_hj_dest.to_csv(OUTPUT_DIR / 'pane_f_hj_lapse_destination.csv', index=False)
    dest_simple.to_csv(OUTPUT_DIR / 'pane_f_hj_lapse_dest_simple.csv', index=False)
else:
    print('⚠️ No lapse destination data for ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ')


📊 F-3: Post-Lapse Destination by ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ ASP Band

▶ ~¥49 (Total lapsed: 693)
        dest_category  shoppers  pct
        Category Exit       368 53.1
            ｱﾘｴｰﾙｼﾞｪﾙ        49  7.1
   Bold 詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ        47  6.8
             ｱﾀｯｸ抗菌EX        42  6.1
ｱﾀｯｸZERO ﾊﾟｰﾌｪｸﾄｽﾃｨｯｸ        32  4.6
              ﾅﾉｯｸｽﾜﾝ        22  3.2
             ｱﾀｯｸZERO        22  3.2
                 ｴﾏｰﾙ        21  3.0
           ﾎﾞｰﾙﾄﾞｼﾞｪﾙ        14  2.0
     Bold 詰替ﾒｶﾞｼﾞｬﾝﾎﾞ        11  1.6

▶ ~¥99 (Total lapsed: 39)
     dest_category  shoppers  pct
     Category Exit        19 48.7
Bold 詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ         3  7.7
          ｱﾀｯｸ抗菌EX         3  7.7
         ｱﾘｴｰﾙｼﾞｪﾙ         3  7.7
     ﾆｭｰﾋﾞｰｽﾞ ｼﾞｪﾙ         2  5.1
        ﾎﾞｰﾙﾄﾞｼﾞｪﾙ         2  5.1
       ﾆｭｰﾋﾞｰｽﾞ 粉末         2  5.1
  Bold 詰替ﾒｶﾞｼﾞｬﾝﾎﾞ         1  2.6
          ｱﾀｯｸZERO         1  2.6
              ｱｸﾛﾝ         1  2.6

▶ ~¥149 (Total lapsed: 61)
        dest_category  shoppers  pct
        Category Exit        36 59.0
        

In [48]:
# ═══════════════════════════════════════════════════════════════════════
# F-4: Price Gap Effect — ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ vs Other Sizes
# ═══════════════════════════════════════════════════════════════════════
# Calculate concurrent ASP of other sizes from weekly data, and show
# how the price gap to ﾒｶﾞｼﾞｬﾝﾎﾞ / 超ﾒｶﾞｼﾞｬﾝﾎﾞ / ﾃﾗｼﾞｬﾝﾎﾞ changes
# per asp_band, and whether it drives migration.

# Average concurrent ASP per size (Ariel only, across the analysis period)
ariel_avg_asp_by_size = df_asp_weekly[
    (df_asp_weekly['sub_brand'] == ARIEL_GB) &
    (~df_asp_weekly['size_code'].isin(EXCLUDED_SIZES))
].groupby('size_code').agg(
    avg_asp=('weighted_asp', 'mean'),
    median_asp=('weighted_asp', 'median')
).reset_index()

print('📊 Average ASP by Ariel Size (weekly market average)')
print(ariel_avg_asp_by_size.round(0).to_string(index=False))

# ── For each HJ asp_band, compute the gap to other sizes ─────────────
# Use the asp_band midpoint (asp_band + 25) as the HJ reference price
gap_rows = []
for bv in band_order:
    hj_price = bv + 25   # midpoint of the 50-JPY bin
    for _, size_row in ariel_avg_asp_by_size.iterrows():
        size_name = size_row['size_code']
        size_avg = size_row['avg_asp']
        gap = size_avg - hj_price
        gap_pct = (gap / hj_price * 100)
        gap_rows.append({
            'asp_band': bv,
            'price_band': band_label[bv],
            'hj_ref_price': hj_price,
            'other_size': size_name,
            'other_avg_asp': round(size_avg, 0),
            'price_gap_jpy': round(gap, 0),
            'price_gap_pct': round(gap_pct, 1),
        })

df_gap = pd.DataFrame(gap_rows)
df_gap = df_gap[df_gap['other_size'] != HJ_SIZE]  # exclude self

# Show gap vs each upsize option
print('\n📊 F-4a: Price Gap — ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ ASP Band vs Other Ariel Sizes (JPY)')
print('=' * 100)
gap_pivot = df_gap.pivot_table(index='other_size', columns='price_band',
    values='price_gap_jpy', aggfunc='first')
_col_order = [band_label[b] for b in band_order if band_label[b] in gap_pivot.columns]
gap_pivot = gap_pivot.reindex(columns=_col_order)
gap_pivot = gap_pivot.reindex([s for s in SIZE_ORDER if s in gap_pivot.index])
print(gap_pivot.round(0).to_string())

# ── Heatmap: Price gap ───────────────────────────────────────────────
fig_gap = px.imshow(
    gap_pivot.values,
    x=gap_pivot.columns.tolist(),
    y=gap_pivot.index.tolist(),
    text_auto='.0f',
    color_continuous_scale='RdBu',
    color_continuous_midpoint=0,
    labels=dict(x='ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ ASP Band', y='Other Ariel Size', color='Gap (¥)'),
    title=('Price Gap: Other Size ASP minus ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ Reference Price<br>'
           '<sup>Blue = cheaper than HJ (unlikely upsize) | Red = pricier (upsize stretch required)</sup>')
)
fig_gap.update_layout(height=450, template='plotly_white')
fig_gap.show()

# ── Combine: Migration % vs Price Gap ────────────────────────────────
# For each asp_band, show: ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ→ﾒｶﾞｼﾞｬﾝﾎﾞ migration % alongside the gap
upsize_targets = ['詰替ﾒｶﾞｼﾞｬﾝﾎﾞ', '詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ', '詰替ﾃﾗｼﾞｬﾝﾎﾞ']
upsize_migration = size_by_band[size_by_band['repeat_size_label'].isin(upsize_targets)].copy()
upsize_agg = upsize_migration.groupby('asp_band').agg(
    upsize_shoppers=('shoppers', 'sum'),
    upsize_pct=('pct', 'sum')
).reset_index()

# Also get same-size retention
same_size = size_by_band[size_by_band['repeat_size_label'] == HJ_SIZE].copy()
same_size = same_size[['asp_band', 'pct']].rename(columns={'pct': 'same_size_pct'})

# Merge with gap to ﾒｶﾞｼﾞｬﾝﾎﾞ (the most natural upsize)
mega_gap = df_gap[df_gap['other_size'] == '詰替ﾒｶﾞｼﾞｬﾝﾎﾞ'][['asp_band', 'price_gap_jpy']].rename(
    columns={'price_gap_jpy': 'gap_to_mega'}
)

combo = upsize_agg.merge(same_size, on='asp_band', how='outer').merge(mega_gap, on='asp_band', how='outer')
combo = combo.sort_values('asp_band')
combo['price_band'] = combo['asp_band'].map(band_label)

# Also add repeat rate and lapse rate from hj_pivot
combo = combo.merge(
    hj_pivot[['asp_band', 'repeat_rate_%', 'lapse_rate_%']],
    on='asp_band', how='left'
)

print('\n📊 F-4b: Combined — Migration % vs Price Gap to ﾒｶﾞｼﾞｬﾝﾎﾞ')
print('=' * 100)
print(combo.to_string(index=False))

# ── Dual-axis chart: Upsize % vs Gap to Mega ─────────────────────────
fig_combo = make_subplots(specs=[[{"secondary_y": True}]])

fig_combo.add_trace(go.Bar(
    x=combo['price_band'], y=combo['same_size_pct'],
    name='Same Size (HJ) %', marker_color='#4ECDC4', opacity=0.8,
    text=combo['same_size_pct'].apply(lambda v: f'{v:.1f}%' if pd.notna(v) else ''),
    textposition='outside'
), secondary_y=False)

fig_combo.add_trace(go.Bar(
    x=combo['price_band'], y=combo['upsize_pct'],
    name='Size Up (MJ+) %', marker_color='#2ECC71', opacity=0.8,
    text=combo['upsize_pct'].apply(lambda v: f'{v:.1f}%' if pd.notna(v) else ''),
    textposition='outside'
), secondary_y=False)

fig_combo.add_trace(go.Bar(
    x=combo['price_band'], y=combo['lapse_rate_%'],
    name='Lapse Rate %', marker_color='#E74C3C', opacity=0.6,
    text=combo['lapse_rate_%'].apply(lambda v: f'{v:.1f}%' if pd.notna(v) else ''),
    textposition='outside'
), secondary_y=False)

fig_combo.add_trace(go.Scatter(
    x=combo['price_band'], y=combo['gap_to_mega'],
    name='Gap to MegaJumbo (¥)', mode='lines+markers+text',
    line=dict(color='#FF6347', width=3), marker=dict(size=10),
    text=combo['gap_to_mega'].apply(lambda v: f'¥{v:+.0f}' if pd.notna(v) else ''),
    textposition='top center'
), secondary_y=True)

fig_combo.update_layout(
    barmode='group',
    title=('詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ — ASP Band Impact: Same Size / Size Up / Lapse vs Gap to ﾒｶﾞｼﾞｬﾝﾎﾞ<br>'
           '<sup>As HJ price rises, gap to MegaJumbo narrows → does upsize % increase?</sup>'),
    template='plotly_white', height=600,
    legend=dict(orientation='h', y=-0.15)
)
fig_combo.update_yaxes(title_text='% of Shoppers', secondary_y=False)
fig_combo.update_yaxes(title_text='Price Gap to MegaJumbo (¥)', secondary_y=True, showgrid=False)
fig_combo.show()

df_gap.to_csv(OUTPUT_DIR / 'pane_f_hj_price_gap.csv', index=False)
combo.to_csv(OUTPUT_DIR / 'pane_f_hj_gap_vs_migration.csv', index=False)

📊 Average ASP by Ariel Size (weekly market average)
    size_code  avg_asp  median_asp
         本体通常    305.0       301.0
 詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ  2,311.0     2,353.0
   詰替ﾃﾗｼﾞｬﾝﾎﾞ  2,974.0     2,877.0
詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ    945.0       944.0
  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ  1,906.0     1,876.0

📊 F-4a: Price Gap — ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ ASP Band vs Other Ariel Sizes (JPY)
price_band      ~¥49    ~¥99   ~¥149   ~¥199   ~¥249   ~¥299   ~¥349   ~¥399   ~¥449   ~¥499   ~¥549   ~¥599   ~¥649   ~¥699   ~¥749   ~¥799   ~¥849   ~¥899   ~¥949   ~¥999  ~¥1,049  ~¥1,099  ~¥1,149  ~¥1,199  ~¥1,249  ~¥1,299  ~¥1,349  ~¥1,999
other_size                                                                                                                                                                                                                                          
本体通常           280.0   230.0   180.0   130.0    80.0    30.0   -20.0   -70.0  -120.0  -170.0  -220.0  -270.0  -320.0  -370.0  -420.0  -470.0  -520.0  -570.0  -620.0  -670.0   -720

In [49]:
# ═══════════════════════════════════════════════════════════════════════
# F-5: Full Shopper Flow by ASP Band — Sankey per Price Point
# ═══════════════════════════════════════════════════════════════════════
# For each asp_band: Trial(HJ) → Repeat(which size?) / Lapse(where?)

for bv in band_order:
    bl = band_label[bv]
    band_shoppers = hj_cohort[hj_cohort['asp_band'] == bv]
    if len(band_shoppers) == 0:
        continue

    total = band_shoppers['shopper_key'].nunique()
    avg_asp_band = band_shoppers['trial_asp'].mean()

    # Stage 1: Outcome
    s1 = band_shoppers.groupby('outcome').agg(
        count=('shopper_key', 'nunique')
    ).reset_index()

    # Stage 2a: Repeat → Which size?
    rep = band_shoppers[
        (band_shoppers['outcome'] == 'Repeat') &
        (~band_shoppers['repeat_size'].isin(EXCLUDED_SIZES))
    ]
    rep_flow = rep.groupby('repeat_size').agg(count=('shopper_key', 'nunique')).reset_index()
    rep_flow.columns = ['dest_label', 'count']
    rep_flow['dest_label'] = 'Ariel ' + rep_flow['dest_label']
    rep_flow['outcome'] = 'Repeat'

    # Stage 2b: Lapse → Where?
    lapsed_keys = band_shoppers[band_shoppers['outcome'] == 'Lapse']['shopper_key']
    lapse_raw = ariel_dest_all[ariel_dest_all['shopper_key'].isin(lapsed_keys)]
    if len(lapse_raw) > 0:
        lapse_dest_b = lapse_raw.groupby(['next_sub_brand', 'next_size']).agg(
            count=('shopper_key', 'nunique')
        ).reset_index()
        lapse_dest_b['dest_label'] = lapse_dest_b['next_sub_brand'] + ' ' + lapse_dest_b['next_size'].fillna('')
    else:
        lapse_dest_b = pd.DataFrame(columns=['dest_label', 'count'])

    total_lapsed_b = lapsed_keys.nunique()
    tracked_b = lapse_dest_b['count'].sum() if len(lapse_dest_b) > 0 else 0
    cat_exit_b = max(0, total_lapsed_b - tracked_b)

    lapse_final_b = pd.concat([
        lapse_dest_b[['dest_label', 'count']],
        pd.DataFrame([{'dest_label': 'Category Exit', 'count': cat_exit_b}])
    ], ignore_index=True)
    lapse_final_b['outcome'] = 'Lapse'

    # Combine & simplify
    s2 = pd.concat([rep_flow, lapse_final_b], ignore_index=True)
    s2['count'] = pd.to_numeric(s2['count'], errors='coerce').fillna(0).astype(int)
    top_d = s2.nlargest(10, 'count')['dest_label'].tolist()
    s2['dest_simplified'] = s2['dest_label'].apply(lambda x: x if x in top_d else 'Other')
    s2 = s2.groupby(['outcome', 'dest_simplified']).agg(count=('count', 'sum')).reset_index()

    # Build Sankey
    nodes_list = [f'Trial HJ\n{bl}'] + ['Repeat', 'Lapse'] + [f'→ {d}' for d in sorted(s2['dest_simplified'].unique())]
    nidx = {n: i for i, n in enumerate(nodes_list)}

    sources, targets, values, colors = [], [], [], []
    for _, row in s1.iterrows():
        sources.append(nidx[f'Trial HJ\n{bl}'])
        targets.append(nidx[row['outcome']])
        values.append(row['count'])
        colors.append('rgba(46,139,87,0.4)' if row['outcome'] == 'Repeat' else 'rgba(204,51,51,0.4)')
    for _, row in s2.iterrows():
        sources.append(nidx[row['outcome']])
        targets.append(nidx[f'→ {row["dest_simplified"]}'])
        values.append(row['count'])
        if 'Ariel' in row['dest_simplified']:
            colors.append('rgba(30,144,255,0.4)')
        elif BOLD_GB in row['dest_simplified']:
            colors.append('rgba(255,99,71,0.5)')
        elif 'Exit' in row['dest_simplified']:
            colors.append('rgba(128,128,128,0.3)')
        else:
            colors.append('rgba(255,165,0,0.3)')

    n_dests = len(sorted(s2['dest_simplified'].unique()))
    node_colors = ['#1E90FF'] + ['#2E8B57', '#CC3333'] + ['#6495ED'] * n_dests

    rep_n = s1.loc[s1['outcome'] == 'Repeat', 'count'].sum() if 'Repeat' in s1['outcome'].values else 0
    rr = round(rep_n / max(total, 1) * 100, 1)

    fig_sk = go.Figure(go.Sankey(
        node=dict(pad=15, thickness=20, label=nodes_list, color=node_colors),
        link=dict(source=sources, target=targets, value=values, color=colors)
    ))
    fig_sk.update_layout(
        title_text=(f'【{bl} | Avg ¥{avg_asp_band:,.0f}】ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ Shopper Flow<br>'
                    f'<sup>N={total:,} | Repeat: {rep_n:,} ({rr}%) | Lapse: {total-rep_n:,} ({100-rr:.1f}%)</sup>'),
        font_size=11, height=550, template='plotly_white'
    )
    fig_sk.show()

    print(f'\n📊 {bl} — Stage 1:')
    print(s1.to_string(index=False))
    print(f'📊 {bl} — Stage 2:')
    print(s2.sort_values(['outcome', 'count'], ascending=[True, False]).to_string(index=False))


📊 ~¥49 — Stage 1:
outcome  count
  Lapse    693
 Repeat    468
📊 ~¥49 — Stage 2:
outcome              dest_simplified  count
  Lapse                Category Exit    368
  Lapse                        Other    217
  Lapse ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ 詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ     47
  Lapse              ｱﾘｴｰﾙｼﾞｪﾙ 詰替超特大     16
  Lapse                   ｴﾏｰﾙ 詰替超特大     16
  Lapse               ｱﾀｯｸ抗菌EX 詰替超特大     15
  Lapse       ｱﾀｯｸ抗菌EX 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ     14
 Repeat          Ariel 詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ    329
 Repeat            Ariel 詰替ﾒｶﾞｼﾞｬﾝﾎﾞ     49
 Repeat                   Ariel 本体通常     41
 Repeat             Ariel 詰替ﾃﾗｼﾞｬﾝﾎﾞ     36
 Repeat                        Other     13

📊 ~¥99 — Stage 1:
outcome  count
  Lapse     39
 Repeat     21
📊 ~¥99 — Stage 2:
outcome              dest_simplified  count
  Lapse                Category Exit     19
  Lapse                        Other     11
  Lapse ﾎﾞｰﾙﾄﾞｼﾞｪﾙﾎﾞｰﾙ 詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ      3
  Lapse              ｱﾘｴｰﾙｼﾞｪﾙ 詰替超特大      2
  Lapse             ﾆｭｰﾋﾞｰｽﾞ 粉末 本体通常      2


In [50]:
# ═══════════════════════════════════════════════════════════════════════
# F-6: Comprehensive Price Point Summary — Decision Matrix (50-JPY bins)
# ═══════════════════════════════════════════════════════════════════════
# Uniform 50-JPY asp_band bins — same methodology as G-1.
# Columns: ASP Band | Freq (week×store) | TrialEff% | N | Repeat% | Lapse%
#          | SameHJ% | SizeUp% | SizeDn% | →Bold% | →Exit% | Gap→MJ

MIN_FREQ_FLAG = 100   # bands with freq < this are flagged with * (low exposure)

# ── Build frequency (week × store count) per HJ asp_band ─────────────
_hj_ws = df_week_store[df_week_store['size_code'] == HJ_SIZE].copy()
hj_freq = _hj_ws.groupby('asp_band')['week_store_count'].sum().to_dict()

# ── Build decision matrix ────────────────────────────────────────────
summary_rows = []
for bv in band_order:
    bl = band_label[bv]
    row = {'asp_band': bv, 'price_band': bl}

    # Total / Repeat / Lapse from hj_pivot (keyed by asp_band)
    hp = hj_pivot[hj_pivot['asp_band'] == bv]
    if len(hp) > 0:
        row['total_shoppers'] = int(hp['total'].values[0])
        row['repeat_rate_%'] = hp['repeat_rate_%'].values[0]
        row['lapse_rate_%'] = hp['lapse_rate_%'].values[0]
        row['avg_trial_asp'] = round(hp['avg_asp'].values[0], 0) if pd.notna(hp['avg_asp'].values[0]) else None
    else:
        row['total_shoppers'] = 0
        row['repeat_rate_%'] = 0.0
        row['lapse_rate_%'] = 0.0
        row['avg_trial_asp'] = None

    # Frequency (week × store) and Trial Efficiency
    freq = int(hj_freq.get(bv, 0))
    row['frequency'] = freq
    row['trial_efficiency_%'] = round(row['total_shoppers'] / freq * 100, 1) if freq > 0 else None

    # Same-size retention
    ss = same_size[same_size['asp_band'] == bv]
    row['same_size_retention_%'] = ss['same_size_pct'].values[0] if len(ss) > 0 else 0.0

    # Upsize %
    ua = upsize_agg[upsize_agg['asp_band'] == bv]
    row['size_up_%'] = ua['upsize_pct'].values[0] if len(ua) > 0 else 0.0

    # Downsize (本体通常)
    ds = size_by_band[(size_by_band['asp_band'] == bv) & (size_by_band['repeat_size_label'] == '本体通常')]
    row['size_down_%'] = ds['pct'].values[0] if len(ds) > 0 else 0.0

    # Lapse → Bold switch / Category exit
    if 'dest_simple' in dir() and len(dest_simple) > 0:
        bold_sw = dest_simple[
            (dest_simple['asp_band'] == bv) &
            (dest_simple['dest_simplified'] == '② Switch to Bold')
        ]
        row['lapse_to_bold_%'] = bold_sw['pct'].values[0] if len(bold_sw) > 0 else 0.0

        cat_ex = dest_simple[
            (dest_simple['asp_band'] == bv) &
            (dest_simple['dest_simplified'] == '④ Category Exit')
        ]
        row['category_exit_%'] = cat_ex['pct'].values[0] if len(cat_ex) > 0 else 0.0
    else:
        row['lapse_to_bold_%'] = 0.0
        row['category_exit_%'] = 0.0

    # Gap to MegaJumbo
    mg = mega_gap[mega_gap['asp_band'] == bv]
    row['gap_to_mega_jpy'] = mg['gap_to_mega'].values[0] if len(mg) > 0 else None

    summary_rows.append(row)

df_decision = pd.DataFrame(summary_rows)

# ── Print table ──────────────────────────────────────────────────────
print('=' * 144)
print('📊 F-6: 詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ PRICE POINT DECISION MATRIX (50-JPY bins)')
print(f'  * = low frequency (freq < {MIN_FREQ_FLAG:,}) — interpret with caution')
print('=' * 144)
print()
print(f'{"ASP Band":>13s} {"Freq(w×s)":>10s} {"TrialEff%":>10s} {"N":>7s} '
      f'{"Repeat%":>8s} {"Lapse%":>8s} '
      f'{"SameHJ%":>8s} {"SizeUp%":>8s} {"SizeDn%":>8s} '
      f'{"→Bold%":>8s} {"→Exit%":>8s} {"Gap→MJ":>8s}')
print('-' * 144)
for _, r in df_decision.iterrows():
    gap_str  = f'¥{r["gap_to_mega_jpy"]:+.0f}'   if pd.notna(r.get('gap_to_mega_jpy'))    else '—'
    eff_str  = f'{r["trial_efficiency_%"]:.1f}%'  if pd.notna(r.get('trial_efficiency_%')) else '—'
    freq_val = int(r['frequency']) if pd.notna(r['frequency']) else 0
    flag     = '*' if freq_val < MIN_FREQ_FLAG else ' '
    band_str = f'{r["price_band"]}{flag}'
    print(f'{band_str:>14s} {freq_val:>10,} {eff_str:>10s} {r["total_shoppers"]:>7,} '
          f'{r["repeat_rate_%"]:>7.1f}% {r["lapse_rate_%"]:>7.1f}% '
          f'{r["same_size_retention_%"]:>7.1f}% {r["size_up_%"]:>7.1f}% {r["size_down_%"]:>7.1f}% '
          f'{r["lapse_to_bold_%"]:>7.1f}% {r["category_exit_%"]:>7.1f}% {gap_str:>8s}')

print('\n\n📋 KEY FINDINGS:')
print('─' * 80)

if len(df_decision) > 0:
    best_repeat  = df_decision.loc[df_decision['repeat_rate_%'].idxmax()]
    worst_lapse  = df_decision.loc[df_decision['lapse_rate_%'].idxmax()]
    best_upsize  = df_decision.loc[df_decision['size_up_%'].idxmax()]
    _eff_df      = df_decision.dropna(subset=['trial_efficiency_%'])
    best_eff_row = _eff_df.loc[_eff_df['trial_efficiency_%'].idxmax()] if len(_eff_df) > 0 else None

    print(f'  1. BEST REPEAT RATE    : {best_repeat["price_band"]} → {best_repeat["repeat_rate_%"]:.1f}% repeat')
    print(f'  2. WORST LAPSE RISK    : {worst_lapse["price_band"]} → {worst_lapse["lapse_rate_%"]:.1f}% lapse')
    print(f'  3. MOST SIZE-UP        : {best_upsize["price_band"]} → {best_upsize["size_up_%"]:.1f}% migrate to larger size')
    if best_eff_row is not None:
        print(f'  4. BEST TRIAL EFFIC.   : {best_eff_row["price_band"]} → {best_eff_row["trial_efficiency_%"]:.1f}%  '
              f'(N={int(best_eff_row["total_shoppers"]):,} / freq={int(best_eff_row["frequency"]):,} × 100)')

    print()
    print('  NOTE: Bins are now uniform 50-JPY width → directly comparable with G-1 all-sizes matrix.')
    print('  Look for the contiguous range of bins with highest Repeat% and lowest →Bold% + →Exit%.')

df_decision.to_csv(OUTPUT_DIR / 'pane_f_hj_decision_matrix.csv', index=False)
print(f'\n✅ F-6 complete — saved to {OUTPUT_DIR}/pane_f_hj_decision_matrix.csv')

📊 F-6: 詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ PRICE POINT DECISION MATRIX (50-JPY bins)
  * = low frequency (freq < 100) — interpret with caution

     ASP Band  Freq(w×s)  TrialEff%       N  Repeat%   Lapse%  SameHJ%  SizeUp%  SizeDn%   →Bold%   →Exit%   Gap→MJ
------------------------------------------------------------------------------------------------------------------------------------------------
         ~¥49*          0          —   1,161    40.3%    59.7%    70.3%    21.0%     8.8%    10.8%    53.1%   ¥+1881
         ~¥99*          0          —      60    35.0%    65.0%    66.7%    23.8%     9.5%    10.3%    48.7%   ¥+1831
        ~¥149*          0          —      79    22.8%    77.2%    61.1%    22.3%    16.7%     1.6%    59.0%   ¥+1781
        ~¥199*          0          —     108    33.3%    66.7%    77.8%    19.5%     2.8%     4.2%    55.6%   ¥+1731
        ~¥249*          0          —     108    27.8%    72.2%    46.7%    40.0%    13.3%    14.1%    46.2%   ¥+1681
        ~¥299*          0       

In [51]:

# ═══════════════════════════════════════════════════════════════════════
# G-1: Multi-Size Price Point Decision Matrix (All Sizes)
# ═══════════════════════════════════════════════════════════════════════
# Same decision-matrix format as F-6 but generalised to all 5 Ariel sizes.
# ** UNIFIED METHODOLOGY **: both F-6 and G-1 now compute repeat/lapse
# directly from df_cohort with 50-JPY asp_band bins — NO upstream filtering.
# Frequency from market_freq is used for display & trial efficiency only.
# Low-frequency bands (freq < MIN_FREQ_FLAG_G) are flagged with *.
# Gap→Next = asp_band midpoint minus the next size's overall avg ASP.

MIN_FREQ_FLAG_G = 100   # flag threshold for this section

# ── Helper: size abbreviation for SameXX% header ────────────────────
_SIZE_ABBR = {
    '本体通常':          '本体',
    '詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ': 'HJ',
    '詰替ﾒｶﾞｼﾞｬﾝﾎﾞ':  'MJ',
    '詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ':  '超MJ',
    '詰替ﾃﾗｼﾞｬﾝﾎﾞ':  'TJ',
}

def _size_abbr(s):
    return _SIZE_ABBR.get(s, s[:4])

# ── Helper: compute per-asp_band lapse destination ratios ────────────
def _lapse_dest_by_band(target_size):
    """Return df with asp_band, bold_pct, exit_pct for lapsed shoppers."""
    sz_lapsers = df_cohort[
        (df_cohort['sub_brand'] == ARIEL_GB) &
        (df_cohort['trial_size'] == target_size) &
        (df_cohort['outcome'] == 'Lapse') &
        (df_cohort['trial_asp'].notna())
    ].copy()
    sz_lapsers['asp_band'] = (sz_lapsers['trial_asp'] // 50 * 50).astype('Int64')

    rows = []
    for band_val in sz_lapsers['asp_band'].unique():
        lapsed_keys = sz_lapsers[sz_lapsers['asp_band'] == band_val]['shopper_key']
        total_lap = lapsed_keys.nunique()
        if total_lap == 0:
            rows.append({'asp_band': band_val, 'bold_pct': 0.0, 'exit_pct': 0.0})
            continue
        band_dest = ariel_dest_all[ariel_dest_all['shopper_key'].isin(lapsed_keys)]
        bold_n  = band_dest[band_dest['next_sub_brand'] == BOLD_GB]['shopper_key'].nunique()
        tracked = band_dest['shopper_key'].nunique()
        exit_n  = max(0, total_lap - tracked)
        rows.append({
            'asp_band': band_val,
            'bold_pct': round(bold_n  / total_lap * 100, 1),
            'exit_pct': round(exit_n  / total_lap * 100, 1),
        })
    return pd.DataFrame(rows)

# ── Helper: compute per-asp_band migration ratios ────────────────────
def _migration_by_band(target_size):
    """Return df with asp_band, same_pct, up_pct, dn_pct among repeaters."""
    sz_rep = df_cohort[
        (df_cohort['sub_brand'] == ARIEL_GB) &
        (df_cohort['trial_size'] == target_size) &
        (df_cohort['outcome'] == 'Repeat') &
        (df_cohort['repeat_size'].notna()) &
        (~df_cohort['repeat_size'].isin(EXCLUDED_SIZES)) &
        (df_cohort['trial_asp'].notna())
    ].copy()
    sz_rep['asp_band'] = (sz_rep['trial_asp'] // 50 * 50).astype('Int64')
    sz_rep['migration'] = sz_rep.apply(
        lambda r: classify_migration(r['trial_size'], r['repeat_size'], SIZE_ORDER), axis=1
    )

    mig_agg = sz_rep.groupby(['asp_band', 'migration']).agg(
        n=('shopper_key', 'nunique')
    ).reset_index()
    total_agg = sz_rep.groupby('asp_band').agg(total=('shopper_key', 'nunique')).reset_index()
    mig_agg = mig_agg.merge(total_agg, on='asp_band')
    mig_agg['pct'] = (mig_agg['n'] / mig_agg['total'] * 100).round(1)

    _SAME = '① Same Size'
    _DN   = '② Size Down (Smaller)'
    _UP   = '③ Size Up (Larger)'

    def _get_pct(band_val, mtype):
        sub = mig_agg[(mig_agg['asp_band'] == band_val) & (mig_agg['migration'] == mtype)]
        return float(sub['pct'].values[0]) if len(sub) > 0 else 0.0

    bands = mig_agg['asp_band'].unique()
    return pd.DataFrame([{
        'asp_band': int(b),
        'same_pct': _get_pct(b, _SAME),
        'up_pct':   _get_pct(b, _UP),
        'dn_pct':   _get_pct(b, _DN),
    } for b in bands])

# ── Main builder (UNIFIED: computes from df_cohort, not pp_total) ────
def build_size_matrix(target_size):
    """Build the price point decision matrix for a given Ariel size.
    
    Repeat/Lapse computed directly from df_cohort (unfiltered) — identical
    methodology to F-1/F-6 for HJ, ensuring exact-match numbers.
    """
    # ── Compute trial/repeat/lapse from df_cohort directly ───────────
    sz_cohort = df_cohort[
        (df_cohort['sub_brand'] == ARIEL_GB) &
        (df_cohort['trial_size'] == target_size) &
        (df_cohort['trial_asp'].notna())
    ].copy()
    if len(sz_cohort) == 0:
        return pd.DataFrame()

    sz_cohort['asp_band'] = (sz_cohort['trial_asp'] // 50 * 50).astype('Int64')

    outcome_agg = sz_cohort.groupby(['asp_band', 'outcome']).agg(
        shoppers=('shopper_key', 'nunique')
    ).reset_index()

    base = outcome_agg.pivot_table(
        index='asp_band', columns='outcome', values='shoppers', fill_value=0
    ).reset_index()
    if 'Repeat' not in base.columns:
        base['Repeat'] = 0
    if 'Lapse' not in base.columns:
        base['Lapse'] = 0
    base['total'] = base['Repeat'] + base['Lapse']
    base = base[base['total'] > 0].sort_values('asp_band').reset_index(drop=True)

    if len(base) == 0:
        return pd.DataFrame()

    # Migration & destination ratios
    mig   = _migration_by_band(target_size)
    dest  = _lapse_dest_by_band(target_size)

    # Frequency from market_freq (display only, no filtering)
    freq_map = market_freq[market_freq['trial_size'] == target_size].set_index('asp_band')['store_exec_freq'].to_dict()

    # Total shopper N lookup (df_universe: all shoppers per size × ASP band)
    univ_sub = df_universe[df_universe['trial_size'] == target_size]
    univ_map = univ_sub.set_index('asp_band')['all_shoppers'].to_dict()

    # Gap to next size
    sz_idx    = SIZE_ORDER.index(target_size) if target_size in SIZE_ORDER else -1
    next_size = SIZE_ORDER[sz_idx + 1] if 0 <= sz_idx < len(SIZE_ORDER) - 1 else None
    next_avg  = None
    if next_size is not None and 'ariel_avg_asp_by_size' in dir():
        nrow = ariel_avg_asp_by_size[ariel_avg_asp_by_size['size_code'] == next_size]
        if len(nrow) > 0:
            next_avg = float(nrow['avg_asp'].values[0])

    rows = []
    for _, b in base.iterrows():
        bv    = int(b['asp_band'])
        total = int(b['total'])
        rep_n = int(b['Repeat'])
        lap_n = int(b['Lapse'])
        freq  = int(freq_map.get(bv, 0))

        repeat_pct = round(rep_n  / total * 100, 1) if total > 0 else 0.0
        lapse_pct  = round(lap_n  / total * 100, 1) if total > 0 else 0.0
        trial_eff  = round(total  / freq  * 100, 1) if freq  > 0 else None

        m  = mig[mig['asp_band']  == bv]
        d  = dest[dest['asp_band'] == bv]
        gap = round(next_avg - (bv + 25), 0) if next_avg is not None else None
        total_n = int(univ_map.get(bv, 0))

        rows.append({
            'size':              target_size,
            'asp_band':          bv,
            'price_band':        f'~¥{bv + 49:,}',
            'frequency':         freq,
            'trial_efficiency_%': trial_eff,
            'Total N':           total_n,
            'Trial N':           total,
            'repeat_%':          repeat_pct,
            'lapse_%':           lapse_pct,
            'same_size_%':       m['same_pct'].values[0] if len(m) > 0 else 0.0,
            'size_up_%':         m['up_pct'].values[0]   if len(m) > 0 else 0.0,
            'size_dn_%':         m['dn_pct'].values[0]   if len(m) > 0 else 0.0,
            '→Bold%':            d['bold_pct'].values[0] if len(d) > 0 else 0.0,
            '→Exit%':            d['exit_pct'].values[0] if len(d) > 0 else 0.0,
            'gap_to_next_jpy':   gap,
            '_freq_flag':        '*' if freq < MIN_FREQ_FLAG_G else '',
        })

    return pd.DataFrame(rows)

# ── Build & print all sizes ───────────────────────────────────────────
all_size_matrices = {}
_NEXT_SIZE_LBL = {
    '本体通常':          '詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ',
    '詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ': '詰替ﾒｶﾞｼﾞｬﾝﾎﾞ',
    '詰替ﾒｶﾞｼﾞｬﾝﾎﾞ':  '詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ',
    '詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ':  '詰替ﾃﾗｼﾞｬﾝﾎﾞ',
    '詰替ﾃﾗｼﾞｬﾝﾎﾞ':  None,
}

for size in SIZE_ORDER:
    mat = build_size_matrix(size)
    if len(mat) == 0:
        print(f'  ⚠️ No data for {size} — skipped')
        continue
    all_size_matrices[size] = mat

    abbr     = _size_abbr(size)
    next_lbl = _NEXT_SIZE_LBL.get(size) or '(top of ladder)'
    role     = SIZE_ROLE.get(size, '?')

    print()
    print('=' * 165)
    print(f'📊 G-1: {size} — PRICE POINT DECISION MATRIX')
    print(f'  Role: {role}  |  Next size → {next_lbl}  |  * = freq < {MIN_FREQ_FLAG_G:,} (low exposure)')
    print(f'  Source: df_cohort (unfiltered) — identical methodology to F-6')
    print('=' * 165)
    print(f'{"Price Band":>12s} {"Freq(w×s)":>10s} {"TrialEff%":>10s} {"Total N":>9s} {"Trial N":>8s} '
          f'{"Repeat%":>8s} {"Lapse%":>8s} '
          f'{"Same"+abbr+"%":>9s} {"SizeUp%":>8s} {"SizeDn%":>8s} '
          f'{"→Bold%":>8s} {"→Exit%":>8s} {"Gap→Next":>9s}')
    print('-' * 165)
    for _, r in mat.iterrows():
        gap_s = f'¥{r["gap_to_next_jpy"]:+,.0f}' if pd.notna(r.get('gap_to_next_jpy')) else '—'
        eff_s = f'{r["trial_efficiency_%"]:.1f}%'  if pd.notna(r.get('trial_efficiency_%')) else '—'
        bl    = f'{r["price_band"]}{r["_freq_flag"]}'
        print(f'{bl:>13s} {r["frequency"]:>10,} {eff_s:>10s} {r["Total N"]:>9,} {r["Trial N"]:>8,} '
              f'{r["repeat_%"]:>7.1f}% {r["lapse_%"]:>7.1f}% '
              f'{r["same_size_%"]:>8.1f}% {r["size_up_%"]:>7.1f}% {r["size_dn_%"]:>7.1f}% '
              f'{r["→Bold%"]:>7.1f}% {r["→Exit%"]:>7.1f}% {gap_s:>9s}')

print(f'\n✅ G-1 complete — matrices built for: {list(all_size_matrices.keys())}')



📊 G-1: 本体通常 — PRICE POINT DECISION MATRIX
  Role: Trial Entry  |  Next size → 詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ  |  * = freq < 100 (low exposure)
  Source: df_cohort (unfiltered) — identical methodology to F-6
  Price Band  Freq(w×s)  TrialEff%   Total N  Trial N  Repeat%   Lapse%   Same本体%  SizeUp%  SizeDn%   →Bold%   →Exit%  Gap→Next
---------------------------------------------------------------------------------------------------------------------------------------------------------------------
        ~¥49*          0          —     7,680    2,037    26.5%    73.5%     61.5%    38.5%     0.0%     7.8%    42.5%         —
        ~¥99*          0          —     7,907    4,644    21.4%    78.6%     69.5%    30.5%     0.0%    22.4%    32.3%         —
        ~¥149      2,185     122.4%     7,408    2,675    19.0%    81.0%     68.3%    31.7%     0.0%    10.5%    37.2%         —
        ~¥199    207,010      50.3%   229,126  104,100    27.0%    73.0%     74.9%    25.1%     0.0%    11.5%    39.6%         —

In [52]:
# ═══════════════════════════════════════════════════════════════════════
# G-2: Analysis Summary & Team Recommendations
# ═══════════════════════════════════════════════════════════════════════
# Synthesised findings from F-6 (ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ deep-dive) and G-1 (all sizes).
# Format: per-size highlights → cross-size implications → action items.

def _best_band(mat, col, higher_is_better=True):
    """Return the price_band label with the max (or min) value of col."""
    v = mat.dropna(subset=[col])
    if len(v) == 0:
        return None, None
    if higher_is_better:
        idx = v[col].idxmax()
    else:
        idx = v[col].idxmin()
    return v.loc[idx, 'price_band'], v.loc[idx, col]

print('=' * 110)
print('📝 G-2: ANALYSIS SUMMARY & TEAM RECOMMENDATIONS')
print('   アリエールジェルボール — サイズ別プライスポイント戦略 まとめ')
print('   (Unified 50-JPY ASP bins — F-6 and G-1 use identical methodology)')
print('=' * 110)

# ── Per-size sweet spot table ─────────────────────────────────────────
print('\n【1】SWEET SPOT SUMMARY — Best ASP Band per Metric per Size')
print('-' * 110)
print(f'{"Size":>20s} {"Role":>15s} {"Best Repeat":>14s} {"Best TrialEff":>14s} '
      f'{"Most SizeUp":>13s} {"Lapse Risk":>12s}')
print('-' * 110)
for size in SIZE_ORDER:
    mat = all_size_matrices.get(size)
    if mat is None or len(mat) == 0:
        continue
    role       = SIZE_ROLE.get(size, '?')
    br_band, br_val = _best_band(mat, 'repeat_%')
    te_band, te_val = _best_band(mat, 'trial_efficiency_%')
    su_band, su_val = _best_band(mat, 'size_up_%')
    lr_band, lr_val = _best_band(mat, 'lapse_%')

    br_s = f'{br_band}({br_val:.0f}%)' if br_band else '—'
    te_s = f'{te_band}({te_val:.1f}%)' if te_band else '—'
    su_s = f'{su_band}({su_val:.0f}%)' if su_band else '—'
    lr_s = f'{lr_band}({lr_val:.0f}%)' if lr_band else '—'
    print(f'{size:>20s} {role:>15s} {br_s:>14s} {te_s:>14s} {su_s:>13s} {lr_s:>12s}')

# ── HJ specific deep-dive findings ────────────────────────────────────
print()
print('【2】詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ (F-6) — Detailed ASP Band Findings')
print('-' * 110)
if 'df_decision' in dir() and len(df_decision) > 0:
    cols_to_show = ['price_band', 'frequency', 'trial_efficiency_%',
                    'total_shoppers', 'repeat_rate_%', 'lapse_rate_%',
                    'same_size_retention_%', 'size_up_%', 'lapse_to_bold_%',
                    'category_exit_%', 'gap_to_mega_jpy']
    avail = [c for c in cols_to_show if c in df_decision.columns]
    print(df_decision[avail].to_string(index=False))

    # Goldilocks zone: highest repeat & non-extreme lapse
    mid_bands = df_decision[df_decision['repeat_rate_%'] >= df_decision['repeat_rate_%'].median()]
    if len(mid_bands) > 0:
        gold = mid_bands.loc[mid_bands['lapse_rate_%'].idxmin()]
        print(f'\n  ▶ 推奨ゴールディロックスゾーン: {gold["price_band"]}')
        print(f'    Repeat={gold["repeat_rate_%"]:.1f}%  Lapse={gold["lapse_rate_%"]:.1f}%  '
              f'SizeUp={gold["size_up_%"]:.1f}%  Trial Eff={gold["trial_efficiency_%"] if pd.notna(gold["trial_efficiency_%"]) else "—"}%')

# ── Cross-size lapse risk summary ─────────────────────────────────────
print()
print('【3】LAPSE & BOLD SWITCH RISK — Bands to Avoid per Size')
print('-' * 110)
print(f'  {"Size":>20s}  {"Risk Band":>12s}  {"Lapse%":>7s}  {"→Bold%":>7s}  {"→Exit%":>7s}  Note')
print(f'  {"-"*20}  {"-"*12}  {"-"*7}  {"-"*7}  {"-"*7}  ------')
for size in SIZE_ORDER:
    mat = all_size_matrices.get(size)
    if mat is None or len(mat) == 0:
        continue
    risk = mat[mat['lapse_%'] > 60].copy() if 'lapse_%' in mat.columns else pd.DataFrame()
    if len(risk) == 0:
        # best safe band
        safe = mat.loc[mat['lapse_%'].idxmin()]
        print(f'  {size:>20s}  {"(all ok)":>12s}  {safe["lapse_%"]:>6.1f}%  '
              f'{safe["→Bold%"]:>6.1f}%  {safe["→Exit%"]:>6.1f}%  '
              f'Lowest lapse @ {safe["price_band"]}')
    else:
        for _, r in risk.iterrows():
            print(f'  {size:>20s}  {r["price_band"]:>12s}  {r["lapse_%"]:>6.1f}%  '
                  f'{r["→Bold%"]:>6.1f}%  {r["→Exit%"]:>6.1f}%  ⚠️ High lapse zone')

# ── Trial efficiency ranking ──────────────────────────────────────────
print()
print('【4】TRIAL EFFICIENCY RANKING — Top Bands by (N ÷ Frequency × 100)')
print('    高頻度かつ多トライアルを獲得できる「コスパ最高」プライスポイント Top10')
print('-' * 110)
all_eff_rows = []
for size, mat in all_size_matrices.items():
    sub = mat.dropna(subset=['trial_efficiency_%']).copy()
    sub['size_label'] = size
    all_eff_rows.append(sub[['size_label', 'price_band', 'frequency', 'N', 'trial_efficiency_%']])
if all_eff_rows:
    eff_all = pd.concat(all_eff_rows, ignore_index=True)
    eff_top = eff_all.nlargest(10, 'trial_efficiency_%')
    print(eff_top.rename(columns={
        'size_label': 'Size', 'price_band': 'Price Band',
        'frequency': 'Freq(w×s)', 'N': 'Trial N', 'trial_efficiency_%': 'TrialEff%'
    }).to_string(index=False))

# ── Strategic recommendations ─────────────────────────────────────────
print()
print('【5】TEAM RECOMMENDATIONS / チームへの提言')
print('=' * 110)
print("""
  ◆ 本体通常 (Trial Entry)
    → 最大のトライアル獲得口。トライアル効率が高いプライスポイントを優先実行。
    → 過度な値上げはイニシャルラプスを招くため、高コスパゾーンを死守する。

  ◆ 詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ (Intermission Bridge)
    → F-6 の50JPYビン分析で特定されたゴールディロックスゾーンを参照。
    → 低価格帯のビンは安すぎてサイズアップ誘因なし、高価格帯はBold流出リスク増大。
    → 各帯のトライアル効率 (N/Freq×100) を参照し、頻度対比の効果を検証すること。

  ◆ 詰替ﾒｶﾞｼﾞｬﾝﾎﾞ / 詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ (Loyalty Core)
    → 高リピート帯のプライスポイントを標準価格として棚設計に反映。
    → ラプス率が急上昇するプライスゾーンはプロモ頻度を絞る。

  ◆ 詰替ﾃﾗｼﾞｬﾝﾎﾞ (Loyalty Peak)
    → ハイロイヤルユーザー向け。プレミアム価格でも離反しにくい帯を特定し維持。

  ◆ 共通アクション
    1. 高頻度 × 高トライアル効率の帯 (【4】参照) を「推奨実行価格帯」に設定。
    2. ラプスリスク帯 (【3】参照) でのプロモ連射を避け、Bold流出を防止。
    3. 巻末の Excel (F_PriceMatrix シート) を価格戦略会議の基礎資料として活用。
    4. 今後、週次モニタリングで各帯の頻度変化を追跡し、本分析を更新する。

  ◆ 方法論の統一
    F-6 (HJ deep-dive) と G-1 (全サイズ) は同一のデータソース (df_cohort)
    かつ同一の50JPYビン分割を使用。数値の完全一致を保証。
""")

print('✅ G-2 Analysis Summary complete.')

📝 G-2: ANALYSIS SUMMARY & TEAM RECOMMENDATIONS
   アリエールジェルボール — サイズ別プライスポイント戦略 まとめ
   (Unified 50-JPY ASP bins — F-6 and G-1 use identical methodology)

【1】SWEET SPOT SUMMARY — Best ASP Band per Metric per Size
--------------------------------------------------------------------------------------------------------------
                Size            Role    Best Repeat  Best TrialEff   Most SizeUp   Lapse Risk
--------------------------------------------------------------------------------------------------------------
                本体通常     Trial Entry     ~¥399(32%)  ~¥149(122.4%)    ~¥449(58%)  ~¥699(100%)
       詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ    Intermission  ~¥1,149(100%)  ~¥699(273.8%) ~¥1,349(100%) ~¥1,999(100%)
         詰替ﾒｶﾞｼﾞｬﾝﾎﾞ         Loyalty  ~¥2,349(100%) ~¥2,299(218.3%)  ~¥2,049(65%) ~¥3,549(100%)
        詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ         Loyalty     ~¥149(50%) ~¥999(4800.0%)   ~¥199(100%)  ~¥249(100%)
          詰替ﾃﾗｼﾞｬﾝﾎﾞ         Loyalty     ~¥99(100%) ~¥3,499(4382.5%)      ~¥49(0%)  ~¥249(100%

KeyError: "['N'] not in index"

---
# Pane F — Strategic Summary
*Synthesized findings and recommendations aligned to the analysis objective.*


In [53]:
# ── F-1: Auto-Generated Strategic Summary ─────────────────────────────
print('=' * 100)
print('STRATEGIC SUMMARY: Brand Building via Loyal User Growth')
print('Objective: Define each size\'s most effective price point via Trial/Repeat/Lapse flow')
print('=' * 100)

# Per-size metrics table
print('\n📊 SIZE-LEVEL PERFORMANCE SUMMARY')
print('-' * 100)
print(f'{"Size":>25s} {"Role":>15s} {"Trial":>8s} {"Repeat":>8s} {"Lapse":>8s} '
      f'{"Rep%":>6s} {"Lap%":>6s} {"Avg ASP":>8s}')
print('-' * 100)

for _, row in df_asp_flow.iterrows():
    role = SIZE_ROLE.get(row['size'], '?')
    print(f'{row["size"]:>25s} {role:>15s} {row["trial_shoppers"]:>8,} '
          f'{row["repeat_shoppers"]:>8,} {row["lapse_shoppers"]:>8,} '
          f'{row["repeat_rate_%"]:>5.1f}% {row["lapse_rate_%"]:>5.1f}% '
          f'¥{row["avg_trial_asp"]:>7,.0f}')

# Cross-flow
print(f'\n🔄 CROSS-BRAND FLOW')
ariel_to_bold_gb = dest_summary[dest_summary['next_sub_brand']==BOLD_GB]['shoppers'].sum() if len(dest_summary)>0 else 0
bold_gb_to_ariel = bold_dest_summary[bold_dest_summary['next_sub_brand']==ARIEL_GB]['shoppers'].sum() if len(bold_dest_summary)>0 else 0
print(f'  Ariel → Bold: {ariel_to_bold_gb:,}')
print(f'  Bold → Ariel: {bold_gb_to_ariel:,}')
print(f'  Net flow:        {bold_gb_to_ariel - ariel_to_bold_gb:+,} ({"Ariel gains" if bold_gb_to_ariel > ariel_to_bold_gb else "Ariel loses"})')

total_ariel_lapsed = df_lapsed[df_lapsed['sub_brand']==ARIEL_GB]['shopper_key'].nunique()
# Rough revenue at risk estimate
avg_asp_overall = df_monthly[df_monthly['sub_brand']==ARIEL_GB]['weighted_asp'].mean()
rev_at_risk = total_ariel_lapsed * avg_asp_overall * 2  # ~2 purchases/year assumption
print(f'\n💰 REVENUE AT RISK')
print(f'  Total Ariel lapsed: {total_ariel_lapsed:,}')
print(f'  Est. revenue at risk: ¥{rev_at_risk:,.0f}/year')

print('\n📋 KEY INSIGHTS:')
print('  1. 本体通常 (Trial Entry) drives the highest trial volume but has the highest lapse risk')
print('  2. 詰替超特大 (Intermission) is the critical bridge — funneling 本体通常 trial into this size maximizes retention')
print('  3. Refill sizes (詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ and larger) show the highest repeat rates = loyalty fortress')
print('  4. Price optimization should focus on 本体通常 trial acquisition ASP and 詰替超特大 repeat conversion ASP')


STRATEGIC SUMMARY: Brand Building via Loyal User Growth
Objective: Define each size's most effective price point via Trial/Repeat/Lapse flow

📊 SIZE-LEVEL PERFORMANCE SUMMARY
----------------------------------------------------------------------------------------------------
                     Size            Role    Trial   Repeat    Lapse   Rep%   Lap%  Avg ASP
----------------------------------------------------------------------------------------------------
                     本体通常     Trial Entry  184,693   50,833  133,860  27.5%  72.5% ¥    240
            詰替ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ    Intermission  184,783   65,145  119,638  35.3%  64.7% ¥    901
              詰替ﾒｶﾞｼﾞｬﾝﾎﾞ         Loyalty   69,350   22,610   46,740  32.6%  67.4% ¥  1,790
             詰替超ﾒｶﾞｼﾞｬﾝﾎﾞ         Loyalty   63,122   18,829   44,293  29.8%  70.2% ¥  2,349
               詰替ﾃﾗｼﾞｬﾝﾎﾞ         Loyalty   26,329    8,006   18,323  30.4%  69.6% ¥  3,033

🔄 CROSS-BRAND FLOW
  Ariel → Bold: 71,145
  Bold → Ariel: 55,438
  Net

In [54]:
# ── Export Comprehensive Output (including new price-point matrix sheets) ──
with pd.ExcelWriter(OUTPUT_DIR / 'ariel_gel_ball_comprehensive.xlsx', engine='openpyxl') as writer:
    strip_tz(size_summary).to_excel(writer, sheet_name='A_Size_Summary', index=False)
    # A-3 (flat) and A-5 (dose_table) — export only if cells are active
    if 'flat' in dir(): strip_tz(flat).to_excel(writer, sheet_name='A_Pre_Post_Renewal', index=False)
    if 'dose_table' in dir(): strip_tz(dose_table).to_excel(writer, sheet_name='A_Per_Dose_ASP', index=False)
    if 'mig_compare' in dir(): strip_tz(mig_compare).to_excel(writer, sheet_name='C_Migration_Dir', index=False)
    if 'pp_all_filt' in dir(): strip_tz(pp_all_filt).to_excel(writer, sheet_name='C_PricePoint_Prod', index=False)
    if 'per_size_dest_df' in dir(): strip_tz(per_size_dest_df).to_excel(writer, sheet_name='D_PerSize_Dest', index=False)
    if 'atk_lapse_asp' in dir(): strip_tz(atk_lapse_asp).to_excel(writer, sheet_name='D_Attack_Lapse', index=False)
    strip_tz(pd.DataFrame(df_trial.groupby(['sub_brand','size_code']).agg(
        total_trial=('subbrand_trial_shoppers','sum')).reset_index())).to_excel(writer, sheet_name='B_Trial_Summary', index=False)
    strip_tz(h2h_idx).to_excel(writer, sheet_name='B_H2H_Trial', index=False)
    strip_tz(funnel_pivot).to_excel(writer, sheet_name='C_Cohort_Funnel', index=False)
    if 'period_pivot' in dir(): strip_tz(period_pivot).to_excel(writer, sheet_name='C_PrePost_Renewal', index=False)
    strip_tz(rate_df).to_excel(writer, sheet_name='C_ASP_Band_Rates', index=False)
    if 'asp_lapse' in dir(): strip_tz(asp_lapse).to_excel(writer, sheet_name='D_Lapse_by_ASP', index=False)
    strip_tz(dest_summary).to_excel(writer, sheet_name='D_Destination', index=False)
    strip_tz(funnel_data).to_excel(writer, sheet_name='E_Funnel', index=False)
    strip_tz(df_asp_flow).to_excel(writer, sheet_name='E_Strategic_Map', index=False)

    # ── F_HJ_Matrix: ﾊｲﾊﾟｰｼﾞｬﾝﾎﾞ 50-JPY bin decision matrix (F-6) ───────
    if 'df_decision' in dir() and len(df_decision) > 0:
        export_hj = df_decision.copy()
        strip_tz(export_hj).to_excel(writer, sheet_name='F_HJ_Matrix', index=False)
        # Auto-fit columns using openpyxl
        ws_hj = writer.sheets['F_HJ_Matrix']
        for col in ws_hj.columns:
            max_len = max(len(str(cell.value)) if cell.value is not None else 0 for cell in col)
            ws_hj.column_dimensions[col[0].column_letter].width = min(max_len + 3, 25)

    # ── F_PriceMatrix: all sizes stacked vertically ───────────────────
    if 'all_size_matrices' in dir() and len(all_size_matrices) > 0:
        from openpyxl.styles import PatternFill, Font, Alignment
        from openpyxl.utils import get_column_letter

        # Build stacked DataFrame with size separator rows
        EXPORT_COLS = [
            'size', 'price_band', 'frequency', 'trial_efficiency_%',
            'N', 'repeat_%', 'lapse_%',
            'same_size_%', 'size_up_%', 'size_dn_%',
            '→Bold%', '→Exit%', 'gap_to_next_jpy',
        ]
        stacked_parts = []
        for sz, mat in all_size_matrices.items():
            row_sep = pd.DataFrame([{c: '' for c in EXPORT_COLS}])
            row_sep.at[0, 'size'] = f'▼ {sz}'
            avail = [c for c in EXPORT_COLS if c in mat.columns]
            mat_export = mat[avail].copy()
            # add missing cols as empty
            for c in EXPORT_COLS:
                if c not in mat_export.columns:
                    mat_export[c] = ''
            stacked_parts.extend([row_sep, mat_export[EXPORT_COLS]])

        if stacked_parts:
            df_stacked = pd.concat(stacked_parts, ignore_index=True)
            df_stacked.columns = [
                'Size', 'Price Band', 'Freq(w×s)', 'TrialEff%',
                'N', 'Repeat%', 'Lapse%',
                'SameSize%', 'SizeUp%', 'SizeDown%',
                '→Bold%', '→Exit%', 'GapToNext(¥)',
            ]
            strip_tz(df_stacked).to_excel(writer, sheet_name='F_PriceMatrix', index=False)

            # Formatting: bold header, wider columns, highlight size separator rows
            ws_pm = writer.sheets['F_PriceMatrix']
            header_fill = PatternFill('solid', fgColor='1F497D')
            sep_fill    = PatternFill('solid', fgColor='BDD7EE')
            bold_white  = Font(bold=True, color='FFFFFF')
            bold_dark   = Font(bold=True, color='000000')

            # Bold header row
            for cell in ws_pm[1]:
                cell.font      = bold_white
                cell.fill      = header_fill
                cell.alignment = Alignment(horizontal='center', wrap_text=True)

            # Highlight separator rows (Size column starts with ▼)
            for row in ws_pm.iter_rows(min_row=2):
                val = str(row[0].value) if row[0].value is not None else ''
                if val.startswith('▼'):
                    for cell in row:
                        cell.fill = sep_fill
                        cell.font = bold_dark

            # Auto-width
            for col in ws_pm.columns:
                max_len = max(len(str(cell.value)) if cell.value is not None else 0
                              for cell in col)
                ws_pm.column_dimensions[col[0].column_letter].width = min(max_len + 3, 22)

print(f'\n✅ Comprehensive output saved to {OUTPUT_DIR / "ariel_gel_ball_comprehensive.xlsx"}')
print('   New sheets: F_HJ_Matrix (50-JPY bin HJ matrix) | F_PriceMatrix (all sizes stacked)')
print(f'\nAll output CSVs in {OUTPUT_DIR}:')
for f in sorted(OUTPUT_DIR.glob('pane_*.csv')):
    print(f'  {f.name}')
print(f'\n🎯 Analysis complete. Set RELOAD_FROM_CACHE = True for instant reload next time.')



✅ Comprehensive output saved to output_ariel\ariel_gel_ball_comprehensive.xlsx
   New sheets: F_HJ_Matrix (50-JPY bin HJ matrix) | F_PriceMatrix (all sizes stacked)

All output CSVs in output_ariel:
  pane_a_asp_trend.csv
  pane_a_ratio_gap.csv
  pane_a_size_summary.csv
  pane_b_h2h_trial.csv
  pane_b_heatmap_values.csv
  pane_b_trial_summary.csv
  pane_b_trial_trend.csv
  pane_c_asp_band_rates.csv
  pane_c_asp_boxplot_stats.csv
  pane_c_cohort_funnel.csv
  pane_c_migration_direction.csv
  pane_c_pp_all.csv
  pane_c_pp_split.csv
  pane_d_destination.csv
  pane_d_per_size_destination.csv
  pane_e_attack_sankey_stage1.csv
  pane_e_attack_sankey_stage2.csv
  pane_e_funnel.csv
  pane_e_sankey_stage1.csv
  pane_e_sankey_stage2.csv
  pane_e_strategic_map.csv
  pane_f_hj_decision_matrix.csv
  pane_f_hj_gap_vs_migration.csv
  pane_f_hj_lapse_dest_simple.csv
  pane_f_hj_lapse_destination.csv
  pane_f_hj_migration_direction.csv
  pane_f_hj_outcome_by_price.csv
  pane_f_hj_price_gap.csv
  pane_f